# Noisy optimisation under a fixed evaluation budget

## Master experiment notebook

The study asks when noisy optimisation is limited by finding good candidates,
making useful search updates, or recognising the best candidate already found.
Stages A and B establish the effects of geometry and noise. Stage C compares
evaluation allocation and final recommendation. BBOB is a supporting transfer check.

This notebook contains the source for all nine supplied notebooks, plus a new Stage C
mechanism experiment. It writes readable Python scripts into `master_code/`, then runs
selected stages in separate namespaces. Original functions and numerical settings are
retained in the reproduction sections. Each stage has its own results directory.

### Run instructions

1. Run the dependency cell in a fresh Colab/Jupyter kernel.
2. Set the output folder and run mode in the configuration cell.
3. Use **Run all**. Source cells prepare every stage; only selected experiments execute.
4. Download the final ZIP and save a copy of this executed notebook. The ZIP includes
   scripts, settings, CSVs, plots, and compressed seed histories.

The default analyses the completed Stage C archive. The repository includes
results/optimisation_results_full.zip. Set SHORTLIST_ARCHIVE to the uploaded
file's path if running in Colab. Run all to produce a separate shortlist ZIP.
Use the new mode to regenerate the underlying histories and run both analyses.
For Colab persistence, mount Google Drive and set `RESULTS_ROOT` to a Drive folder before
running. Local `/content` files disappear when a Colab runtime is discarded.

### Contents

- [Configuration and reproducibility](#Configuration)
- [Stage A geometry and sensitivity checks](#Stage-A---conditioning,-rotation-and-bending)
- [Stage B observation noise](#Stage-B---observation-noise)
- [Stage C fixed allocations](#Stage-C---fixed-repeat-allocations)
- [New Stage C investigations](#New-Stage-C---search,-selection-and-evaluation-allocation)
- [Export and next report revision](#Export-and-next-report-revision)

The earlier development baseline, geometry pilot, and original transfer experiment
are also retained in their own sections and are selected by `everything` mode.

In [ ]:
%pip install -q numpy==2.4.6 scipy==1.16.3 pandas==2.2.3 matplotlib==3.10.0 cma==4.5.0 coco-experiment==2.8.3 Jinja2==3.1.6

## Configuration

`new` runs the requested investigations. `report` reproduces the original main stages
and runs the new investigation. `everything` also reruns the earlier development
notebooks and the original transfer sample. `custom` uses your explicit stage list.

The new full experiment uses seeds 10100-10123, four valley shape/orientation settings
at kappa 100, and noise SDs 0, 0.001, and 0.01. Optional supporting BBOB experiments
use instances 201-224. The smoke profile uses two seed blocks and noise SDs 0 and 0.01,
while preserving the 96-call budget, 72-call prefix, 1,024-point pool, and policy mechanics.

In [ ]:
from pathlib import Path
import os

# For Colab persistence, uncomment these lines and choose a Drive output folder.
# from google.colab import drive
# drive.mount("/content/drive")
# RESULTS_ROOT = Path("/content/drive/MyDrive/noisy_optimisation_results")

PROFILE = "full"                   # "smoke" or "full"
RUN_MODE = "shortlist"             # "shortlist", "new", "report", "everything", "custom"
SHORTLIST_ARCHIVE = Path("results/optimisation_results_full.zip")
RESULTS_ROOT = Path("optimisation_shortlist_results" if RUN_MODE == "shortlist" else "optimisation_results")
INCLUDE_SUPPORTING_TRANSFER = True
CUSTOM_STAGES = ["mechanisms"]

for name in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS"):
    os.environ[name] = "1"

ALL_STAGES = ["baseline", "geometry_pilot", "stage_a", "stage_b", "stage_b_budget",
              "stage_c_fixed", "stage_c_dynamic", "stage_c_final", "transfer_original", "mechanisms", "shortlist"]
RUN_MODES = {
    "shortlist": ["shortlist"],
    "new": ["mechanisms", "shortlist"],
    "report": ["stage_a", "stage_b", "stage_b_budget", "stage_c_fixed",
               "stage_c_dynamic", "stage_c_final", "mechanisms", "shortlist"],
    "everything": ALL_STAGES,
    "custom": CUSTOM_STAGES,
}
if PROFILE not in ("smoke", "full") or RUN_MODE not in RUN_MODES:
    raise ValueError("Invalid profile or run mode.")
RUN_STAGES = RUN_MODES[RUN_MODE]
if set(RUN_STAGES) - set(ALL_STAGES):
    raise ValueError("Unknown stage in CUSTOM_STAGES.")
RESULTS_ROOT = RESULTS_ROOT.expanduser().resolve()
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
Path("master_code").mkdir(exist_ok=True)
print("Profile:", PROFILE)
print("Selected stages:", RUN_STAGES)
print("Results:", RESULTS_ROOT)

In [ ]:
import hashlib
import importlib
import importlib.metadata
import json
import runpy
import shutil
import sys
import zipfile
from datetime import datetime, timezone

VERSIONS = {name: importlib.metadata.version(name)
            for name in ("numpy", "scipy", "pandas", "matplotlib", "cma", "coco-experiment", "Jinja2")}
for name in ("numpy", "scipy", "pandas", "matplotlib"):
    if importlib.import_module(name).__version__ != VERSIONS[name]:
        raise RuntimeError("A previously imported package differs from the installed version. Restart the kernel.")
print({"python": sys.version.split()[0], **VERSIONS})
RUN_STATUS = {}


def run_stage(name):
    if name not in RUN_STAGES:
        print(f"Prepared {name}; execution skipped by RUN_MODE.")
        return
    script = Path("master_code") / f"{name}.py"
    provenance = dict(profile=PROFILE, source_sha256=hashlib.sha256(script.read_bytes()).hexdigest(),
                      python=sys.version.split()[0], packages=VERSIONS,
                      supporting_transfer=INCLUDE_SUPPORTING_TRANSFER if name == "mechanisms" else None)
    directory = RESULTS_ROOT / ("stage_c_mechanisms" if name == "mechanisms" else name) / PROFILE
    directory.mkdir(parents=True, exist_ok=True)
    marker = directory / "master_provenance.json"
    if marker.exists() and json.loads(marker.read_text(encoding="utf-8")) != provenance:
        raise ValueError(f"{name}: existing source/configuration differs. Choose a new RESULTS_ROOT.")
    marker.write_text(json.dumps(provenance, indent=2), encoding="utf-8")
    shutil.copyfile(script, directory / "experiment.py")
    print(f"Running {name} ({PROFILE})", flush=True)
    RUN_STATUS[name] = "running"
    try:
        runpy.run_path(str(script), init_globals={
            "MASTER_PROFILE": PROFILE, "MASTER_ROOT": RESULTS_ROOT,
            "MASTER_INCLUDE_TRANSFER": INCLUDE_SUPPORTING_TRANSFER,
            "MASTER_SHORTLIST_ARCHIVE": SHORTLIST_ARCHIVE,
        })
    except Exception:
        RUN_STATUS[name] = "failed"
        raise
    RUN_STATUS[name] = "completed"
    print(f"Completed {name}", flush=True)

## Development baseline

Source reused from `stage_a_baseline.ipynb`. The configuration above controls execution. Code cells in this section prepare the stage script; the final cell runs it when selected.

In [ ]:
%%writefile master_code/baseline.py
from pathlib import Path
import os

MASTER_PROFILE = globals().get("MASTER_PROFILE", os.environ.get("OPT_PROFILE", "smoke"))
MASTER_ROOT = Path(globals().get("MASTER_ROOT", os.environ.get("OPT_RESULTS_ROOT", "optimisation_results"))).resolve()
MASTER_INCLUDE_TRANSFER = globals().get("MASTER_INCLUDE_TRANSFER", os.environ.get("OPT_INCLUDE_TRANSFER", "1") == "1")
if MASTER_PROFILE not in ("full", "smoke"):
    raise ValueError("Choose full or smoke.")

In [ ]:
%%writefile -a master_code/baseline.py

import csv
import math
import random
from pathlib import Path

### Define the black-box problem

The optimiser submits a two-number point in the unit disk and gets a loss back.
It seeks the smallest loss within 96 calls. The experimenter knows the hidden
minimum and uses that knowledge only to check and score the run.

For an aligned point, the loss is

$$f_0(x)=\frac{(x_1-a)^2+\kappa(x_2-b)^2}{(1+\kappa)/4+a^2+\kappa b^2}.$$

The centre $(a,b)$ is the minimum and its loss is zero.
The coefficient $\kappa$ controls the curvature ratio.
The denominator makes the mean loss of a uniformly drawn disk point equal to one.
This scale makes comparisons between paired orientations interpretable.
It does not imply that all conditioning levels have the same geometric difficulty.

The rotated partner is $f_\theta(x)=f_0(R_\theta^T x)$.
Both the centre and the point coordinates rotate. The circular domain maps onto itself.

In [ ]:
%%writefile -a master_code/baseline.py

"""Paired quadratic valleys on the unit disk. All angles are in radians."""

from dataclasses import dataclass
from math import cos, sin, hypot, sqrt, tau


def rotate(point, angle):
    x, y = point
    return cos(angle) * x - sin(angle) * y, sin(angle) * x + cos(angle) * y


def uniform_disk(rng):
    radius = sqrt(rng.random())
    return rotate((radius, 0.0), tau * rng.random())


@dataclass(frozen=True)
class Valley:
    centre: tuple
    angle: float
    curvature_ratio: float = 100.0

    def clean(self, point):
        if hypot(*point) > 1.0 + 1e-12:
            raise ValueError("Point outside the unit disk")
        x, y = rotate(point, -self.angle)
        a, b = self.centre
        k = self.curvature_ratio
        # Each coordinate of a uniform point on the disk has second moment 1/4.
        mean_gap = (1 + k) / 4 + a*a + k*b*b
        return ((x - a)**2 + k*(y - b)**2) / mean_gap

    def observe(self, point, sigma, rng):
        return self.clean(point) + sigma * rng.gauss(0.0, 1.0)

### Choose one demonstration instance

Use a fixed development seed, a hidden centre at radius 0.4, and a 96-call budget.
We compare curvature ratios 1, 10 and 100 in orientations 0° and 45°.
The sphere at ratio 1 is the control. It has equal curvature in every direction.

In [ ]:
%%writefile -a master_code/baseline.py

SEED = 1100
BUDGET = 96
CONDITION_NUMBERS = (1, 10, 100)
ANGLE = math.pi / 4

centre_rng = random.Random(SEED)
centre = rotate((0.4, 0.0), centre_rng.uniform(0, math.tau))
print("Hidden centre for scoring:", centre)

### Validate the objective before testing algorithms

Check three facts. Each instance has minimum zero. A point and its rotated
counterpart have the same clean loss. A sphere rotated about its fixed centre
returns the same loss. These deterministic checks can detect implementation
mistakes without relying on an optimiser's behaviour.

In [ ]:
%%writefile -a master_code/baseline.py

check_rng = random.Random(SEED + 1)
check_points = [uniform_disk(check_rng) for _ in range(100)]

for kappa in CONDITION_NUMBERS:
    aligned = Valley(centre, 0.0, kappa)
    rotated = Valley(centre, ANGLE, kappa)
    assert abs(aligned.clean(centre)) < 1e-12
    assert abs(rotated.clean(rotate(centre, ANGLE))) < 1e-12
    for point in check_points:
        assert math.isclose(
            aligned.clean(point),
            rotated.clean(rotate(point, ANGLE)),
            rel_tol=1e-10, abs_tol=1e-12,
        )

sphere = Valley(centre, 0.0, 1)
for point in check_points:
    displacement = (0.2 * point[0], 0.2 * point[1])
    turned = rotate(displacement, ANGLE)
    first = tuple(c + d for c, d in zip(centre, displacement))
    second = tuple(c + d for c, d in zip(centre, turned))
    assert math.isclose(sphere.clean(first), sphere.clean(second), abs_tol=1e-12)
print("Objective and rotation checks passed.")

### Run random search

Random search samples each new point uniformly in the disk.
The square root in uniform_disk is needed for uniform area sampling.
It returns the lowest observed point after the budget is spent.
Here observations are noise-free, so the best observed loss is the true gap.

For an exact symmetry check, rotate the same random points for the paired run.
This explicitly couples the random baseline. A shared integer seed alone does
not guarantee equivalent trajectories for adaptive algorithms.

In [ ]:
%%writefile -a master_code/baseline.py

def run_random_search(evaluate, points):
    best = math.inf
    history = []
    for call, point in enumerate(points, start=1):
        value = evaluate(point)
        best = min(best, value)
        history.append({
            "call": call, "x": point[0], "y": point[1],
            "value": value, "best_gap": best,
        })
    return history

proposal_rng = random.Random(SEED + 2)
proposals = [uniform_disk(proposal_rng) for _ in range(BUDGET)]
paired_proposals = [rotate(point, ANGLE) for point in proposals]

histories = {}
for kappa in CONDITION_NUMBERS:
    first = run_random_search(Valley(centre, 0.0, kappa).clean, proposals)
    second = run_random_search(Valley(centre, ANGLE, kappa).clean, paired_proposals)
    histories[kappa, 0] = first
    histories[kappa, 45] = second
    assert len(first) == len(second) == BUDGET
    for a, b in zip(first, second):
        assert math.isclose(a["best_gap"], b["best_gap"], abs_tol=1e-12)
    print(f"kappa={kappa:3d} | aligned={first[-1]['best_gap']:.6g}"
          f" | rotated={second[-1]['best_gap']:.6g}")

### Plot progress

The vertical axis shows the smallest gap reached so far.
The paired curves should overlap. A single development run explains the setup;
performance claims require the independent repeated instances used in the full study.

In [ ]:
%%writefile -a master_code/baseline.py

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), sharey=True)
for ax, kappa in zip(axes, CONDITION_NUMBERS):
    for angle, style in ((0, "-"), (45, "--")):
        run = histories[kappa, angle]
        ax.semilogy(
            [row["call"] for row in run],
            [max(row["best_gap"], 1e-14) for row in run],
            style, label=f"{angle} degrees",
        )
    ax.set_title(f"Curvature ratio {kappa}")
    ax.set_xlabel("Objective calls")
    ax.grid(alpha=0.25)
axes[0].set_ylabel("Best true objective gap")
axes[-1].legend()
fig.tight_layout()
plt.show()

### Save the baseline and move to the next step

Save the points, observations and running gaps so the plot can be reconstructed.
Next, replace the random proposal rule with BO and CMA-ES while retaining the
objective, feasible region, budget and scoring. Their shared initial information
and independent repeated instances are specified in the full protocol.

Keep a notebook for explanation, visual inspection and experiments. Once stable,
put reusable functions in Python modules and import them from the notebook.
Both .ipynb and .py files can be published in the same GitHub repository.

This demonstration uses Python's random generator and one development seed.
It is an introduction to the benchmark; the complete stage_a.py uses the frozen
NumPy candidate streams and 24 final instances, so numerical curves differ.

In [ ]:
%%writefile -a master_code/baseline.py

destination = MASTER_ROOT / "baseline" / MASTER_PROFILE / "baseline_random_search.csv"
destination.parent.mkdir(parents=True, exist_ok=True)
rows = [
    {"seed": SEED, "kappa": kappa, "angle": angle, **row}
    for (kappa, angle), run in histories.items()
    for row in run
]
with destination.open("w", newline="") as stream:
    writer = csv.DictWriter(stream, fieldnames=rows[0].keys())
    writer.writeheader()
    writer.writerows(rows)
print(f"Saved {len(rows)} observations to {destination}")

In [ ]:
run_stage("baseline")

## Early straight-valley geometry study

Source reused from `stage-a-geometry.ipynb`. The configuration above controls execution. Code cells in this section prepare the stage script; the final cell runs it when selected.

In [ ]:
%%writefile master_code/geometry_pilot.py
from pathlib import Path
import os

MASTER_PROFILE = globals().get("MASTER_PROFILE", os.environ.get("OPT_PROFILE", "smoke"))
MASTER_ROOT = Path(globals().get("MASTER_ROOT", os.environ.get("OPT_RESULTS_ROOT", "optimisation_results"))).resolve()
MASTER_INCLUDE_TRANSFER = globals().get("MASTER_INCLUDE_TRANSFER", os.environ.get("OPT_INCLUDE_TRANSFER", "1") == "1")
if MASTER_PROFILE not in ("full", "smoke"):
    raise ValueError("Choose full or smoke.")

In [ ]:
%%writefile -a master_code/geometry_pilot.py

import csv
import json
import math
import os
import random
import runpy
import sys
import time
from collections import defaultdict
from dataclasses import dataclass
from importlib.metadata import distribution, version
from math import cos, sin, hypot, sqrt, tau
from pathlib import Path

for variable in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS"):
    os.environ[variable] = "1"

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display
from matplotlib.lines import Line2D
from scipy.linalg import cholesky, cho_solve, solve_triangular
from scipy.optimize import minimize
from scipy.spatial.distance import cdist

# Load the pinned pure-Python reference CMA-ES class.
pure_cma = distribution("cma").locate_file("cma/more_algorithms/purecma.py")
CMAES = runpy.run_path(str(pure_cma))["CMAES"]

OUT = MASTER_ROOT / "geometry_pilot" / MASTER_PROFILE
KINDS = (1, 10, 100)
ANGLES = (0, 45)
BUDGET = 96
POPULATION = 6
POOL = 1024
JITTER = 1e-7
PILOT_SEEDS = range(9000, 9002 if MASTER_PROFILE == "smoke" else 9002)
FINAL_SEEDS = range(9100, 9102 if MASTER_PROFILE == "smoke" else 9124)
REPLICATION_SEEDS = range(9200, 9202 if MASTER_PROFILE == "smoke" else 9224)
CHECKPOINTS = (24, 48, 96)
SEEDS = list(FINAL_SEEDS)
FLOOR = 1e-14

# Main runs are always analysed. Turn off either follow-up for a shorter run.
RUN_SENSITIVITY = True
RUN_REPLICATION = True

print("Python", sys.version.split()[0])
print({package: version(package) for package in ("numpy", "scipy", "cma", "matplotlib")})

### Objective and random-search baseline

The objective is a shifted quadratic divided by its mean over the unit disk. The true minimum is zero, so the best observed value equals the best true gap in this noiseless stage. For a paired rotated problem, the centre, points and valley rotate together. The disk stays fixed under rotation. Uniform random search rotates the same proposal sequence along with the objective, so its paired curves should overlap.

In [ ]:
%%writefile -a master_code/geometry_pilot.py

def rotate(point, angle):
    x, y = point
    return cos(angle) * x - sin(angle) * y, sin(angle) * x + cos(angle) * y


def uniform_disk(rng):
    radius = sqrt(rng.random())
    return rotate((radius, 0.0), tau * rng.random())


@dataclass(frozen=True)
class Valley:
    centre: tuple
    angle: float
    curvature_ratio: float = 100.0

    def clean(self, point):
        if hypot(*point) > 1.0 + 1e-12:
            raise ValueError("Point outside the unit disk")
        x, y = rotate(point, -self.angle)
        a, b = self.centre
        k = self.curvature_ratio
        # Each coordinate of a uniform point on the disk has second moment 1/4.
        mean_gap = (1 + k) / 4 + a*a + k*b*b
        return ((x - a)**2 + k*(y - b)**2) / mean_gap

    def observe(self, point, sigma, rng):
        return self.clean(point) + sigma * rng.gauss(0.0, 1.0)


def rotation(degree):
    angle = math.radians(degree)
    return np.array([rotate((1, 0), angle), rotate((0, 1), angle)]).T


def disk(rng, size):
    u = rng.random((size, 2))
    angle = 2 * np.pi * u[:, 0]
    return np.sqrt(u[:, 1:]) * np.column_stack((np.cos(angle), np.sin(angle)))

### Bayesian optimisation

Both BO variants refit GP hyperparameters every six calls and minimise the same lower confidence bound. Pool BO evaluates its best candidate among 1,024 draws. Refined BO also starts a constrained local search from two promising pool points. Both have the same objective-call budget. The optional refinement spends extra computation evaluating the surrogate.

In [ ]:
%%writefile -a master_code/geometry_pilot.py

def covariance(x, y, theta):
    lengths = np.exp(theta[1:])
    return np.exp(theta[0] - .5 * cdist(x / lengths, y / lengths, "sqeuclidean"))


def fit(x, values, theta, refit, noise_sd=0.0, jitter=JITTER):
    output_scale = values.std() or 1
    y = (values - values.mean()) / output_scale
    ident = np.eye(len(x))

    def factor(parameters):
        signal = covariance(x, x, parameters)
        lower = cholesky(signal + (jitter + (noise_sd/output_scale)**2) * ident, lower=True)
        return signal, lower, cho_solve((lower, True), y)

    def likelihood(parameters):
        signal, lower, alpha = factor(parameters)
        loss = .5 * (y @ alpha) + np.log(np.diag(lower)).sum()
        weights = .5 * (cho_solve((lower, True), ident) - np.outer(alpha, alpha)) * signal
        gradient = [weights.sum()]
        if len(parameters) == 2:
            gradient.append((weights * cdist(x, x, "sqeuclidean")).sum() / np.exp(2*parameters[1]))
        else:
            for axis, log_length in enumerate(parameters[1:]):
                diff = x[:, axis, None] - x[:, axis]
                gradient.append((weights * diff**2).sum() / np.exp(2*log_length))
        return loss, np.array(gradient)

    status = -1
    if refit:
        bounds = [(-6, 6)] + [(math.log(.02), math.log(2))] * (len(theta)-1)
        result = minimize(likelihood, theta, jac=True, bounds=bounds, method="L-BFGS-B",
                          options={"maxiter": 50})
        theta, status = result.x, int(result.status)
    _, lower, alpha = factor(theta)
    return theta, lower, alpha, status


def predict(x, candidates, theta, lower, alpha):
    cross = covariance(x, candidates, theta)
    projected = solve_triangular(lower, cross, lower=True, check_finite=False)
    variance = np.maximum(0, np.exp(theta[0]) - np.einsum("ij,ij->j", projected, projected))
    return cross.T @ alpha, np.sqrt(variance)

In [ ]:
%%writefile -a master_code/geometry_pilot.py

def disk_constraint(point):
    return 1 - point @ point


def disk_jacobian(point):
    return -2 * point


def refined_candidate(x, pool, theta, lower, alpha, pool_lcb):
    """Refine two promising pool candidates with the same fitted LCB."""
    precision = np.broadcast_to(np.exp(-2 * theta[1:]), (2,))
    amplitude = np.exp(theta[0])

    def lcb(p):
        delta = x - p
        kernel = amplitude * np.exp(-.5 * ((delta ** 2) @ precision))
        derivative = kernel[:, None] * delta * precision
        projected = solve_triangular(lower, kernel, lower=True, check_finite=False)
        weights = solve_triangular(lower.T, projected, lower=False, check_finite=False)
        # A tiny variance floor avoids an undefined derivative at zero.
        variance = amplitude - projected @ projected
        sd = np.sqrt(max(variance, 1e-20))
        mean_gradient = derivative.T @ alpha
        gradient = mean_gradient + 2 * (derivative.T @ weights) / sd if variance > 1e-20 else mean_gradient
        return kernel @ alpha - 2 * sd, gradient

    best = pool[int(np.argmin(pool_lcb))]
    best_lcb = float(np.min(pool_lcb))
    starts = [best]
    distant = np.linalg.norm(pool - best, axis=1) >= .05
    if distant.any():
        starts.append(pool[np.argmin(np.where(distant, pool_lcb, np.inf))])
    warnings, improved = 0, False
    for start in starts:
        result = minimize(lcb, start, jac=True, method="SLSQP",
                          constraints=[{"type": "ineq", "fun": disk_constraint,
                                        "jac": disk_jacobian}],
                          options={"maxiter": 25, "ftol": 1e-10})
        warnings += int(not result.success)
        if not np.all(np.isfinite(result.x)) or np.linalg.norm(result.x) > 1 + 1e-8:
            continue
        candidate = result.x / max(1., np.linalg.norm(result.x))
        # Compare with the stable triangular-factor prediction used for the pool.
        mean, sd = predict(x, candidate[None, :], theta, lower, alpha)
        score = float((mean - 2 * sd)[0])
        if score < best_lcb - 1e-12:
            best, best_lcb, improved = candidate, score, True
    return best, warnings, int(improved)


def run_bo(evaluate, q, initial, seed, isotropic, refine, jitter=JITTER):
    rng = np.random.default_rng(seed)
    points = list(initial)
    scores = [evaluate(p) for p in points]
    theta = np.log([1, .4] if isotropic else [1, .4, .4])
    fits, warnings, changed = [], 0, 0
    started = time.perf_counter()
    while len(points) < BUDGET:
        count = len(points)
        x = np.array(points)
        theta, lower, alpha, status = fit(x, np.array(scores), theta, count % 6 == 0, jitter=jitter)
        if status >= 0:
            fits.append([count, status, *np.exp(theta).tolist()])
        pool = disk(rng, POOL) @ q.T
        mean, sd = predict(x, pool, theta, lower, alpha)
        acq = mean - 2 * sd
        if refine:
            candidate, failures, used = refined_candidate(x, pool, theta, lower, alpha, acq)
            warnings += failures
            changed += used
        else:
            candidate = pool[np.argmin(acq)]
        points.append(candidate)
        scores.append(evaluate(candidate))
    return points, scores, dict(fits=fits, local_warnings=warnings,
                                local_improved=changed, seconds=time.perf_counter()-started)

### CMA-ES

The reference CMA-ES update runs after each population of six. We rotate the Gaussian innovations explicitly and map them through the symmetric covariance square root. This pairs the random draws in physical coordinates while preserving the marginal sampling distribution. Infeasible proposals are rejected before evaluation. Every completed run uses 16 generations and 96 feasible evaluations.

In [ ]:
%%writefile -a master_code/geometry_pilot.py

def run_coupled_cma(evaluate, q, seed):
    """Use C^(1/2) Q z for matched draws, retaining the reference tell update."""
    state = CMAES([0., 0.], .4, popsize=POPULATION)
    rng = np.random.default_rng(seed)
    points, scores, states = [], [], []
    rejected = 0
    for _ in range(BUDGET // POPULATION):
        state.C.update_eigensystem(state.counteval, state.params.lazy_gap_evals)
        basis = np.asarray(state.C.eigenbasis)
        root = (basis * np.sqrt(state.C.eigenvalues)) @ basis.T
        population = []
        while len(population) < POPULATION:
            point = np.asarray(state.xmean) + state.sigma * (root @ (q @ rng.standard_normal(2)))
            if np.linalg.norm(point) > 1:
                rejected += 1
                if rejected > 10000:
                    raise RuntimeError("CMA rejection cap")
                continue
            population.append(point.tolist())
        values = [evaluate(p) for p in population]
        state.tell(population, values)
        points.extend(population)
        scores.extend(values)
        covariance = np.array([[state.C[i][j] for j in range(2)] for i in range(2)])
        eigvals, vectors = np.linalg.eigh(covariance)
        major_axis = vectors[:, -1]
        states.append(dict(call=state.counteval, ratio=float(eigvals[-1]/eigvals[0]),
                           axis_error=float(np.degrees(np.arccos(np.clip(abs(major_axis @ q[:, 0]), 0, 1)))),
                           sigma=state.sigma, covariance=covariance.tolist(), mean=state.xmean))
    assert state.counteval == BUDGET
    return points, scores, dict(states=states, rejections=rejected)

### Paired experiment and saving

Each block generates one hidden centre. It reuses that centre and matches the random streams across conditioning levels and orientations. The block also audits the best point available to the finite pool using clean scores outside optimisation. That audit does not influence the optimiser. The notebook runs blocks sequentially, so it does not require multiprocessing support in the Jupyter kernel.

In [ ]:
%%writefile -a master_code/geometry_pilot.py

def write_csv(path, rows):
    with path.open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


def run_block(task):
    seed, phase = task
    centre = rotate((.4, 0), random.Random(seed).uniform(0, math.tau))
    records, diagnostics = [], []
    condition_numbers = {"sensitivity": (100,), "replication": (10, 100)}.get(phase, KINDS)
    for k in condition_numbers:
        cma_paths = {}
        a, b = centre
        m = (1 + k) / 4 + a*a + k*b*b
        # Score the discrete proposal bank offline, separately from all solvers.
        canonical_bank = disk(np.random.default_rng(seed + 20000), (BUDGET-POPULATION)*POOL)
        floor = np.min(((canonical_bank[:,0]-a)**2 + k*(canonical_bank[:,1]-b)**2) / m)
        for degree in ANGLES:
            q = rotation(degree)
            valley = Valley(centre, math.radians(degree), k)
            cp, cs, cd = run_coupled_cma(valley.clean, q, seed + 30000)
            cma_paths[degree] = (cp, cs, cd)
            initial = cp[:POPULATION]
            if phase == "sensitivity":
                methods = []
                settings = [("ARD_local_jitter", False, True, 1e-5)]
            elif phase == "replication":
                methods = []
                settings = [("ARD_local", False, True, 1e-7), ("ISO_local", True, True, 1e-7)]
            else:
                uniform = disk(np.random.default_rng(seed + 10000), BUDGET) @ q.T
                methods = [("random", uniform, [valley.clean(p) for p in uniform], {}),
                           ("CMA", cp, cs, cd)]
                settings = [("ARD_pool", False, False, 1e-7), ("ARD_local", False, True, 1e-7)]
                if k == 100:
                    settings.extend([("ISO_pool", True, False, 1e-7), ("ISO_local", True, True, 1e-7)])
            for method, isotropic, refine, jitter in settings:
                points, scores, info = run_bo(
                    valley.clean, q, initial, seed + 20000, isotropic, refine, jitter=jitter
                )
                methods.append((method, points, scores, info))
            for method, points, scores, info in methods:
                assert len(points) == len(scores) == BUDGET
                assert np.all(np.isfinite(scores)) and min(scores) >= 0
                for call, (p, score, best) in enumerate(zip(points, scores, np.minimum.accumulate(scores)), 1):
                    records.append(dict(seed=seed,k=k,angle=degree,method=method,
                                        call=call,x=p[0],y=p[1],gap=score,best=best))
                diagnostics.append(dict(seed=seed,k=k,angle=degree,method=method,
                                        pool_floor=min(float(floor), min(cs[:POPULATION])), **info))
        x0, y0, state0 = cma_paths[0]
        x45, y45, state45 = cma_paths[45]
        q = rotation(45)
        np.testing.assert_allclose(np.asarray(x0) @ q.T, x45, rtol=0, atol=1e-10)
        np.testing.assert_allclose(y0, y45, rtol=0, atol=1e-11)
        assert state0["rejections"] == state45["rejections"]
        for s0, s45 in zip(state0["states"], state45["states"]):
            np.testing.assert_allclose(q @ np.array(s0["covariance"]) @ q.T,
                                       s45["covariance"], rtol=0, atol=1e-10)
    return records, diagnostics



def run_phase(phase, seeds):
    destination = OUT / phase
    destination.mkdir(parents=True, exist_ok=True)
    for seed in seeds:
        if (destination / f"{seed}.csv").exists() and (destination / f"{seed}.json").exists():
            print(f"Reusing {phase} block {seed}", flush=True)
            continue
        rows, diagnostics = run_block((seed, phase))
        write_csv(destination / f"{seed}.csv", rows)
        (destination / f"{seed}.json").write_text(
            json.dumps(diagnostics), encoding="utf-8"
        )
        print(f"Saved {phase} block {seed}", flush=True)

### Run the pilot

The assertions check the hidden minimum, objective rotation, and coupled CMA-ES paths. The pilot seeds are separate from the main and replication seeds.

In [ ]:
%%writefile -a master_code/geometry_pilot.py

def check_benchmark():
    q = rotation(45)
    centre = (.2, .1)
    points = disk(np.random.default_rng(47), 15)
    for k in KINDS:
        aligned = Valley(centre, 0., k)
        rotated = Valley(centre, math.pi / 4, k)
        assert aligned.clean(centre) < 1e-12
        assert rotated.clean(q @ centre) < 1e-12
        np.testing.assert_allclose(
            [aligned.clean(point) for point in points],
            [rotated.clean(q @ point) for point in points],
            rtol=0, atol=1e-12,
        )


check_benchmark()
run_phase("pilot", PILOT_SEEDS)

### Run the main experiment

The 24 blocks below provide the unit of replication for comparisons and bootstrap intervals. The random baseline and both ARD acquisitions run at all three condition numbers. The isotropic controls run at condition number 100.

In [ ]:
%%writefile -a master_code/geometry_pilot.py

run_phase("final", FINAL_SEEDS)

### Run the sensitivity and replication experiments

The first cell repeats refined ARD at condition number 100 with larger GP jitter and the same 24 blocks. The second runs the targeted follow-up at condition numbers 10 and 100 on independent seeds. Each follow-up can be disabled in the configuration cell.

In [ ]:
%%writefile -a master_code/geometry_pilot.py

if RUN_SENSITIVITY:
    run_phase("sensitivity", FINAL_SEEDS)

In [ ]:
%%writefile -a master_code/geometry_pilot.py

if RUN_REPLICATION:
    run_phase("replication", REPLICATION_SEEDS)

### Validate saved histories and compute paired summaries

The validation checks the loss and running best for every recorded point, the call budget and the shared initial information. A rotation contrast is computed within each block as rotated minus aligned. Positive values mean rotation increased the gap. Log contrasts describe ratios, and raw contrasts describe absolute changes. The bootstrap resamples whole blocks. The targeted replication is summarised separately.

In [ ]:
%%writefile -a master_code/geometry_pilot.py

def load_runs():
    histories, diagnostics = {}, {}
    phases = ("final", "sensitivity") if RUN_SENSITIVITY else ("final",)
    for phase in phases:
        for seed in SEEDS:
            grouped = defaultdict(list)
            centre = rotate((.4, 0), random.Random(seed).uniform(0, math.tau))
            with (OUT/phase/f"{seed}.csv").open(encoding="utf-8") as stream:
                for row in csv.DictReader(stream):
                    assert int(row["seed"]) == seed
                    grouped[int(row["k"]), int(row["angle"]), row["method"]].append(row)
            assert len(grouped) == (28 if phase == "final" else 2)
            for (k, angle, method), rows in grouped.items():
                key = seed, k, angle, method
                assert [int(r["call"]) for r in rows] == list(range(1, BUDGET+1))
                valley = Valley(centre, math.radians(angle), k)
                points = [[float(r["x"]), float(r["y"])] for r in rows]
                gaps = np.array([float(r["gap"]) for r in rows])
                np.testing.assert_allclose(gaps, [valley.clean(p) for p in points], rtol=1e-12, atol=1e-14)
                best = np.minimum.accumulate(gaps)
                np.testing.assert_allclose(best, [float(r["best"]) for r in rows], rtol=0, atol=0)
                histories[key] = best
                if phase == "final" and method.startswith(("ARD", "ISO")):
                    cma = grouped[k, angle, "CMA"][:POPULATION]
                    assert all((r["x"],r["y"]) == (s["x"],s["y"]) for r,s in zip(rows,cma))
                if phase == "sensitivity":
                    np.testing.assert_array_equal(best[:POPULATION], histories[seed,k,angle,"CMA"][:POPULATION])
            for item in json.loads((OUT/phase/f"{seed}.json").read_text(encoding="utf-8")):
                key = seed, item["k"], item["angle"], item["method"]
                diagnostics[key] = item
    assert histories.keys() == diagnostics.keys()
    return histories, diagnostics


def stat(difference, draws):
    lo, hi = np.quantile(difference[draws].mean(axis=1), [.025,.975])
    return dict(mean=float(difference.mean()), lo=float(lo), hi=float(hi))


def logs(values):
    return np.log10(np.maximum(values, FLOOR))


def analyse(histories, diagnostics):
    draws = np.random.default_rng(9301).integers(0, len(SEEDS), (10000,len(SEEDS)))

    def gaps(k, angle, method, call):
        return np.array([histories[s,k,angle,method][call-1] for s in SEEDS])

    def rotation(k, method, call):
        return logs(gaps(k,45,method,call))-logs(gaps(k,0,method,call))

    effects, interactions, acquisition, fits, covariances, conditioning = [], [], [], [], [], []
    for k in KINDS:
        methods = ["random","CMA","ARD_pool","ARD_local"]
        if k == 100:
            methods += ["ISO_pool", "ISO_local"]
            if RUN_SENSITIVITY:
                methods.append("ARD_local_jitter")
        for method in methods:
            for call in CHECKPOINTS:
                aligned, rotated = (gaps(k,a,method,call) for a in ANGLES)
                log_effect = stat(rotation(k,method,call), draws)
                effects.append(dict(k=k,method=method,call=call,aligned_median=np.median(aligned),
                                    rotated_median=np.median(rotated), **log_effect,
                                    **{"raw_"+n:v for n,v in stat(rotated-aligned,draws).items()}))
        for call in CHECKPOINTS:
            for method in ("ARD_pool","ARD_local"):
                d = rotation(k,method,call)-rotation(k,"CMA",call)
                interactions.append(dict(contrast="BO_minus_CMA_rotation",k=k,method=method,
                                         call=call,**stat(d,draws)))
                for angle in ANGLES:
                    d = logs(gaps(k,angle,method,call))-logs(gaps(k,angle,"CMA",call))
                    interactions.append(dict(contrast=f"BO_minus_CMA_at_{angle}",k=k,method=method,
                                             call=call,**stat(d,draws)))
            interactions.append(dict(contrast="local_minus_pool_rotation",k=k,method="ARD",call=call,
                                     **stat(rotation(k,"ARD_local",call)-rotation(k,"ARD_pool",call),draws)))
        for angle in ANGLES:
            pool = gaps(k,angle,"ARD_pool",BUDGET)
            local = gaps(k,angle,"ARD_local",BUDGET)
            floors = np.array([diagnostics[s,k,angle,"ARD_pool"]["pool_floor"] for s in SEEDS])
            assert np.all(pool >= floors-1e-12)
            acquisition.append(dict(k=k,angle=angle,pool_floor_median=np.median(floors),
                                    pool_hits_floor=np.isclose(pool,floors,rtol=1e-9,atol=1e-14).sum(),
                                    local_beats_floor=(local < floors-1e-14).sum(),
                                    local_beats_pool=(local < pool-1e-14).sum(),
                                    **stat(logs(local)-logs(pool),draws)))
        for method in methods[2:]:
            for angle in ANGLES:
                ds = [diagnostics[s,k,angle,method] for s in SEEDS]
                fs = [f for d in ds for f in d["fits"]]
                fits.append(dict(k=k,angle=angle,method=method,fits=len(fs),
                                 gp_warnings=sum(f[1]!=0 for f in fs),
                                 length_bounds=sum(any(v<=.020001 or v>=1.99999 for v in f[3:]) for f in fs),
                                 median_length_ratio=np.median([max(f[3:])/min(f[3:]) for d in ds for f in d["fits"][-1:]]),
                                 local_warnings=sum(d["local_warnings"] for d in ds),
                                 local_improved=sum(d["local_improved"] for d in ds),
                                 median_seconds=np.median([d["seconds"] for d in ds])))
        for call in CHECKPOINTS:
            ds = [diagnostics[s,k,0,"CMA"]["states"][call//POPULATION-1] for s in SEEDS]
            covariances.append(dict(k=k,call=call,median_ratio=np.median([d["ratio"] for d in ds]),
                                    median_axis_error=np.median([d["axis_error"] for d in ds]),
                                    median_sigma=np.median([d["sigma"] for d in ds])))
    for method in ("ARD_pool","ARD_local"):
        for call in CHECKPOINTS:
            narrow = rotation(100,method,call)-rotation(100,"CMA",call)
            sphere = rotation(1,method,call)-rotation(1,"CMA",call)
            interactions.append(dict(contrast="narrow_minus_sphere_relative_rotation",k=100,method=method,
                                     call=call,**stat(narrow-sphere,draws)))
    for call in CHECKPOINTS:
        checks = [("ARD_minus_ISO_rotation", "ISO_local")]
        if RUN_SENSITIVITY:
            checks.append(("jitter_change_in_rotation", "ARD_local_jitter"))
        for name, method in checks:
            d = rotation(100,method,call)-rotation(100,"ARD_local",call)
            if name == "ARD_minus_ISO_rotation":
                d = -d
            interactions.append(dict(contrast=name,k=100,method=method,call=call,**stat(d,draws)))
        for k in (10,100):
            for angle in ANGLES:
                changes = {}
                for method in ("random","CMA","ARD_pool","ARD_local"):
                    change = logs(gaps(k,angle,method,call))-logs(gaps(1,angle,method,call))
                    changes[method] = change
                    conditioning.append(dict(k=k,reference_k=1,angle=angle,call=call,
                                             method=method,contrast="within_method",**stat(change,draws)))
                for method in ("ARD_pool","ARD_local"):
                    conditioning.append(dict(k=k,reference_k=1,angle=angle,call=call,
                                             method=method,contrast="BO_minus_CMA",
                                             **stat(changes[method]-changes["CMA"],draws)))
    for name, rows in (("rotation-effects",effects),("interactions",interactions),("acquisition-audit",acquisition),
                       ("fit-diagnostics",fits),("covariance",covariances),("conditioning-effects",conditioning)):
        write_csv(OUT/f"{name}.csv", rows)


def analyse_replication():
    seeds = list(REPLICATION_SEEDS)
    draws = np.random.default_rng(9302).integers(0, len(seeds), (10000,len(seeds)))
    histories, diagnostics = {}, {}
    for seed in seeds:
        centre = rotate((.4, 0), random.Random(seed).uniform(0, math.tau))
        grouped = defaultdict(list)
        with (OUT/"replication"/f"{seed}.csv").open(encoding="utf-8") as stream:
            for row in csv.DictReader(stream):
                assert int(row["seed"]) == seed
                grouped[int(row["k"]), int(row["angle"]), row["method"]].append(row)
        assert len(grouped) == 8
        for (k, angle, method), rows in grouped.items():
            assert [int(r["call"]) for r in rows] == list(range(1, BUDGET+1))
            objective = Valley(centre, math.radians(angle), k)
            points = np.array([[float(r["x"]),float(r["y"])] for r in rows])
            gaps = np.array([float(r["gap"]) for r in rows])
            np.testing.assert_allclose(gaps,[objective.clean(p) for p in points],rtol=1e-12,atol=1e-14)
            best = np.minimum.accumulate(gaps)
            np.testing.assert_array_equal(best,[float(r["best"]) for r in rows])
            histories[seed,k,angle,method] = best
            initial = np.array([[float(r["x"]),float(r["y"])] for r in grouped[k,0,"ARD_local"][:POPULATION]])
            np.testing.assert_allclose(points[:POPULATION], initial @ rotation(angle).T, rtol=0, atol=1e-12)
        for item in json.loads((OUT/"replication"/f"{seed}.json").read_text(encoding="utf-8")):
            diagnostics[seed,item["k"],item["angle"],item["method"]] = item
    assert histories.keys() == diagnostics.keys()

    def terminal(k, angle, method, call):
        return np.array([histories[s,k,angle,method][call-1] for s in seeds])

    rows = []
    for call in (24, 96):
        effects = {}
        for k in (10, 100):
            for method in ("ARD_local", "ISO_local"):
                aligned, rotated = (terminal(k,a,method,call) for a in ANGLES)
                effect = logs(rotated)-logs(aligned)
                effects[k,method] = effect
                rows.append(dict(call=call,k=k,contrast="rotation",method=method,
                                 aligned_median=np.median(aligned),rotated_median=np.median(rotated),
                                 **stat(effect,draws),
                                 **{"raw_"+n:v for n,v in stat(rotated-aligned,draws).items()}))
            d = effects[k,"ARD_local"]-effects[k,"ISO_local"]
            rows.append(dict(call=call,k=k,contrast="ARD_minus_ISO_rotation",method="",
                             aligned_median="",rotated_median="",**stat(d,draws),
                             raw_mean="",raw_lo="",raw_hi=""))
            for angle in ANGLES:
                d = logs(terminal(k,angle,"ARD_local",call))-logs(terminal(k,angle,"ISO_local",call))
                rows.append(dict(call=call,k=k,contrast=f"ARD_minus_ISO_at_{angle}",method="",
                                 aligned_median="",rotated_median="",**stat(d,draws),
                                 raw_mean="",raw_lo="",raw_hi=""))
        d = effects[10,"ARD_local"]-effects[100,"ARD_local"]
        rows.append(dict(call=call,k="",contrast="k10_minus_k100_rotation",method="ARD_local",
                         aligned_median="",rotated_median="",**stat(d,draws),
                         raw_mean="",raw_lo="",raw_hi=""))
    write_csv(OUT/"replication-effects.csv",rows)
    print(f"Validated {len(histories)} targeted replication trajectories.",flush=True)

In [ ]:
%%writefile -a master_code/geometry_pilot.py

histories, diagnostics = load_runs()
analyse(histories, diagnostics)
print(f"Validated {len(histories)} main and sensitivity trajectories.")
if RUN_REPLICATION:
    analyse_replication()
print(f"Saved summary tables in {OUT.resolve()}")

### Plot convergence, including random search

Each curve is the pointwise median of the best gap so far across the 24 main blocks. All methods share a vertical scale across panels. Solid lines show aligned runs and dotted lines show rotated runs. The gray curves show uniform random search in every panel. For exactly overlapping pairs, the dotted curve is drawn in a slightly lighter shade so the pairing remains visible.

In [ ]:
%%writefile -a master_code/geometry_pilot.py

from matplotlib.colors import to_rgb

styles = {
    "random": ("Uniform random search", "#565b61"),
    "CMA": ("CMA-ES", "#9b4d18"),
    "ARD_pool": ("ARD BO, pool", "#567a91"),
    "ARD_local": ("ARD BO, refined", "#145ec1"),
    "ISO_pool": ("Isotropic BO, pool", "#75773a"),
    "ISO_local": ("Isotropic BO, refined", "#187759"),
}
calls = np.arange(1, BUDGET + 1)
fig, axes = plt.subplots(1, len(KINDS), figsize=(15, 4.8), sharey=True)

for ax, k in zip(axes, KINDS):
    for method, (_, color) in styles.items():
        if (SEEDS[0], k, 0, method) not in histories:
            continue
        for angle, linestyle in ((0, "-"), (45, ":")):
            values = np.median([histories[seed, k, angle, method] for seed in SEEDS], axis=0)
            curve_color = color if angle == 0 else tuple(.78*c + .22 for c in to_rgb(color))
            ax.step(calls, np.maximum(values, FLOOR), where="post", color=curve_color,
                    linewidth=1.8 if angle == 0 else 2.4, linestyle=linestyle)
    ax.set_title(f"Condition number {k}")
    ax.set_yscale("log")
    ax.set_xlabel("Objective calls")
    ax.set_xticks([6, 24, 48, 72, 96])
    ax.grid(axis="y", which="major", alpha=.25)

axes[0].set_ylabel("Median best true gap")
method_handles = [Line2D([], [], color=color, lw=2, label=name) for name, color in styles.values()]
angle_handles = [Line2D([], [], color="#333333", ls=style, lw=2, label=label)
                 for style, label in (("-", "0° aligned"), (":", "45° rotated"))]
fig.legend(handles=method_handles + angle_handles, loc="lower center", ncol=4, frameon=False)
fig.suptitle("Stage A: noiseless search on paired quadratic valleys")
fig.tight_layout(rect=(0, .15, 1, .93))
fig.savefig(OUT / "geometry-convergence.png", dpi=200, bbox_inches="tight")
plt.show()

### Display terminal comparisons

The table includes the random-search baseline. Intervals here describe the mean paired log rotation penalty. The CSV files also record raw gaps, direct BO-versus-CMA interactions, acquisition-floor diagnostics and the covariance diagnostics.

In [ ]:
%%writefile -a master_code/geometry_pilot.py

with (OUT / "rotation-effects.csv").open(encoding="utf-8") as stream:
    terminal = [row for row in csv.DictReader(stream) if int(row["call"]) == BUDGET]

table = [
    "| κ | Method | Aligned median gap | Rotated median gap | Mean log10 penalty [95% interval] |",
    "|---:|:---|---:|---:|---:|",
]
for row in terminal:
    label = styles.get(row["method"], ("Refined ARD, higher jitter", None))[0]
    aligned, rotated = float(row["aligned_median"]), float(row["rotated_median"])
    penalty, lo, hi = (float(row[key]) for key in ("mean", "lo", "hi"))
    penalty_text = ("≈ 0 (roundoff)" if max(abs(penalty), abs(lo), abs(hi)) < 1e-10
                    else f"{penalty:+.3f} [{lo:+.3f}, {hi:+.3f}]")
    table.append(f"| {row['k']} | {label} | {aligned:.3g} | {rotated:.3g} | {penalty_text} |")
display(Markdown("\n".join(table)))

The outputs folder contains the figure and CSV tables. `rotation-effects.csv` summarises the aligned and rotated runs. `conditioning-effects.csv` and `interactions.csv` compare how geometry changes the relative performance of the methods. `acquisition-audit.csv` measures how often the pool sets a floor. `fit-diagnostics.csv` and `covariance.csv` describe the GP fits and CMA-ES covariance. If enabled, `replication-effects.csv` contains the independent follow-up. The phase folders contain individual runs and diagnostics.

In [ ]:
run_stage("geometry_pilot")

## Stage A - conditioning, rotation and bending

Source reused from `stage-a-straight-and-curved.ipynb`. The configuration above controls execution. Code cells in this section prepare the stage script; the final cell runs it when selected.

In [ ]:
%%writefile master_code/stage_a.py
from pathlib import Path
import os

MASTER_PROFILE = globals().get("MASTER_PROFILE", os.environ.get("OPT_PROFILE", "smoke"))
MASTER_ROOT = Path(globals().get("MASTER_ROOT", os.environ.get("OPT_RESULTS_ROOT", "optimisation_results"))).resolve()
MASTER_INCLUDE_TRANSFER = globals().get("MASTER_INCLUDE_TRANSFER", os.environ.get("OPT_INCLUDE_TRANSFER", "1") == "1")
if MASTER_PROFILE not in ("full", "smoke"):
    raise ValueError("Choose full or smoke.")

In [ ]:
%%writefile -a master_code/stage_a.py

import csv
import json
import math
import os
import random
import runpy
import sys
import time
from collections import defaultdict
from dataclasses import dataclass
from importlib.metadata import distribution, version
from math import cos, sin, hypot, sqrt, tau
from pathlib import Path

for variable in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS"):
    os.environ[variable] = "1"

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display
from matplotlib.lines import Line2D
from scipy.linalg import cholesky, cho_solve, solve_triangular
from scipy.optimize import minimize
from scipy.spatial.distance import cdist

# Load the pinned pure-Python reference CMA-ES class.
pure_cma = distribution("cma").locate_file("cma/more_algorithms/purecma.py")
CMAES = runpy.run_path(str(pure_cma))["CMAES"]

OUT = MASTER_ROOT / "stage_a" / MASTER_PROFILE
KINDS = (1, 10, 100)
ANGLES = (0, 45)
BEND = 0.75  # One fixed bend for every curved case, in unit-disk coordinates.
BUDGET = 96
POPULATION = 6
POOL = 1024
JITTER = 1e-7
PILOT_SEEDS = range(9000, 9002 if MASTER_PROFILE == "smoke" else 9002)
FINAL_SEEDS = range(9100, 9102 if MASTER_PROFILE == "smoke" else 9124)
REPLICATION_SEEDS = range(9200, 9202 if MASTER_PROFILE == "smoke" else 9224)
BUDGET_CHECK_SEEDS = range(9400, 9402 if MASTER_PROFILE == "smoke" else 9408)
BUDGET_CHECKPOINTS = (24, 48, 96, 192)
BUDGET_METHODS = ("random", "CMA", "ARD_local", "ISO_local")
CHECKPOINTS = (24, 48, 96)
SEEDS = list(FINAL_SEEDS)
FLOOR = 1e-14

# Main runs are always analysed. Turn off any follow-up for a shorter run.
RUN_SENSITIVITY = True
RUN_REPLICATION = True
RUN_BUDGET_CHECK = True

print("Python", sys.version.split()[0])
print({package: version(package) for package in ("numpy", "scipy", "cma", "matplotlib")})

### Objectives and random-search baseline

In coordinates `(u, v)` centred at the minimiser and expressed in the valley's original frame, the raw loss is `u**2 + k * (v - bend * u**2)**2`. The straight case uses `bend = 0`. The curved case fixes `bend = 0.75` for every condition number. The valley floor follows `v = bend * u**2`. Its direction varies with position, while the condition number of the Hessian **at the minimum** is still `k`. Away from the minimum, curvature varies in the curved case.

Each objective is divided by its own analytic mean under uniform sampling on the disk. The normalisation keeps the same interpretation of gap across cases. Bending can still change how much of the disk lies below a given gap threshold. The true minimum is zero, so the best observation equals the best true gap here. For paired rotations, the centre, points and objective rotate together. Uniform random search rotates the same proposal sequence with the objective, so its paired curves should overlap.

In [ ]:
%%writefile -a master_code/stage_a.py

def rotate(point, angle):
    x, y = point
    return cos(angle) * x - sin(angle) * y, sin(angle) * x + cos(angle) * y


@dataclass(frozen=True)
class Valley:
    centre: tuple
    angle: float
    curvature_ratio: float = 100.0
    bend: float = 0.0

    @property
    def normaliser(self):
        # Under uniform disk sampling E[X^2] = 1/4 and E[X^4] = 1/8.
        # Terms odd in either coordinate integrate to zero.
        a, b = self.centre
        if self.bend == 0:
            k = self.curvature_ratio
            return (1 + k) / 4 + a*a + k*b*b
        eu2 = .25 + a*a
        ev2 = .25 + b*b
        eu4 = .125 + 1.5*a*a + a**4
        return eu2 + self.curvature_ratio * (
            ev2 + 2*self.bend*b*eu2 + self.bend**2*eu4
        )

    def canonical_gap(self, x, y):
        a, b = self.centre
        u, v = x - a, y - b
        return (u**2 + self.curvature_ratio*(v - self.bend*u**2)**2) / self.normaliser

    def clean(self, point):
        if hypot(*point) > 1.0 + 1e-12:
            raise ValueError("Point outside the unit disk")
        x, y = rotate(point, -self.angle)
        return self.canonical_gap(x, y)

    def observe(self, point, sigma, rng):
        return self.clean(point) + sigma * rng.gauss(0.0, 1.0)


def rotation(degree):
    angle = math.radians(degree)
    return np.array([rotate((1, 0), angle), rotate((0, 1), angle)]).T


def disk(rng, size):
    u = rng.random((size, 2))
    angle = 2 * np.pi * u[:, 0]
    return np.sqrt(u[:, 1:]) * np.column_stack((np.cos(angle), np.sin(angle)))

### Bayesian optimisation

Both BO variants refit GP hyperparameters every six calls and minimise the same lower confidence bound. Pool BO evaluates its best candidate among 1,024 draws. Refined BO also starts a constrained local search from two promising pool points. Both have the same objective-call budget. The optional refinement spends extra computation evaluating the surrogate.

In [ ]:
%%writefile -a master_code/stage_a.py

def covariance(x, y, theta):
    lengths = np.exp(theta[1:])
    return np.exp(theta[0] - .5 * cdist(x / lengths, y / lengths, "sqeuclidean"))


def fit(x, values, theta, refit, noise_sd=0.0, jitter=JITTER):
    output_scale = values.std() or 1
    y = (values - values.mean()) / output_scale
    ident = np.eye(len(x))

    def factor(parameters):
        signal = covariance(x, x, parameters)
        lower = cholesky(signal + (jitter + (noise_sd/output_scale)**2) * ident, lower=True)
        return signal, lower, cho_solve((lower, True), y)

    def likelihood(parameters):
        signal, lower, alpha = factor(parameters)
        loss = .5 * (y @ alpha) + np.log(np.diag(lower)).sum()
        weights = .5 * (cho_solve((lower, True), ident) - np.outer(alpha, alpha)) * signal
        gradient = [weights.sum()]
        if len(parameters) == 2:
            gradient.append((weights * cdist(x, x, "sqeuclidean")).sum() / np.exp(2*parameters[1]))
        else:
            for axis, log_length in enumerate(parameters[1:]):
                diff = x[:, axis, None] - x[:, axis]
                gradient.append((weights * diff**2).sum() / np.exp(2*log_length))
        return loss, np.array(gradient)

    status = -1
    if refit:
        bounds = [(-6, 6)] + [(math.log(.02), math.log(2))] * (len(theta)-1)
        result = minimize(likelihood, theta, jac=True, bounds=bounds, method="L-BFGS-B",
                          options={"maxiter": 50})
        theta, status = result.x, int(result.status)
    _, lower, alpha = factor(theta)
    return theta, lower, alpha, status


def predict(x, candidates, theta, lower, alpha):
    cross = covariance(x, candidates, theta)
    projected = solve_triangular(lower, cross, lower=True, check_finite=False)
    variance = np.maximum(0, np.exp(theta[0]) - np.einsum("ij,ij->j", projected, projected))
    return cross.T @ alpha, np.sqrt(variance)

In [ ]:
%%writefile -a master_code/stage_a.py

def disk_constraint(point):
    return 1 - point @ point


def disk_jacobian(point):
    return -2 * point


def refined_candidate(x, pool, theta, lower, alpha, pool_lcb):
    """Refine two promising pool candidates with the same fitted LCB."""
    precision = np.broadcast_to(np.exp(-2 * theta[1:]), (2,))
    amplitude = np.exp(theta[0])

    def lcb(p):
        delta = x - p
        kernel = amplitude * np.exp(-.5 * ((delta ** 2) @ precision))
        derivative = kernel[:, None] * delta * precision
        projected = solve_triangular(lower, kernel, lower=True, check_finite=False)
        weights = solve_triangular(lower.T, projected, lower=False, check_finite=False)
        # A tiny variance floor avoids an undefined derivative at zero.
        variance = amplitude - projected @ projected
        sd = np.sqrt(max(variance, 1e-20))
        mean_gradient = derivative.T @ alpha
        gradient = mean_gradient + 2 * (derivative.T @ weights) / sd if variance > 1e-20 else mean_gradient
        return kernel @ alpha - 2 * sd, gradient

    best = pool[int(np.argmin(pool_lcb))]
    best_lcb = float(np.min(pool_lcb))
    starts = [best]
    distant = np.linalg.norm(pool - best, axis=1) >= .05
    if distant.any():
        starts.append(pool[np.argmin(np.where(distant, pool_lcb, np.inf))])
    warnings, improved = 0, False
    for start in starts:
        result = minimize(lcb, start, jac=True, method="SLSQP",
                          constraints=[{"type": "ineq", "fun": disk_constraint,
                                        "jac": disk_jacobian}],
                          options={"maxiter": 25, "ftol": 1e-10})
        warnings += int(not result.success)
        if not np.all(np.isfinite(result.x)) or np.linalg.norm(result.x) > 1 + 1e-8:
            continue
        candidate = result.x / max(1., np.linalg.norm(result.x))
        # Compare with the stable triangular-factor prediction used for the pool.
        mean, sd = predict(x, candidate[None, :], theta, lower, alpha)
        score = float((mean - 2 * sd)[0])
        if score < best_lcb - 1e-12:
            best, best_lcb, improved = candidate, score, True
    return best, warnings, int(improved)


def run_bo(evaluate, q, initial, seed, isotropic, refine, jitter=JITTER, budget=BUDGET):
    rng = np.random.default_rng(seed)
    points = list(initial)
    scores = [evaluate(p) for p in points]
    theta = np.log([1, .4] if isotropic else [1, .4, .4])
    fits, warnings, changed = [], 0, 0
    started = time.perf_counter()
    while len(points) < budget:
        count = len(points)
        x = np.array(points)
        theta, lower, alpha, status = fit(x, np.array(scores), theta, count % 6 == 0, jitter=jitter)
        if status >= 0:
            fits.append([count, status, *np.exp(theta).tolist()])
        pool = disk(rng, POOL) @ q.T
        mean, sd = predict(x, pool, theta, lower, alpha)
        acq = mean - 2 * sd
        if refine:
            candidate, failures, used = refined_candidate(x, pool, theta, lower, alpha, acq)
            warnings += failures
            changed += used
        else:
            candidate = pool[np.argmin(acq)]
        points.append(candidate)
        scores.append(evaluate(candidate))
    return points, scores, dict(fits=fits, local_warnings=warnings,
                                local_improved=changed, seconds=time.perf_counter()-started)

### CMA-ES

The reference CMA-ES update runs after each population of six. We rotate the Gaussian innovations explicitly and map them through the symmetric covariance square root. This pairs the random draws in physical coordinates while preserving the marginal sampling distribution. Infeasible proposals are rejected before evaluation. A 96-call run uses 16 generations and a 192-call run uses 32.

In [ ]:
%%writefile -a master_code/stage_a.py

def run_coupled_cma(evaluate, q, seed, budget=BUDGET):
    """Use C^(1/2) Q z for matched draws, retaining the reference tell update."""
    state = CMAES([0., 0.], .4, popsize=POPULATION)
    rng = np.random.default_rng(seed)
    points, scores, states = [], [], []
    rejected = 0
    for _ in range(budget // POPULATION):
        state.C.update_eigensystem(state.counteval, state.params.lazy_gap_evals)
        basis = np.asarray(state.C.eigenbasis)
        root = (basis * np.sqrt(state.C.eigenvalues)) @ basis.T
        population = []
        while len(population) < POPULATION:
            point = np.asarray(state.xmean) + state.sigma * (root @ (q @ rng.standard_normal(2)))
            if np.linalg.norm(point) > 1:
                rejected += 1
                if rejected > 10000:
                    raise RuntimeError("CMA rejection cap")
                continue
            population.append(point.tolist())
        values = [evaluate(p) for p in population]
        state.tell(population, values)
        points.extend(population)
        scores.extend(values)
        covariance = np.array([[state.C[i][j] for j in range(2)] for i in range(2)])
        eigvals, vectors = np.linalg.eigh(covariance)
        major_axis = vectors[:, -1]
        states.append(dict(call=state.counteval, ratio=float(eigvals[-1]/eigvals[0]),
                           axis_error=float(np.degrees(np.arccos(np.clip(abs(major_axis @ q[:, 0]), 0, 1)))),
                           sigma=state.sigma, covariance=covariance.tolist(), mean=state.xmean))
    assert state.counteval == budget
    return points, scores, dict(states=states, rejections=rejected)

### Paired experiment and saving

Each block generates one hidden centre. It reuses that centre and matches the random streams across conditioning levels and orientations. The block also audits the best point available to the finite pool using clean scores outside optimisation. That audit does not influence the optimiser. The notebook runs blocks sequentially, so it does not require multiprocessing support in the Jupyter kernel.

In [ ]:
%%writefile -a master_code/stage_a.py

def write_csv(path, rows):
    with path.open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


def run_block(task, budget=BUDGET):
    seed, phase = task
    bend = BEND if phase in ("curved", "pilot-curved") else 0.
    if budget <= POPULATION or budget % POPULATION:
        raise ValueError("Use a budget of at least two complete CMA-ES generations.")
    centre = rotate((.4, 0), random.Random(seed).uniform(0, math.tau))
    records, diagnostics = [], []
    condition_numbers = {"sensitivity": (100,), "replication": (10, 100)}.get(phase, KINDS)
    for k in condition_numbers:
        cma_paths = {}
        # Score the discrete proposal bank offline, separately from all solvers.
        canonical_bank = disk(np.random.default_rng(seed + 20000), (budget-POPULATION)*POOL)
        canonical_objective = Valley(centre, 0., k, bend)
        floor = np.min(canonical_objective.canonical_gap(canonical_bank[:, 0], canonical_bank[:, 1]))
        for degree in ANGLES:
            q = rotation(degree)
            valley = Valley(centre, math.radians(degree), k, bend)
            cp, cs, cd = run_coupled_cma(valley.clean, q, seed + 30000, budget=budget)
            cma_paths[degree] = (cp, cs, cd)
            initial = cp[:POPULATION]
            if phase == "sensitivity":
                methods = []
                settings = [("ARD_local_jitter", False, True, 1e-5)]
            elif phase == "replication":
                methods = []
                settings = [("ARD_local", False, True, 1e-7), ("ISO_local", True, True, 1e-7)]
            else:
                uniform = disk(np.random.default_rng(seed + 10000), budget) @ q.T
                methods = [("random", uniform, [valley.clean(p) for p in uniform], {}),
                           ("CMA", cp, cs, cd)]
                settings = [("ARD_local", False, True, JITTER), ("ISO_local", True, True, JITTER)]
                if phase != "budget":
                    settings += [("ARD_pool", False, False, JITTER), ("ISO_pool", True, False, JITTER)]
            for method, isotropic, refine, jitter in settings:
                points, scores, info = run_bo(
                    valley.clean, q, initial, seed + 20000, isotropic, refine, jitter=jitter, budget=budget
                )
                methods.append((method, points, scores, info))
            for method, points, scores, info in methods:
                assert len(points) == len(scores) == budget
                assert np.all(np.isfinite(scores)) and min(scores) >= 0
                for call, (p, score, best) in enumerate(zip(points, scores, np.minimum.accumulate(scores)), 1):
                    records.append(dict(seed=seed,k=k,angle=degree,method=method,
                                        call=call,x=p[0],y=p[1],gap=score,best=best))
                diagnostics.append(dict(seed=seed,k=k,angle=degree,method=method,
                                        pool_floor=min(float(floor), min(cs[:POPULATION])), **info))
        x0, y0, state0 = cma_paths[0]
        x45, y45, state45 = cma_paths[45]
        q = rotation(45)
        np.testing.assert_allclose(np.asarray(x0) @ q.T, x45, rtol=0, atol=1e-10)
        np.testing.assert_allclose(y0, y45, rtol=0, atol=1e-11)
        assert state0["rejections"] == state45["rejections"]
        for s0, s45 in zip(state0["states"], state45["states"]):
            np.testing.assert_allclose(q @ np.array(s0["covariance"]) @ q.T,
                                       s45["covariance"], rtol=0, atol=1e-10)
    return records, diagnostics



def run_phase(phase, seeds, budget=BUDGET):
    destination = OUT / phase
    destination.mkdir(parents=True, exist_ok=True)
    for seed in seeds:
        if (destination / f"{seed}.csv").exists() and (destination / f"{seed}.json").exists():
            print(f"Reusing {phase} block {seed}", flush=True)
            continue
        rows, diagnostics = run_block((seed, phase), budget=budget)
        write_csv(destination / f"{seed}.csv", rows)
        (destination / f"{seed}.json").write_text(
            json.dumps(diagnostics), encoding="utf-8"
        )
        print(f"Saved {phase} block {seed}", flush=True)

### Run the pilot

The assertions check the hidden minimum, objective rotation, and coupled CMA-ES paths. The pilot seeds are separate from the main and replication seeds.

In [ ]:
%%writefile -a master_code/stage_a.py

def check_benchmark():
    q = rotation(45)
    centre = (.2, .1)
    points = disk(np.random.default_rng(47), 15)
    for bend in (0., BEND):
        for k in KINDS:
            aligned = Valley(centre, 0., k, bend)
            rotated = Valley(centre, math.pi / 4, k, bend)
            assert aligned.clean(centre) < 1e-12
            assert rotated.clean(q @ centre) < 1e-12
            np.testing.assert_allclose(
                [aligned.clean(point) for point in points],
                [rotated.clean(q @ point) for point in points],
                rtol=0, atol=1e-12,
            )


check_benchmark()
run_phase("pilot", PILOT_SEEDS)
run_phase("pilot-curved", PILOT_SEEDS)

### Run the main experiment

The same 24 blocks provide the unit of replication for every comparison. Both straight and curved runs use random search, CMA-ES, and all four BO configurations at each condition number and rotation. The hidden centre and random streams are matched across shapes. The curved run writes to a separate directory.

In [ ]:
%%writefile -a master_code/stage_a.py

run_phase("final", FINAL_SEEDS)

In [ ]:
%%writefile -a master_code/stage_a.py

run_phase("curved", FINAL_SEEDS)

### Run the sensitivity and replication experiments

The first cell repeats refined ARD at condition number 100 with larger GP jitter and the same 24 blocks. The second runs the targeted follow-up at condition numbers 10 and 100 on independent seeds. Each follow-up can be disabled in the configuration cell.

In [ ]:
%%writefile -a master_code/stage_a.py

if RUN_SENSITIVITY:
    run_phase("sensitivity", FINAL_SEEDS)

In [ ]:
%%writefile -a master_code/stage_a.py

if RUN_REPLICATION:
    run_phase("replication", REPLICATION_SEEDS)

### Run the quick budget check

Run uniform random search, CMA-ES, and both refined BO kernels to 192 calls on eight separate instance blocks. Read the 24-, 48-, 96- and 192-call results from the same runs. Each checkpoint completes a CMA-ES generation. Compare checkpoints within these eight blocks. The main 96-call analysis still uses its 24 blocks. This is a sensitivity check, and its eight instances give limited precision for rotation effects.

In [ ]:
%%writefile -a master_code/stage_a.py

if RUN_BUDGET_CHECK:
    run_phase("budget", BUDGET_CHECK_SEEDS, budget=max(BUDGET_CHECKPOINTS))

### Validate saved histories and compute paired summaries

The validation checks the loss and running best for every recorded point, the call budget and the shared initial information. A rotation contrast is computed within each block as rotated minus aligned. Positive values mean rotation increased the gap. Log contrasts describe ratios, and raw contrasts describe absolute changes. The bootstrap resamples whole blocks. The targeted replication is summarised separately.

In [ ]:
%%writefile -a master_code/stage_a.py

def load_runs(phases, seeds, budget=BUDGET):
    histories, diagnostics = {}, {}
    expected = {"final": 36, "curved": 36, "budget": 24, "sensitivity": 2}
    for phase in phases:
        bend = BEND if phase == "curved" else 0.
        for seed in seeds:
            grouped = defaultdict(list)
            centre = rotate((.4, 0), random.Random(seed).uniform(0, math.tau))
            with (OUT/phase/f"{seed}.csv").open(encoding="utf-8") as stream:
                for row in csv.DictReader(stream):
                    assert int(row["seed"]) == seed
                    grouped[int(row["k"]), int(row["angle"]), row["method"]].append(row)
            assert len(grouped) == expected[phase]
            for (k, angle, method), rows in grouped.items():
                key = seed, k, angle, method
                assert [int(r["call"]) for r in rows] == list(range(1, budget+1))
                valley = Valley(centre, math.radians(angle), k, bend)
                points = [[float(r["x"]), float(r["y"])] for r in rows]
                gaps = np.array([float(r["gap"]) for r in rows])
                np.testing.assert_allclose(gaps, [valley.clean(p) for p in points], rtol=1e-12, atol=1e-14)
                best = np.minimum.accumulate(gaps)
                np.testing.assert_allclose(best, [float(r["best"]) for r in rows], rtol=0, atol=0)
                histories[key] = best
                if phase in ("final", "curved", "budget") and method.startswith(("ARD", "ISO")):
                    cma = grouped[k, angle, "CMA"][:POPULATION]
                    assert all((r["x"],r["y"]) == (s["x"],s["y"]) for r,s in zip(rows,cma))
                if phase == "sensitivity":
                    np.testing.assert_array_equal(best[:POPULATION], histories[seed,k,angle,"CMA"][:POPULATION])
            for item in json.loads((OUT/phase/f"{seed}.json").read_text(encoding="utf-8")):
                key = seed, item["k"], item["angle"], item["method"]
                diagnostics[key] = item
    assert histories.keys() == diagnostics.keys()
    return histories, diagnostics


def stat(difference, draws):
    lo, hi = np.quantile(difference[draws].mean(axis=1), [.025,.975])
    return dict(mean=float(difference.mean()), lo=float(lo), hi=float(hi))


def logs(values):
    return np.log10(np.maximum(values, FLOOR))


def analyse(histories, diagnostics, output_dir, include_jitter=False):
    output_dir.mkdir(parents=True, exist_ok=True)
    draws = np.random.default_rng(9301).integers(0, len(SEEDS), (10000,len(SEEDS)))

    def gaps(k, angle, method, call):
        return np.array([histories[s,k,angle,method][call-1] for s in SEEDS])

    def rotation(k, method, call):
        return logs(gaps(k,45,method,call))-logs(gaps(k,0,method,call))

    effects, interactions, acquisition, fits, covariances, conditioning = [], [], [], [], [], []
    for k in KINDS:
        methods = ["random", "CMA", "ARD_pool", "ARD_local", "ISO_pool", "ISO_local"]
        if k == 100 and include_jitter:
            methods.append("ARD_local_jitter")
        for method in methods:
            for call in CHECKPOINTS:
                aligned, rotated = (gaps(k,a,method,call) for a in ANGLES)
                log_effect = stat(rotation(k,method,call), draws)
                effects.append(dict(k=k,method=method,call=call,aligned_median=np.median(aligned),
                                    rotated_median=np.median(rotated), **log_effect,
                                    **{"raw_"+n:v for n,v in stat(rotated-aligned,draws).items()}))
        for call in CHECKPOINTS:
            for method in ("ARD_pool","ARD_local"):
                d = rotation(k,method,call)-rotation(k,"CMA",call)
                interactions.append(dict(contrast="BO_minus_CMA_rotation",k=k,method=method,
                                         call=call,**stat(d,draws)))
                for angle in ANGLES:
                    d = logs(gaps(k,angle,method,call))-logs(gaps(k,angle,"CMA",call))
                    interactions.append(dict(contrast=f"BO_minus_CMA_at_{angle}",k=k,method=method,
                                             call=call,**stat(d,draws)))
            interactions.append(dict(contrast="local_minus_pool_rotation",k=k,method="ARD",call=call,
                                     **stat(rotation(k,"ARD_local",call)-rotation(k,"ARD_pool",call),draws)))
        for angle in ANGLES:
            pool = gaps(k,angle,"ARD_pool",BUDGET)
            local = gaps(k,angle,"ARD_local",BUDGET)
            floors = np.array([diagnostics[s,k,angle,"ARD_pool"]["pool_floor"] for s in SEEDS])
            assert np.all(pool >= floors-1e-12)
            acquisition.append(dict(k=k,angle=angle,pool_floor_median=np.median(floors),
                                    pool_hits_floor=np.isclose(pool,floors,rtol=1e-9,atol=1e-14).sum(),
                                    local_beats_floor=(local < floors-1e-14).sum(),
                                    local_beats_pool=(local < pool-1e-14).sum(),
                                    **stat(logs(local)-logs(pool),draws)))
        for method in methods[2:]:
            for angle in ANGLES:
                ds = [diagnostics[s,k,angle,method] for s in SEEDS]
                fs = [f for d in ds for f in d["fits"]]
                fits.append(dict(k=k,angle=angle,method=method,fits=len(fs),
                                 gp_warnings=sum(f[1]!=0 for f in fs),
                                 length_bounds=sum(any(v<=.020001 or v>=1.99999 for v in f[3:]) for f in fs),
                                 median_length_ratio=np.median([max(f[3:])/min(f[3:]) for d in ds for f in d["fits"][-1:]]),
                                 local_warnings=sum(d["local_warnings"] for d in ds),
                                 local_improved=sum(d["local_improved"] for d in ds),
                                 median_seconds=np.median([d["seconds"] for d in ds])))
        for call in CHECKPOINTS:
            ds = [diagnostics[s,k,0,"CMA"]["states"][call//POPULATION-1] for s in SEEDS]
            covariances.append(dict(k=k,call=call,median_ratio=np.median([d["ratio"] for d in ds]),
                                    median_axis_error=np.median([d["axis_error"] for d in ds]),
                                    median_sigma=np.median([d["sigma"] for d in ds])))
    for method in ("ARD_pool","ARD_local"):
        for call in CHECKPOINTS:
            narrow = rotation(100,method,call)-rotation(100,"CMA",call)
            sphere = rotation(1,method,call)-rotation(1,"CMA",call)
            interactions.append(dict(contrast="narrow_minus_sphere_relative_rotation",k=100,method=method,
                                     call=call,**stat(narrow-sphere,draws)))
    for call in CHECKPOINTS:
        for k in KINDS:
            for mode in ("pool", "local"):
                d = rotation(k, f"ARD_{mode}", call) - rotation(k, f"ISO_{mode}", call)
                interactions.append(dict(contrast="ARD_minus_ISO_rotation", k=k,
                                         method=mode, call=call, **stat(d, draws)))
        d = rotation(10, "ARD_local", call) - rotation(100, "ARD_local", call)
        interactions.append(dict(contrast="k10_minus_k100_rotation", k=10,
                                 method="ARD_local", call=call, **stat(d, draws)))
        if include_jitter:
            d = rotation(100, "ARD_local_jitter", call) - rotation(100, "ARD_local", call)
            interactions.append(dict(contrast="jitter_change_in_rotation", k=100,
                                     method="ARD_local_jitter", call=call, **stat(d, draws)))
        for k in (10,100):
            for angle in ANGLES:
                changes = {}
                for method in ("random","CMA","ARD_pool","ARD_local"):
                    change = logs(gaps(k,angle,method,call))-logs(gaps(1,angle,method,call))
                    changes[method] = change
                    conditioning.append(dict(k=k,reference_k=1,angle=angle,call=call,
                                             method=method,contrast="within_method",**stat(change,draws)))
                for method in ("ARD_pool","ARD_local"):
                    conditioning.append(dict(k=k,reference_k=1,angle=angle,call=call,
                                             method=method,contrast="BO_minus_CMA",
                                             **stat(changes[method]-changes["CMA"],draws)))
    for name, rows in (("rotation-effects",effects),("interactions",interactions),("acquisition-audit",acquisition),
                       ("fit-diagnostics",fits),("covariance",covariances),("conditioning-effects",conditioning)):
        write_csv(output_dir / f"{name}.csv", rows)


def analyse_replication():
    seeds = list(REPLICATION_SEEDS)
    draws = np.random.default_rng(9302).integers(0, len(seeds), (10000,len(seeds)))
    histories, diagnostics = {}, {}
    for seed in seeds:
        centre = rotate((.4, 0), random.Random(seed).uniform(0, math.tau))
        grouped = defaultdict(list)
        with (OUT/"replication"/f"{seed}.csv").open(encoding="utf-8") as stream:
            for row in csv.DictReader(stream):
                assert int(row["seed"]) == seed
                grouped[int(row["k"]), int(row["angle"]), row["method"]].append(row)
        assert len(grouped) == 8
        for (k, angle, method), rows in grouped.items():
            assert [int(r["call"]) for r in rows] == list(range(1, BUDGET+1))
            objective = Valley(centre, math.radians(angle), k)
            points = np.array([[float(r["x"]),float(r["y"])] for r in rows])
            gaps = np.array([float(r["gap"]) for r in rows])
            np.testing.assert_allclose(gaps,[objective.clean(p) for p in points],rtol=1e-12,atol=1e-14)
            best = np.minimum.accumulate(gaps)
            np.testing.assert_array_equal(best,[float(r["best"]) for r in rows])
            histories[seed,k,angle,method] = best
            initial = np.array([[float(r["x"]),float(r["y"])] for r in grouped[k,0,"ARD_local"][:POPULATION]])
            np.testing.assert_allclose(points[:POPULATION], initial @ rotation(angle).T, rtol=0, atol=1e-12)
        for item in json.loads((OUT/"replication"/f"{seed}.json").read_text(encoding="utf-8")):
            diagnostics[seed,item["k"],item["angle"],item["method"]] = item
    assert histories.keys() == diagnostics.keys()

    def terminal(k, angle, method, call):
        return np.array([histories[s,k,angle,method][call-1] for s in seeds])

    rows = []
    for call in (24, 96):
        effects = {}
        for k in (10, 100):
            for method in ("ARD_local", "ISO_local"):
                aligned, rotated = (terminal(k,a,method,call) for a in ANGLES)
                effect = logs(rotated)-logs(aligned)
                effects[k,method] = effect
                rows.append(dict(call=call,k=k,contrast="rotation",method=method,
                                 aligned_median=np.median(aligned),rotated_median=np.median(rotated),
                                 **stat(effect,draws),
                                 **{"raw_"+n:v for n,v in stat(rotated-aligned,draws).items()}))
            d = effects[k,"ARD_local"]-effects[k,"ISO_local"]
            rows.append(dict(call=call,k=k,contrast="ARD_minus_ISO_rotation",method="",
                             aligned_median="",rotated_median="",**stat(d,draws),
                             raw_mean="",raw_lo="",raw_hi=""))
            for angle in ANGLES:
                d = logs(terminal(k,angle,"ARD_local",call))-logs(terminal(k,angle,"ISO_local",call))
                rows.append(dict(call=call,k=k,contrast=f"ARD_minus_ISO_at_{angle}",method="",
                                 aligned_median="",rotated_median="",**stat(d,draws),
                                 raw_mean="",raw_lo="",raw_hi=""))
        d = effects[10,"ARD_local"]-effects[100,"ARD_local"]
        rows.append(dict(call=call,k="",contrast="k10_minus_k100_rotation",method="ARD_local",
                         aligned_median="",rotated_median="",**stat(d,draws),
                         raw_mean="",raw_lo="",raw_hi=""))
    write_csv(OUT/"replication-effects.csv",rows)
    print(f"Validated {len(histories)} targeted replication trajectories.",flush=True)

In [ ]:
%%writefile -a master_code/stage_a.py

phases = ("final", "sensitivity") if RUN_SENSITIVITY else ("final",)
histories, diagnostics = load_runs(phases, SEEDS)
curved_histories, curved_diagnostics = load_runs(("curved",), SEEDS)
straight_summary = OUT / "straight-summary"
curved_summary = OUT / "curved-summary"
analyse(histories, diagnostics, straight_summary, include_jitter=RUN_SENSITIVITY)
analyse(curved_histories, curved_diagnostics, curved_summary)
print(f"Validated {len(histories)} straight and {len(curved_histories)} curved trajectories.")
if RUN_REPLICATION:
    analyse_replication()
print(f"Saved summary tables in {OUT.resolve()}")

### Plot all six main panels

Top row: straight valleys at condition numbers 1, 10 and 100. Bottom row: curved valleys at the same three levels. Each curve is the pointwise median of the best gap so far across the same 24 blocks. All six panels share a vertical scale. Solid lines show aligned runs and dotted lines show rotated runs. Every panel includes random search, CMA-ES, pooled and refined ARD BO, and pooled and refined isotropic BO.

In [ ]:
%%writefile -a master_code/stage_a.py

from matplotlib.colors import to_rgb

styles = {
    "random": ("Uniform random search", "#565b61"),
    "CMA": ("CMA-ES", "#9b4d18"),
    "ARD_pool": ("ARD BO, pool", "#567a91"),
    "ARD_local": ("ARD BO, refined", "#145ec1"),
    "ISO_pool": ("Isotropic BO, pool", "#75773a"),
    "ISO_local": ("Isotropic BO, refined", "#187759"),
}


def plot_convergence(rows, seeds, methods, ticks, title, filename):
    # Each row contains a shape label and its corresponding run histories.
    calls = np.arange(1, max(ticks) + 1)
    height = 4.8 if len(rows) == 1 else 8.4
    fig, axes = plt.subplots(len(rows), len(KINDS), figsize=(15, height),
                             sharex=True, sharey=True, squeeze=False)
    for i, (shape, histories) in enumerate(rows):
        for ax, k in zip(axes[i], KINDS):
            for method in methods:
                _, color = styles[method]
                for angle, linestyle in ((0, "-"), (45, ":")):
                    values = np.median([histories[s, k, angle, method] for s in seeds], axis=0)
                    curve_color = color if angle == 0 else tuple(.78*c + .22 for c in to_rgb(color))
                    ax.step(calls, np.maximum(values, FLOOR), where="post", color=curve_color,
                            linewidth=1.8 if angle == 0 else 2.4, linestyle=linestyle)
            ax.set_title(f"{shape} | κ = {k}")
            ax.set_yscale("log")
            ax.set_xticks(ticks)
            ax.grid(axis="y", which="major", alpha=.25)
            if i == len(rows) - 1:
                ax.set_xlabel("Objective calls")
        axes[i, 0].set_ylabel("Median best true gap")
    method_handles = [Line2D([], [], color=styles[m][1], lw=2, label=styles[m][0]) for m in methods]
    angle_handles = [Line2D([], [], color="#333333", ls=style, lw=2, label=label)
                     for style, label in (("-", "0° aligned"), (":", "45° rotated"))]
    fig.legend(handles=method_handles + angle_handles, loc="lower center", ncol=4, frameon=False)
    fig.suptitle(title)
    fig.tight_layout(rect=(0, .15 if len(rows) == 1 else .09, 1, .93))
    fig.savefig(OUT / filename, dpi=200, bbox_inches="tight")
    plt.show()


plot_convergence((("Straight", histories), ("Curved", curved_histories)), SEEDS, tuple(styles),
                 (6, 24, 48, 72, 96), "Stage A: straight and curved valleys (24 paired blocks)",
                 "straight-vs-curved-convergence.png")

### Display terminal comparisons

The table includes the random-search baseline. Intervals here describe the mean paired log rotation penalty. The CSV files also record raw gaps, direct BO-versus-CMA interactions, acquisition-floor diagnostics and the covariance diagnostics.

In [ ]:
%%writefile -a master_code/stage_a.py

for shape, directory in (("Straight", straight_summary), ("Curved", curved_summary)):
    with (directory / "rotation-effects.csv").open(encoding="utf-8") as stream:
        terminal = [row for row in csv.DictReader(stream) if int(row["call"]) == BUDGET]
    table = [
        f"### {shape}",
        "",
        "| κ | Method | Aligned median gap | Rotated median gap | Mean log10 penalty [95% interval] |",
        "|---:|:---|---:|---:|---:|",
    ]
    for row in terminal:
        label = styles.get(row["method"], ("Refined ARD, higher jitter", None))[0]
        aligned, rotated = float(row["aligned_median"]), float(row["rotated_median"])
        penalty, lo, hi = (float(row[key]) for key in ("mean", "lo", "hi"))
        penalty_text = ("≈ 0 (roundoff)" if max(abs(penalty), abs(lo), abs(hi)) < 1e-10
                        else f"{penalty:+.3f} [{lo:+.3f}, {hi:+.3f}]")
        table.append(f"| {row['k']} | {label} | {aligned:.3g} | {rotated:.3g} | {penalty_text} |")
    display(Markdown("\n".join(table)))

### Test whether the effect of bending depends on conditioning

For each seed, method, condition number and angle, subtract the log gap on the straight valley from the log gap on the curved valley. This gives a paired **bend effect**. A positive value means a larger gap on the curved valley. To ask whether bending matters more at one condition number than another, subtract those two bend effects within the same seed before bootstrapping. To ask whether bending changes the rotation penalty, subtract the 0° bend effect from the 45° bend effect within each seed. The final contrast compares this change between BO and CMA-ES.

The bootstrap resamples 24 entire blocks, preserving all pairings across shape, condition number, angle and method. The primary checkpoint is 96 calls. The 24- and 48-call estimates describe earlier search. All intervals are pointwise.

In [ ]:
%%writefile -a master_code/stage_a.py

def analyse_bending(straight, curved):
    draws = np.random.default_rng(9601).integers(0, len(SEEDS), (10000, len(SEEDS)))

    def gaps(histories, k, angle, method, call):
        return np.array([histories[s, k, angle, method][call - 1] for s in SEEDS])

    effects, contrasts = [], []

    def save_contrast(name, k, reference_k, angle, method, call, difference):
        contrasts.append(dict(contrast=name, k=k, reference_k=reference_k, angle=angle,
                              method=method, call=call, **stat(difference, draws)))

    for call in CHECKPOINTS:
        bends, rotations = {}, {}
        for k in KINDS:
            for method in styles:
                for angle in ANGLES:
                    straight_gap = gaps(straight, k, angle, method, call)
                    curved_gap = gaps(curved, k, angle, method, call)
                    change = logs(curved_gap) - logs(straight_gap)
                    bends[k, angle, method] = change
                    effects.append(dict(k=k, angle=angle, method=method, call=call,
                                        straight_median=np.median(straight_gap),
                                        curved_median=np.median(curved_gap), **stat(change, draws),
                                        **{"raw_" + name: value
                                           for name, value in stat(curved_gap - straight_gap, draws).items()}))
                rotation_change = bends[k, 45, method] - bends[k, 0, method]
                rotations[k, method] = rotation_change
                save_contrast("bend_change_in_rotation", k, "", "", method, call, rotation_change)
            for method in ("ARD_pool", "ARD_local", "ISO_pool", "ISO_local"):
                save_contrast("bend_change_in_BO_minus_CMA_rotation", k, "", "", method, call,
                              rotations[k, method] - rotations[k, "CMA"])
        for k, reference in ((10, 1), (100, 1), (100, 10)):
            for method in styles:
                for angle in ANGLES:
                    save_contrast("difference_in_bend_effect", k, reference, angle, method, call,
                                  bends[k, angle, method] - bends[reference, angle, method])
                save_contrast("difference_in_bend_change_in_rotation", k, reference, "", method, call,
                              rotations[k, method] - rotations[reference, method])
    write_csv(OUT / "bend-effects.csv", effects)
    write_csv(OUT / "bend-contrasts.csv", contrasts)
    return contrasts


bend_contrasts = analyse_bending(histories, curved_histories)
table = ["| κ | Method | Change in rotation effect after bending [95% interval] |",
         "|---:|:---|---:|"]
for row in bend_contrasts:
    if (row["call"] == BUDGET and row["contrast"] == "bend_change_in_rotation"
            and row["method"] in ("CMA", "ARD_local", "ISO_local")):
        mean, lo, hi = (row[key] for key in ("mean", "lo", "hi"))
        text = ("≈ 0 (roundoff)" if max(abs(mean), abs(lo), abs(hi)) < 1e-10
                else f"{mean:+.3f} [{lo:+.3f}, {hi:+.3f}]")
        table.append(f"| {row['k']} | {styles[row['method']][0]} | {text} |")
display(Markdown("\n".join(table)))

### Summarise the budget check

The table shows refined ARD's paired log rotation effect minus the paired CMA-ES effect. Positive values mean rotation increased BO's gap more than CMA-ES's. Compare the 96- and 192-call estimates within these same eight blocks. Intervals are pointwise. The separate CSV also contains raw effects, median gaps and paired changes in the relative rotation contrast from 96 to 192.

In [ ]:
%%writefile -a master_code/stage_a.py

def analyse_budget_check(histories, seeds):
    n = len(seeds)
    draws = np.random.default_rng(9501).integers(0, n, (10000, n))

    def gaps(k, angle, method, call):
        return np.array([histories[s, k, angle, method][call - 1] for s in seeds])

    def penalty(k, method, call):
        return logs(gaps(k, 45, method, call)) - logs(gaps(k, 0, method, call))

    effects, contrasts = [], []
    for k in KINDS:
        for call in BUDGET_CHECKPOINTS:
            for method in BUDGET_METHODS:
                aligned, rotated = (gaps(k, angle, method, call) for angle in ANGLES)
                effects.append(dict(k=k, call=call, method=method,
                                    aligned_median=np.median(aligned), rotated_median=np.median(rotated),
                                    **stat(penalty(k, method, call), draws),
                                    **{"raw_" + key: value for key, value in stat(rotated - aligned, draws).items()}))
            for method in ("ARD_local", "ISO_local"):
                relative = penalty(k, method, call) - penalty(k, "CMA", call)
                contrasts.append(dict(k=k, call=call, method=method, contrast="BO_minus_CMA_rotation",
                                      **stat(relative, draws)))
                if call > BUDGET:
                    at_96 = penalty(k, method, BUDGET) - penalty(k, "CMA", BUDGET)
                    contrasts.append(dict(k=k, call=call, method=method, contrast="change_since_96",
                                          **stat(relative - at_96, draws)))
    write_csv(OUT / "budget-effects.csv", effects)
    write_csv(OUT / "budget-contrasts.csv", contrasts)

    table = ["| κ | Calls | Relative rotation effect, mean [95% interval] |",
             "|---:|---:|---:|"]
    for row in contrasts:
        if row["method"] == "ARD_local" and row["contrast"] == "BO_minus_CMA_rotation":
            table.append(f"| {row['k']} | {row['call']} | {row['mean']:+.3f} [{row['lo']:+.3f}, {row['hi']:+.3f}] |")
    display(Markdown("\n".join(table)))


if RUN_BUDGET_CHECK:
    budget_histories, _ = load_runs(("budget",), BUDGET_CHECK_SEEDS, budget=max(BUDGET_CHECKPOINTS))
    analyse_budget_check(budget_histories, BUDGET_CHECK_SEEDS)
    plot_convergence((("Straight", budget_histories),), BUDGET_CHECK_SEEDS, BUDGET_METHODS,
                     BUDGET_CHECKPOINTS, "Stage A: budget sensitivity (eight separate blocks)",
                     "budget-convergence.png")

The outputs folder contains `straight-vs-curved-convergence.png`, the main six-panel figure. The `straight-summary/` and `curved-summary/` folders each contain rotation, conditioning, acquisition and solver diagnostics. `bend-effects.csv` compares straight and curved performance. `bend-contrasts.csv` tests how bending changes the rotation effect and whether its effect differs across condition numbers. The optional straight-valley checks write separate replication and budget CSVs. The phase folders contain the individual runs and diagnostics.

In [ ]:
run_stage("stage_a")

## Stage B - observation noise

Source reused from `stage_b_noise.ipynb`. The configuration above controls execution. Code cells in this section prepare the stage script; the final cell runs it when selected.

In [ ]:
%%writefile master_code/stage_b.py
from pathlib import Path
import os

MASTER_PROFILE = globals().get("MASTER_PROFILE", os.environ.get("OPT_PROFILE", "smoke"))
MASTER_ROOT = Path(globals().get("MASTER_ROOT", os.environ.get("OPT_RESULTS_ROOT", "optimisation_results"))).resolve()
MASTER_INCLUDE_TRANSFER = globals().get("MASTER_INCLUDE_TRANSFER", os.environ.get("OPT_INCLUDE_TRANSFER", "1") == "1")
if MASTER_PROFILE not in ("full", "smoke"):
    raise ValueError("Choose full or smoke.")

In [ ]:
%%writefile -a master_code/stage_b.py

import gzip
import json
import math
import os
import random
import runpy
import sys
import time
from dataclasses import dataclass
from importlib.metadata import distribution, version
from math import cos, sin, hypot, tau
from pathlib import Path
from time import perf_counter

for variable in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS"):
    os.environ[variable] = "1"

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.linalg import cholesky, cho_solve, solve_triangular
from scipy.optimize import minimize
from scipy.spatial.distance import cdist

pure_cma = distribution("cma").locate_file("cma/more_algorithms/purecma.py")
CMAES = runpy.run_path(str(pure_cma))["CMAES"]

PROFILE = MASTER_PROFILE
OUTPUT_ROOT = MASTER_ROOT / "stage_b"
BUDGET, POPULATION, POOL = 96, 6, 1024
JITTER, LOG_FLOOR = 1e-7, 1e-14
CHECKPOINTS = (24, 48, 96)
ANGLES = (0, 45)
METHODS = ("Random", "CMA-ES", "ARD BO", "Isotropic BO")
if PROFILE == "full":
    SEEDS = list(range(9100, 9124))
    KINDS, BENDS, NOISE = (1, 10, 100), (0., .75), (0., 1e-4, 1e-3, 1e-2)
elif PROFILE == "smoke":
    SEEDS = [9100]
    KINDS, BENDS, NOISE = (100,), (.75,), (0., 1e-3)
else:
    raise ValueError("Use 'smoke' or 'full'.")
assert BUDGET % POPULATION == 0 and CHECKPOINTS[-1] == BUDGET

SOFTWARE = {"python": sys.version.split()[0], **{
    name: version(name) for name in ("numpy", "scipy", "matplotlib", "pandas", "cma")
}}
SETTINGS = dict(schema=1, profile=PROFILE, seeds=SEEDS, k=list(KINDS),
                bends=list(BENDS), noise=list(NOISE), angles=list(ANGLES),
                budget=BUDGET, population=POPULATION, pool=POOL, jitter=JITTER,
                checkpoints=list(CHECKPOINTS), methods=list(METHODS), software=SOFTWARE)
OUT = OUTPUT_ROOT / PROFILE
print(SOFTWARE)
print(f"{PROFILE}: {len(SEEDS)} blocks; outputs under {OUT}")

### Objectives and paired observations

We reuse Stage A's objective, analytic normalisation and hidden centre seeds.
At each call the optimiser sees only the objective plus Gaussian noise with
standard deviation 0, 0.0001, 0.001 or 0.01 in normalised objective units.
Within a run the draws are independent across calls. Within a seed, we reuse
the standard normal draw at each call index across methods, geometries and
noise levels. This pairs the initial BO and CMA-ES observations as well as
the two rotations. Once methods choose different points, the pairing is by
call index, not by location. Both orientations rotate the search draws
with the objective, exactly as in Stage A.

In [ ]:
%%writefile -a master_code/stage_b.py

def rotate(point, angle):
    x, y = point
    return cos(angle) * x - sin(angle) * y, sin(angle) * x + cos(angle) * y


@dataclass(frozen=True)
class Valley:
    centre: tuple
    angle: float
    curvature_ratio: float = 100.0
    bend: float = 0.0

    @property
    def normaliser(self):
        # Under uniform disk sampling E[X^2] = 1/4 and E[X^4] = 1/8.
        # Terms odd in either coordinate integrate to zero.
        a, b = self.centre
        if self.bend == 0:
            k = self.curvature_ratio
            return (1 + k) / 4 + a*a + k*b*b
        eu2 = .25 + a*a
        ev2 = .25 + b*b
        eu4 = .125 + 1.5*a*a + a**4
        return eu2 + self.curvature_ratio * (
            ev2 + 2*self.bend*b*eu2 + self.bend**2*eu4
        )

    def canonical_gap(self, x, y):
        a, b = self.centre
        u, v = x - a, y - b
        return (u**2 + self.curvature_ratio*(v - self.bend*u**2)**2) / self.normaliser

    def clean(self, point):
        if hypot(*point) > 1.0 + 1e-12:
            raise ValueError("Point outside the unit disk")
        x, y = rotate(point, -self.angle)
        return self.canonical_gap(x, y)

    def observe(self, point, sigma, rng):
        return self.clean(point) + sigma * rng.gauss(0.0, 1.0)


def rotation(degree):
    angle = math.radians(degree)
    return np.array([rotate((1, 0), angle), rotate((0, 1), angle)]).T


def disk(rng, size):
    u = rng.random((size, 2))
    angle = 2 * np.pi * u[:, 0]
    return np.sqrt(u[:, 1:]) * np.column_stack((np.cos(angle), np.sin(angle)))

In [ ]:
%%writefile -a master_code/stage_b.py

def noisy_oracle(valley, noise_sd, z):
    call = 0

    def evaluate(point):
        nonlocal call
        value = valley.clean(point) + noise_sd * z[call]
        call += 1
        return float(value)

    return evaluate

### Search and recommendation

The GP and SLSQP refinement come from Stage A. Known observation variance
enters only the training covariance (in standardised output units).
The acquisition uses the uncertainty of the latent objective. At each
checkpoint BO uses the current length scales and amplitude, conditions on
all observations through that call, and recommends the visited point
with the lowest posterior mean. This readout does not change the search.
For CMA-ES and random search, the recommendation is the lowest observed
point. We also record that rule on the same BO histories.

In [ ]:
%%writefile -a master_code/stage_b.py

def covariance(x, y, theta):
    lengths = np.exp(theta[1:])
    return np.exp(theta[0] - .5 * cdist(x / lengths, y / lengths, "sqeuclidean"))


def fit(x, values, theta, refit, noise_sd=0.0, jitter=JITTER):
    output_scale = values.std() or 1
    y = (values - values.mean()) / output_scale
    ident = np.eye(len(x))

    def factor(parameters):
        signal = covariance(x, x, parameters)
        lower = cholesky(signal + (jitter + (noise_sd/output_scale)**2) * ident, lower=True)
        return signal, lower, cho_solve((lower, True), y)

    def likelihood(parameters):
        signal, lower, alpha = factor(parameters)
        loss = .5 * (y @ alpha) + np.log(np.diag(lower)).sum()
        weights = .5 * (cho_solve((lower, True), ident) - np.outer(alpha, alpha)) * signal
        gradient = [weights.sum()]
        if len(parameters) == 2:
            gradient.append((weights * cdist(x, x, "sqeuclidean")).sum() / np.exp(2*parameters[1]))
        else:
            for axis, log_length in enumerate(parameters[1:]):
                diff = x[:, axis, None] - x[:, axis]
                gradient.append((weights * diff**2).sum() / np.exp(2*log_length))
        return loss, np.array(gradient)

    status = -1
    if refit:
        bounds = [(-6, 6)] + [(math.log(.02), math.log(2))] * (len(theta)-1)
        result = minimize(likelihood, theta, jac=True, bounds=bounds, method="L-BFGS-B",
                          options={"maxiter": 50})
        theta, status = result.x, int(result.status)
    _, lower, alpha = factor(theta)
    return theta, lower, alpha, status


def predict(x, candidates, theta, lower, alpha):
    cross = covariance(x, candidates, theta)
    projected = solve_triangular(lower, cross, lower=True, check_finite=False)
    variance = np.maximum(0, np.exp(theta[0]) - np.einsum("ij,ij->j", projected, projected))
    return cross.T @ alpha, np.sqrt(variance)

In [ ]:
%%writefile -a master_code/stage_b.py

def disk_constraint(point):
    return 1 - point @ point


def disk_jacobian(point):
    return -2 * point


def refined_candidate(x, pool, theta, lower, alpha, pool_lcb):
    """Refine two promising pool candidates with the same fitted LCB."""
    precision = np.broadcast_to(np.exp(-2 * theta[1:]), (2,))
    amplitude = np.exp(theta[0])

    def lcb(p):
        delta = x - p
        kernel = amplitude * np.exp(-.5 * ((delta ** 2) @ precision))
        derivative = kernel[:, None] * delta * precision
        projected = solve_triangular(lower, kernel, lower=True, check_finite=False)
        weights = solve_triangular(lower.T, projected, lower=False, check_finite=False)
        # A tiny variance floor avoids an undefined derivative at zero.
        variance = amplitude - projected @ projected
        sd = np.sqrt(max(variance, 1e-20))
        mean_gradient = derivative.T @ alpha
        gradient = mean_gradient + 2 * (derivative.T @ weights) / sd if variance > 1e-20 else mean_gradient
        return kernel @ alpha - 2 * sd, gradient

    best = pool[int(np.argmin(pool_lcb))]
    best_lcb = float(np.min(pool_lcb))
    starts = [best]
    distant = np.linalg.norm(pool - best, axis=1) >= .05
    if distant.any():
        starts.append(pool[np.argmin(np.where(distant, pool_lcb, np.inf))])
    warnings, improved = 0, False
    for start in starts:
        result = minimize(lcb, start, jac=True, method="SLSQP",
                          constraints=[{"type": "ineq", "fun": disk_constraint,
                                        "jac": disk_jacobian}],
                          options={"maxiter": 25, "ftol": 1e-10})
        warnings += int(not result.success)
        if not np.all(np.isfinite(result.x)) or np.linalg.norm(result.x) > 1 + 1e-8:
            continue
        candidate = result.x / max(1., np.linalg.norm(result.x))
        # Compare with the stable triangular-factor prediction used for the pool.
        mean, sd = predict(x, candidate[None, :], theta, lower, alpha)
        score = float((mean - 2 * sd)[0])
        if score < best_lcb - 1e-12:
            best, best_lcb, improved = candidate, score, True
    return best, warnings, int(improved)


def run_bo(evaluate, q, initial, seed, isotropic, refine, noise_sd, jitter=JITTER, budget=BUDGET):
    rng = np.random.default_rng(seed)
    points = list(initial)
    scores = [evaluate(p) for p in points]
    theta = np.log([1, .4] if isotropic else [1, .4, .4])
    fits, warnings, changed = [], 0, 0
    recommendations = {}
    started = time.perf_counter()
    while len(points) < budget:
        count = len(points)
        x = np.array(points)
        theta, lower, alpha, status = fit(x, np.array(scores), theta, count % 6 == 0,
                                          noise_sd=noise_sd, jitter=jitter)
        if status >= 0:
            fits.append([count, status, *np.exp(theta).tolist()])
        pool = disk(rng, POOL) @ q.T
        mean, sd = predict(x, pool, theta, lower, alpha)
        acq = mean - 2 * sd
        if refine:
            candidate, failures, used = refined_candidate(x, pool, theta, lower, alpha, acq)
            warnings += failures
            changed += used
        else:
            candidate = pool[np.argmin(acq)]
        points.append(candidate)
        scores.append(evaluate(candidate))
        if len(points) in CHECKPOINTS:
            visited = np.asarray(points)
            _, final_lower, final_alpha, _ = fit(
                visited, np.asarray(scores), theta, False, noise_sd=noise_sd, jitter=jitter
            )
            final_mean, _ = predict(visited, visited, theta, final_lower, final_alpha)
            recommendations[len(points)] = int(np.argmin(final_mean))
    return points, scores, dict(fits=fits, local_warnings=warnings,
                                local_improved=changed, seconds=time.perf_counter()-started,
                                recommendations=recommendations)

In [ ]:
%%writefile -a master_code/stage_b.py

def run_coupled_cma(evaluate, q, seed, budget=BUDGET):
    """Use C^(1/2) Q z for matched draws, retaining the reference tell update."""
    state = CMAES([0., 0.], .4, popsize=POPULATION)
    rng = np.random.default_rng(seed)
    points, scores, states = [], [], []
    rejected = 0
    for _ in range(budget // POPULATION):
        state.C.update_eigensystem(state.counteval, state.params.lazy_gap_evals)
        basis = np.asarray(state.C.eigenbasis)
        root = (basis * np.sqrt(state.C.eigenvalues)) @ basis.T
        population = []
        while len(population) < POPULATION:
            point = np.asarray(state.xmean) + state.sigma * (root @ (q @ rng.standard_normal(2)))
            if np.linalg.norm(point) > 1:
                rejected += 1
                if rejected > 10000:
                    raise RuntimeError("CMA rejection cap")
                continue
            population.append(point.tolist())
        values = [evaluate(p) for p in population]
        state.tell(population, values)
        points.extend(population)
        scores.extend(values)
        covariance = np.array([[state.C[i][j] for j in range(2)] for i in range(2)])
        eigvals, vectors = np.linalg.eigh(covariance)
        major_axis = vectors[:, -1]
        states.append(dict(call=state.counteval, ratio=float(eigvals[-1]/eigvals[0]),
                           axis_error=float(np.degrees(np.arccos(np.clip(abs(major_axis @ q[:, 0]), 0, 1)))),
                           sigma=state.sigma, covariance=covariance.tolist(), mean=state.xmean))
    assert state.counteval == budget
    return points, scores, dict(states=states, rejections=rejected)

### Run, validate and save

Clean scores below are computed after each optimiser finishes. We use them
only for assessment. The symmetry checks require matched CMA-ES trajectories
under rotation and unchanged random-search locations across noise levels.
Rotational invariance is a check of the coupled experiment, not evidence
that CMA-ES is unaffected by noise.

In [ ]:
%%writefile -a master_code/stage_b.py

def scores_for_history(valley, points, observations, recommendations):
    xy = np.asarray(points)
    y = np.asarray(observations)
    f = np.array([valley.clean(p) for p in xy])
    assert len(xy) == len(y) == BUDGET and np.isfinite(y).all()
    best = np.minimum.accumulate(f)
    checkpoints = []
    for t in CHECKPOINTS:
        observed_index = int(np.argmin(y[:t]))
        rec_index = recommendations.get(t, observed_index)
        assert 0 <= rec_index < t
        checkpoints.append(dict(
            call=t, recommendation_index=rec_index + 1,
            best_observed_index=observed_index + 1, recommendation=float(f[rec_index]),
            best_visited=float(best[t - 1]), selection_loss=float(f[rec_index] - best[t - 1]),
            best_observed_gap=float(f[observed_index]),
            posterior_mean_gain=float(f[observed_index] - f[rec_index]),
        ))
    return dict(xy=xy.tolist(), observed=y.tolist(), clean=f.tolist(),
                checkpoints=checkpoints, repeat_calls=len(xy) - len(np.unique(xy, axis=0)))


def cma_parent_diagnostics(history):
    f, y = np.asarray(history["clean"]), np.asarray(history["observed"])
    mu = POPULATION // 2
    rows = []
    for start in range(0, BUDGET, POPULATION):
        end = start + POPULATION
        clean_order = np.argsort(f[start:end], kind="stable")
        noisy_order = np.argsort(y[start:end], kind="stable")
        replaced = len(set(noisy_order[:mu]) - set(clean_order[:mu]))
        ordered = f[start:end][clean_order]
        rows.append(dict(
            generation=end // POPULATION, call=end, parent_error=replaced / mu,
            clean_spread=float(ordered[-1] - ordered[0]),
            clean_parent_margin=float(ordered[mu] - ordered[mu - 1]),
        ))
    return rows


def run_seed(seed):
    centre = rotate((.4, 0), random.Random(seed).uniform(0, math.tau))
    random_draws = disk(np.random.default_rng(seed + 10000), BUDGET)
    z = np.random.default_rng(np.random.SeedSequence([seed, 40117])).standard_normal(BUDGET)
    histories = []
    for bend in BENDS:
        for k in KINDS:
            random_base = {}
            for sigma in NOISE:
                cma_paths = {}
                for angle in ANGLES:
                    q = rotation(angle)
                    valley = Valley(centre, math.radians(angle), k, bend)

                    def oracle():
                        return noisy_oracle(valley, sigma, z)

                    cp, cy, ci = run_coupled_cma(oracle(), q, seed + 30000)
                    cma_paths[angle] = (cp, cy, ci)
                    initial = cp[:POPULATION]
                    rp = random_draws @ q.T
                    evaluate_random = oracle()
                    runs = [("Random", rp, [evaluate_random(p) for p in rp], {}),
                            ("CMA-ES", cp, cy, {})]
                    for name, isotropic in (("ARD BO", False), ("Isotropic BO", True)):
                        bp, by, info = run_bo(oracle(), q, initial, seed + 20000,
                                              isotropic, True, noise_sd=sigma)
                        np.testing.assert_array_equal(by[:POPULATION], cy[:POPULATION])
                        runs.append((name, bp, by, info))
                    for name, points, observations, info in runs:
                        row = scores_for_history(valley, points, observations,
                                                 info.pop("recommendations", {}))
                        np.testing.assert_allclose(
                            row["observed"], np.asarray(row["clean"]) + sigma * z,
                            rtol=0, atol=1e-13,
                        )
                        row.update(seed=seed, bend=bend, k=k, sigma=sigma, angle=angle,
                                   method=name, bo_diagnostics=info)
                        if name == "Random":
                            if sigma == 0:
                                random_base[angle] = row["clean"]
                            else:
                                np.testing.assert_array_equal(row["clean"], random_base[angle])
                        if name == "CMA-ES":
                            row["generations"] = cma_parent_diagnostics(row)
                            if sigma == 0:
                                assert all(g["parent_error"] == 0 for g in row["generations"])
                        histories.append(row)
                x0, y0, state0 = cma_paths[0]
                x45, y45, state45 = cma_paths[45]
                q = rotation(45)
                np.testing.assert_allclose(np.asarray(x0) @ q.T, x45, rtol=0, atol=1e-10)
                np.testing.assert_allclose(y0, y45, rtol=0, atol=1e-11)
                assert state0["rejections"] == state45["rejections"]
    return histories


def block_path(seed):
    return OUT / "blocks" / f"{seed}.json.gz"


def save_blocks():
    (OUT / "blocks").mkdir(parents=True, exist_ok=True)
    manifest = OUT / "settings.json"
    if manifest.exists():
        if json.loads(manifest.read_text(encoding="utf-8")) != SETTINGS:
            raise ValueError("Cached settings/software differ. Choose a new OUTPUT_ROOT.")
    else:
        manifest.write_text(json.dumps(SETTINGS, indent=2), encoding="utf-8")
    for i, seed in enumerate(SEEDS, 1):
        path = block_path(seed)
        if path.exists():
            print(f"[{i}/{len(SEEDS)}] Reusing block {seed}", flush=True)
            continue
        started = perf_counter()
        history = run_seed(seed)
        temp = path.with_suffix(".tmp")
        with gzip.open(temp, "wt", encoding="utf-8") as file:
            json.dump(history, file, allow_nan=False, separators=(",", ":"))
        temp.replace(path)
        print(f"[{i}/{len(SEEDS)}] Saved block {seed} ({perf_counter()-started:.1f} s)",
              flush=True)

In [ ]:
%%writefile -a master_code/stage_b.py

save_blocks()

### Tables and paired contrasts

For each checkpoint, report the median true recommendation gap, median best
visited gap, and median selection loss. Selection loss is computed within
each run before taking the median. Also report the BO posterior-mean
recommendation's gain over choosing the best single noisy observation on
the same history. Positive gain means the posterior-mean rule helped.

The primary comparison is at call 96. For each seed, subtract its clean gap
from its noisy gap, or subtract their log10 values for a fold comparison.
A value of +1 on the log scale means a tenfold increase. The comparative
noise penalty subtracts CMA-ES's noise effect from BO's; the
noise-by-rotation contrast subtracts the clean rotation effect from the
noisy rotation effect. Positive values indicate extra degradation in
the first method or rotated orientation, respectively. Two paired geometry
contrasts compare the noise penalty for curved versus straight valleys
and for condition number 100 versus 1. Selection loss is
compared on the raw scale because it is often zero. We retain all pairings
when bootstrapping across seeds. The intervals are pointwise, with no
multiplicity correction. The single-seed smoke profile has no intervals.

In [ ]:
%%writefile -a master_code/stage_b.py

KEYS = ["seed", "bend", "k", "sigma", "angle", "method"]
metrics = ("recommendation", "best_visited", "selection_loss")
checkpoints, generations = [], []
for seed in SEEDS:
    with gzip.open(block_path(seed), "rt", encoding="utf-8") as file:
        for run in json.load(file):
            key = {name: run[name] for name in KEYS}
            checkpoints.extend({**key, **r, "repeat_calls_at_96": run["repeat_calls"]}
                               for r in run["checkpoints"])
            generations.extend({**key, **r} for r in run.get("generations", []))
scores = pd.DataFrame(checkpoints)
ranks = pd.DataFrame(generations)
expected = len(SEEDS)*len(BENDS)*len(KINDS)*len(NOISE)*len(ANGLES)*len(METHODS)
assert len(scores) == expected * len(CHECKPOINTS)
assert not scores.duplicated(KEYS + ["call"]).any()
assert np.allclose(scores.recommendation - scores.best_visited, scores.selection_loss)

scores.to_csv(OUT / "checkpoints.csv", index=False)
ranks.to_csv(OUT / "cma_parent_diagnostics.csv", index=False)
summary = (scores.groupby(KEYS[1:] + ["call"], as_index=False)
           [list(metrics) + ["best_observed_gap", "posterior_mean_gain"]].median())
summary.to_csv(OUT / "median_gaps.csv", index=False)
rank_summary = (ranks.groupby(["bend", "k", "sigma", "angle", "generation", "call"],
                              as_index=False)
                .agg(mean_parent_error=("parent_error", "mean"),
                     median_clean_spread=("clean_spread", "median"),
                     median_parent_margin=("clean_parent_margin", "median")))
rank_summary.to_csv(OUT / "cma_parent_summary.csv", index=False)

example = summary.query(
    "call == @BUDGET and k == 100 and bend == 0.75 and angle == 45"
)[["sigma", "method", *metrics, "posterior_mean_gain"]]
display(example.style.format(precision=3, formatter={
    name: "{:.3e}" for name in [*metrics, "posterior_mean_gain"]
}))

In [ ]:
%%writefile -a master_code/stage_b.py

rng = np.random.default_rng(4569)
bootstrap = (rng.integers(0, len(SEEDS), size=(10000, len(SEEDS)))
             if len(SEEDS) > 1 else None)
terminal = scores[scores["call"] == BUDGET]
blocks = {
    key: table.set_index("seed").reindex(SEEDS)
    for key, table in terminal.groupby(KEYS[1:])
}


def vector(bend, k, sigma, angle, method, metric, scale):
    values = blocks[bend, k, sigma, angle, method][metric].to_numpy()
    assert np.isfinite(values).all()
    return np.log10(np.maximum(values, LOG_FLOOR)) if scale == "log10" else values


def paired_result(values, **label):
    mean = values.mean()
    lo, hi = ((np.nan, np.nan) if bootstrap is None else
              np.quantile(values[bootstrap].mean(axis=1), [.025, .975]))
    return dict(**label, n=len(values), mean=mean, low_95=lo, high_95=hi)


def noise_effect(bend, k, sigma, angle, method, metric, scale):
    return (vector(bend, k, sigma, angle, method, metric, scale)
            - vector(bend, k, 0., angle, method, metric, scale))


contrasts = []
for bend in BENDS:
    for k in KINDS:
        for sigma in NOISE[1:]:
            for metric in metrics:
                for scale in ("raw", "log10") if metric != "selection_loss" else ("raw",):
                    effects = {}
                    for angle in ANGLES:
                        for method in METHODS:
                            delta = noise_effect(bend, k, sigma, angle, method, metric, scale)
                            effects[angle, method] = delta
                            contrasts.append(paired_result(
                                delta, contrast="noisy - clean", bend=bend, k=k, sigma=sigma,
                                angle=angle, method=method, metric=metric, scale=scale))
                        for method in ("ARD BO", "Isotropic BO"):
                            contrasts.append(paired_result(
                                effects[angle, method] - effects[angle, "CMA-ES"],
                                contrast="BO noise effect - CMA-ES noise effect",
                                bend=bend, k=k, sigma=sigma, angle=angle,
                                method=method, metric=metric, scale=scale))
                    for method in METHODS:
                        contrasts.append(paired_result(
                            effects[45, method] - effects[0, method],
                            contrast="noise-by-rotation interaction",
                            bend=bend, k=k, sigma=sigma, angle="45 - 0",
                            method=method, metric=metric, scale=scale))
if PROFILE == "full":
    for sigma in NOISE[1:]:
        for angle in ANGLES:
            for method in METHODS:
                for metric in ("recommendation", "best_visited"):
                    for scale in ("raw", "log10"):
                        label = dict(sigma=sigma, angle=angle, method=method,
                                     metric=metric, scale=scale)
                        for k in KINDS:
                            curved = noise_effect(.75, k, sigma, angle, method, metric, scale)
                            straight = noise_effect(0., k, sigma, angle, method, metric, scale)
                            contrasts.append(paired_result(
                                curved - straight, contrast="noise-by-bend interaction",
                                bend="curved - straight", k=k, **label))
                        for bend in BENDS:
                            high = noise_effect(bend, 100, sigma, angle, method, metric, scale)
                            low = noise_effect(bend, 1, sigma, angle, method, metric, scale)
                            contrasts.append(paired_result(
                                high - low, contrast="noise-by-conditioning interaction",
                                bend=bend, k="100 - 1", **label))
paired = pd.DataFrame(contrasts)
paired.to_csv(OUT / "paired_contrasts_at_96.csv", index=False)
focus = paired[(paired["contrast"] == "noise-by-rotation interaction")
               & (paired["method"].isin(["ARD BO", "Isotropic BO"]))
               & (paired["metric"] == "recommendation") & (paired["scale"] == "log10")
               & (paired["bend"] == .75) & (paired["k"] == 100)]
display(focus[["sigma", "method", "n", "mean", "low_95", "high_95"]].round(3))

### Figures

The first figure gives the main outcome and the second gives the search-only
counterpart. The third shows the selection loss, with a linear region near
zero so exact zeros remain visible. The final figure shows CMA-ES's
generation-by-generation parent ranking errors and the clean score spread.
It shows the aligned orientation because the two CMA-ES orientations are paired.
The dashed line at 0.5 is the expected parent replacement rate if the
observed ranking is independent of the true ranking.

In [ ]:
%%writefile -a master_code/stage_b.py

COLORS = dict(zip(METHODS, ("#6e6e6e", "#b45309", "#0072b2", "#008f6b")))
shape = {0.: "Straight", .75: "Curved"}
sigma_ticks = ["0" if x == 0 else rf"$10^{{{int(round(np.log10(x)))}}}$" for x in NOISE]


def gap_figure(metric, title, ylabel, filename):
    fig, axes = plt.subplots(len(BENDS), len(KINDS), squeeze=False, sharex=True,
                             sharey=True, figsize=(max(5, 3.45*len(KINDS)), 2.9*len(BENDS)+.9),
                             layout="constrained")
    for i, bend in enumerate(BENDS):
        for j, k in enumerate(KINDS):
            ax = axes[i, j]
            panel = summary[(summary.bend == bend) & (summary.k == k)
                            & (summary["call"] == BUDGET)]
            for name in METHODS:
                for angle, line, marker in ((0, "-", "o"), (45, ":", "s")):
                    values = (panel[(panel.method == name) & (panel.angle == angle)]
                              .set_index("sigma")[metric].reindex(NOISE).to_numpy())
                    if metric != "selection_loss":
                        values = np.maximum(values, LOG_FLOOR)
                    ax.plot(range(len(NOISE)), values, color=COLORS[name], ls=line,
                            lw=1.5, marker=marker, ms=4, mfc="none" if angle else COLORS[name])
            if metric == "selection_loss":
                ax.set_yscale("symlog", linthresh=1e-7)
                ax.set_ylim(bottom=0)
            else:
                ax.set_yscale("log")
            ax.set_title(rf"{shape[bend]}, $\kappa={k}$")
            ax.set_xticks(range(len(NOISE)), sigma_ticks)
            ax.grid(axis="y", alpha=.2)
    fig.suptitle(title)
    fig.supxlabel(r"Observation noise standard deviation $\sigma$")
    fig.supylabel(ylabel)
    handles = [Line2D([], [], color=COLORS[m], lw=2, label=m) for m in METHODS]
    handles += [Line2D([], [], color="black", ls=ls, label=f"{angle}°")
                for ls, angle in (("-", 0), (":", 45))]
    fig.legend(handles=handles, loc="outside upper center", ncol=min(6, len(handles)),
               fontsize=8, frameon=False)
    fig.savefig(OUT / filename, dpi=200)
    plt.show()


gap_figure("recommendation", "Returned solution after 96 calls",
           "Median true recommendation gap", "recommendation-gap.png")
gap_figure("best_visited", "Best point visited by 96 calls",
           "Median best true gap", "best-visited-gap.png")
gap_figure("selection_loss", "Quality lost in the final recommendation",
           "Median selection loss", "selection-loss.png")

In [ ]:
%%writefile -a master_code/stage_b.py

fig, axes = plt.subplots(2*len(BENDS), len(KINDS), squeeze=False, sharex=True,
                         figsize=(max(5, 3.45*len(KINDS)), 4.1*len(BENDS)+.65),
                         layout="constrained")
noise_colors = plt.get_cmap("viridis")(np.linspace(.12, .88, len(NOISE)))
for i, bend in enumerate(BENDS):
    for j, k in enumerate(KINDS):
        upper, lower = axes[2*i, j], axes[2*i+1, j]
        group = rank_summary[(rank_summary.bend == bend) & (rank_summary.k == k)
                             & (rank_summary.angle == 0)]
        for sigma, color in zip(NOISE, noise_colors):
            series = group[group.sigma == sigma].sort_values("generation")
            upper.plot(series.generation, series.mean_parent_error, color=color)
            lower.plot(series.generation, np.maximum(series.median_clean_spread, LOG_FLOOR),
                       color=color)
        upper.set_title(rf"{shape[bend]}, $\kappa={k}$")
        upper.set_ylim(-.03, 1.03)
        upper.axhline(.5, color=".55", ls="--", lw=.9, zorder=0)
        lower.set_yscale("log")
        upper.grid(axis="y", alpha=.2)
        lower.grid(axis="y", alpha=.2)
        if j == 0:
            upper.set_ylabel("Mean parent error")
            lower.set_ylabel("Median clean spread")
fig.legend(handles=[Line2D([], [], color=color, label=label)
                    for color, label in zip(noise_colors, sigma_ticks)],
           title="Noise SD", loc="outside upper center", ncol=len(NOISE), frameon=False)
fig.supxlabel("CMA-ES generation (6 calls each)")
fig.savefig(OUT / "cma-parent-ranking.png", dpi=200)
plt.show()

The compressed blocks retain points, observed scores and clean assessment
scores for every call. The CSVs contain checkpoints, median gaps, paired
terminal contrasts and the CMA-ES ranking diagnostics. Start the report
interpretation with the recommendation figure. Use the best-visited and
selection-loss figures to explain how the gaps arose, and use the parent
plot to check when CMA-ES updates were exposed to ranking errors. Parent
error measures membership changes among the top three; it does not measure
changes in order among parents who remain selected. This study tests one
known, constant Gaussian noise model and one call budget. It does not test
an allocation of calls to deliberate reevaluation.

In [ ]:
run_stage("stage_b")

## Stage B - budget extension

Source reused from `stage_b_budget_extension.ipynb`. The configuration above controls execution. Code cells in this section prepare the stage script; the final cell runs it when selected.

In [ ]:
%%writefile master_code/stage_b_budget.py
from pathlib import Path
import os

MASTER_PROFILE = globals().get("MASTER_PROFILE", os.environ.get("OPT_PROFILE", "smoke"))
MASTER_ROOT = Path(globals().get("MASTER_ROOT", os.environ.get("OPT_RESULTS_ROOT", "optimisation_results"))).resolve()
MASTER_INCLUDE_TRANSFER = globals().get("MASTER_INCLUDE_TRANSFER", os.environ.get("OPT_INCLUDE_TRANSFER", "1") == "1")
if MASTER_PROFILE not in ("full", "smoke"):
    raise ValueError("Choose full or smoke.")

In [ ]:
%%writefile -a master_code/stage_b_budget.py

import gzip
import json
import math
import os
import random
import runpy
import sys
import time
from dataclasses import dataclass
from importlib.metadata import distribution, version
from math import cos, sin, hypot, tau
from pathlib import Path
from time import perf_counter

for variable in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS"):
    os.environ[variable] = "1"

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.linalg import cholesky, cho_solve, solve_triangular
from scipy.optimize import minimize
from scipy.spatial.distance import cdist

pure_cma = distribution("cma").locate_file("cma/more_algorithms/purecma.py")
CMAES = runpy.run_path(str(pure_cma))["CMAES"]

PROFILE = MASTER_PROFILE
OUTPUT_ROOT = MASTER_ROOT / "stage_b_budget"
BUDGET, POPULATION, POOL = 192, 6, 1024
JITTER, LOG_FLOOR = 1e-7, 1e-14
CHECKPOINTS = (96, 144, 192)
ANGLES = (0, 45)
METHODS = ("Random", "CMA-ES", "ARD BO", "Isotropic BO")
if PROFILE == "full":
    SEEDS = list(range(9100, 9124))
    KINDS, BENDS, NOISE = (100,), (0., .75), (0., 1e-3, 1e-2)
elif PROFILE == "smoke":
    SEEDS = [9100]
    KINDS, BENDS, NOISE = (100,), (.75,), (0., 1e-2)
else:
    raise ValueError("Use 'smoke' or 'full'.")
assert BUDGET % POPULATION == 0 and CHECKPOINTS[-1] == BUDGET

SOFTWARE = {"python": sys.version.split()[0], **{
    name: version(name) for name in ("numpy", "scipy", "matplotlib", "pandas", "cma")
}}
SETTINGS = dict(schema=1, profile=PROFILE, seeds=SEEDS, k=list(KINDS),
                bends=list(BENDS), noise=list(NOISE), angles=list(ANGLES),
                budget=BUDGET, population=POPULATION, pool=POOL, jitter=JITTER,
                checkpoints=list(CHECKPOINTS), methods=list(METHODS), software=SOFTWARE)
OUT = OUTPUT_ROOT / PROFILE
print(SOFTWARE)
print(f"{PROFILE}: {len(SEEDS)} blocks; outputs under {OUT}")

### Figure layout and optional replot of Stage B

Titles and legends occupy a separate row above the data panels. Panels
share a vertical scale within each figure. The original noise sweep is
split by orientation to keep the paired lines legible.

In [ ]:
%%writefile -a master_code/stage_b_budget.py

COLORS = dict(zip(METHODS, ("#666666", "#b45309", "#0072b2", "#008f6b")))
MARKERS = dict(zip(METHODS, ("D", "s", "o", "^")))
SHAPES = {0.: "Straight", .75: "Curved"}
METRIC_LABELS = {
    "recommendation": "True recommendation gap",
    "best_visited": "Best true gap visited",
    "selection_loss": "Selection loss",
}


def noise_label(sigma):
    return "0" if sigma == 0 else rf"$10^{{{round(np.log10(sigma))}}}$"


def figure_grid(nrows, ncols, title, xlabel, ylabel, handles, row_labels=None):
    fig = plt.figure(figsize=(max(6.4, 3.55*ncols), 2.65*nrows+1.45),
                     layout="constrained")
    grid = fig.add_gridspec(nrows+1, ncols, height_ratios=[.38] + [1]*nrows)
    header = fig.add_subplot(grid[0, :])
    header.set_axis_off()
    header.text(.5, .78, title, ha="center", va="center", fontsize=13)
    header.legend(handles=handles, loc="lower center", ncol=len(handles),
                  fontsize=9, frameon=False, borderaxespad=0)
    axes = np.empty((nrows, ncols), dtype=object)
    for i in range(nrows):
        for j in range(ncols):
            shared = (axes[i, 0] if j else None) if row_labels else (
                axes[0, 0] if i or j else None)
            axes[i, j] = fig.add_subplot(grid[i+1, j], sharey=shared)
            axes[i, j].tick_params(labelleft=(j == 0))
            axes[i, j].grid(axis="y", alpha=.22)
            if row_labels and j == 0:
                axes[i, j].set_ylabel(row_labels[i])
    fig.supxlabel(xlabel)
    if ylabel:
        fig.supylabel(ylabel)
    return fig, axes


def method_handles(methods=METHODS):
    return [Line2D([], [], label=m, color=COLORS[m], marker=MARKERS[m])
            for m in methods]


def metric_scale(ax, metric, values):
    if metric == "selection_loss":
        ax.set_yscale("symlog", linthresh=1e-7)
        return values
    ax.set_yscale("log")
    return np.maximum(values, LOG_FLOOR)


def finish_figure(fig, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=220, bbox_inches="tight", pad_inches=.12)
    plt.show()
    plt.close(fig)


def redraw_stage_b(csv_path):
    original = pd.read_csv(csv_path)
    original = original[original["call"] == 96]
    bends, kinds, noise = [sorted(original[c].unique()) for c in ("bend", "k", "sigma")]
    for angle in ANGLES:
        for metric, label in METRIC_LABELS.items():
            fig, axes = figure_grid(len(bends), len(kinds), f"{label} at 96 calls, {angle}°",
                                    "Observation noise SD", f"Median {label.lower()}",
                                    method_handles())
            for i, bend in enumerate(bends):
                for j, k in enumerate(kinds):
                    ax = axes[i, j]
                    panel = original[(original.bend == bend) & (original.k == k)
                                     & (original.angle == angle)]
                    for method in METHODS:
                        values = panel[panel.method == method].set_index("sigma")[metric]
                        ax.plot(range(len(noise)), metric_scale(ax, metric, values.reindex(noise)),
                                color=COLORS[method], marker=MARKERS[method], ms=4)
                    ax.set_title(rf"{SHAPES[bend]}, $\kappa={k}$")
                    ax.set_xticks(range(len(noise)), [noise_label(s) for s in noise])
            if metric == "selection_loss":
                axes[0, 0].set_ylim(bottom=0)
            finish_figure(fig, OUTPUT_ROOT / "original_replots" / f"{metric}-{angle}.png")

In [ ]:
%%writefile -a master_code/stage_b_budget.py

# Optional. Set this to your existing Stage B median_gaps.csv to redraw it.
STAGE_B_CSV = None  # e.g. Path("stage_b_results/full/median_gaps.csv")
if STAGE_B_CSV is not None:
    redraw_stage_b(STAGE_B_CSV)

### Stage B objective and search implementations

The noise SD enters the BO training covariance and the acquisition uses
the same latent uncertainty as before. At a checkpoint, the notebook
records the rank of the best true visited point under each recommendation
rule. A rank of 1 means that rule identified it. Clean objective values
and the resulting ranks are used only by the analysis.

In [ ]:
%%writefile -a master_code/stage_b_budget.py

def rotate(point, angle):
    x, y = point
    return cos(angle) * x - sin(angle) * y, sin(angle) * x + cos(angle) * y


@dataclass(frozen=True)
class Valley:
    centre: tuple
    angle: float
    curvature_ratio: float = 100.0
    bend: float = 0.0

    @property
    def normaliser(self):
        # Under uniform disk sampling E[X^2] = 1/4 and E[X^4] = 1/8.
        # Terms odd in either coordinate integrate to zero.
        a, b = self.centre
        if self.bend == 0:
            k = self.curvature_ratio
            return (1 + k) / 4 + a*a + k*b*b
        eu2 = .25 + a*a
        ev2 = .25 + b*b
        eu4 = .125 + 1.5*a*a + a**4
        return eu2 + self.curvature_ratio * (
            ev2 + 2*self.bend*b*eu2 + self.bend**2*eu4
        )

    def canonical_gap(self, x, y):
        a, b = self.centre
        u, v = x - a, y - b
        return (u**2 + self.curvature_ratio*(v - self.bend*u**2)**2) / self.normaliser

    def clean(self, point):
        if hypot(*point) > 1.0 + 1e-12:
            raise ValueError("Point outside the unit disk")
        x, y = rotate(point, -self.angle)
        return self.canonical_gap(x, y)

    def observe(self, point, sigma, rng):
        return self.clean(point) + sigma * rng.gauss(0.0, 1.0)


def rotation(degree):
    angle = math.radians(degree)
    return np.array([rotate((1, 0), angle), rotate((0, 1), angle)]).T


def disk(rng, size):
    u = rng.random((size, 2))
    angle = 2 * np.pi * u[:, 0]
    return np.sqrt(u[:, 1:]) * np.column_stack((np.cos(angle), np.sin(angle)))

In [ ]:
%%writefile -a master_code/stage_b_budget.py

def noisy_oracle(valley, noise_sd, z):
    call = 0

    def evaluate(point):
        nonlocal call
        value = valley.clean(point) + noise_sd * z[call]
        call += 1
        return float(value)

    return evaluate

In [ ]:
%%writefile -a master_code/stage_b_budget.py

def covariance(x, y, theta):
    lengths = np.exp(theta[1:])
    return np.exp(theta[0] - .5 * cdist(x / lengths, y / lengths, "sqeuclidean"))


def fit(x, values, theta, refit, noise_sd=0.0, jitter=JITTER):
    output_scale = values.std() or 1
    y = (values - values.mean()) / output_scale
    ident = np.eye(len(x))

    def factor(parameters):
        signal = covariance(x, x, parameters)
        lower = cholesky(signal + (jitter + (noise_sd/output_scale)**2) * ident, lower=True)
        return signal, lower, cho_solve((lower, True), y)

    def likelihood(parameters):
        signal, lower, alpha = factor(parameters)
        loss = .5 * (y @ alpha) + np.log(np.diag(lower)).sum()
        weights = .5 * (cho_solve((lower, True), ident) - np.outer(alpha, alpha)) * signal
        gradient = [weights.sum()]
        if len(parameters) == 2:
            gradient.append((weights * cdist(x, x, "sqeuclidean")).sum() / np.exp(2*parameters[1]))
        else:
            for axis, log_length in enumerate(parameters[1:]):
                diff = x[:, axis, None] - x[:, axis]
                gradient.append((weights * diff**2).sum() / np.exp(2*log_length))
        return loss, np.array(gradient)

    status = -1
    if refit:
        bounds = [(-6, 6)] + [(math.log(.02), math.log(2))] * (len(theta)-1)
        result = minimize(likelihood, theta, jac=True, bounds=bounds, method="L-BFGS-B",
                          options={"maxiter": 50})
        theta, status = result.x, int(result.status)
    _, lower, alpha = factor(theta)
    return theta, lower, alpha, status


def predict(x, candidates, theta, lower, alpha):
    cross = covariance(x, candidates, theta)
    projected = solve_triangular(lower, cross, lower=True, check_finite=False)
    variance = np.maximum(0, np.exp(theta[0]) - np.einsum("ij,ij->j", projected, projected))
    return cross.T @ alpha, np.sqrt(variance)

In [ ]:
%%writefile -a master_code/stage_b_budget.py

def disk_constraint(point):
    return 1 - point @ point


def disk_jacobian(point):
    return -2 * point


def refined_candidate(x, pool, theta, lower, alpha, pool_lcb):
    """Refine two promising pool candidates with the same fitted LCB."""
    precision = np.broadcast_to(np.exp(-2 * theta[1:]), (2,))
    amplitude = np.exp(theta[0])

    def lcb(p):
        delta = x - p
        kernel = amplitude * np.exp(-.5 * ((delta ** 2) @ precision))
        derivative = kernel[:, None] * delta * precision
        projected = solve_triangular(lower, kernel, lower=True, check_finite=False)
        weights = solve_triangular(lower.T, projected, lower=False, check_finite=False)
        # A tiny variance floor avoids an undefined derivative at zero.
        variance = amplitude - projected @ projected
        sd = np.sqrt(max(variance, 1e-20))
        mean_gradient = derivative.T @ alpha
        gradient = mean_gradient + 2 * (derivative.T @ weights) / sd if variance > 1e-20 else mean_gradient
        return kernel @ alpha - 2 * sd, gradient

    best = pool[int(np.argmin(pool_lcb))]
    best_lcb = float(np.min(pool_lcb))
    starts = [best]
    distant = np.linalg.norm(pool - best, axis=1) >= .05
    if distant.any():
        starts.append(pool[np.argmin(np.where(distant, pool_lcb, np.inf))])
    warnings, improved = 0, False
    for start in starts:
        result = minimize(lcb, start, jac=True, method="SLSQP",
                          constraints=[{"type": "ineq", "fun": disk_constraint,
                                        "jac": disk_jacobian}],
                          options={"maxiter": 25, "ftol": 1e-10})
        warnings += int(not result.success)
        if not np.all(np.isfinite(result.x)) or np.linalg.norm(result.x) > 1 + 1e-8:
            continue
        candidate = result.x / max(1., np.linalg.norm(result.x))
        # Compare with the stable triangular-factor prediction used for the pool.
        mean, sd = predict(x, candidate[None, :], theta, lower, alpha)
        score = float((mean - 2 * sd)[0])
        if score < best_lcb - 1e-12:
            best, best_lcb, improved = candidate, score, True
    return best, warnings, int(improved)


def run_bo(evaluate, q, initial, seed, isotropic, refine, noise_sd, jitter=JITTER, budget=BUDGET):
    rng = np.random.default_rng(seed)
    points = list(initial)
    scores = [evaluate(p) for p in points]
    theta = np.log([1, .4] if isotropic else [1, .4, .4])
    fits, warnings, changed = [], 0, 0
    recommendations = {}
    started = time.perf_counter()
    while len(points) < budget:
        count = len(points)
        x = np.array(points)
        theta, lower, alpha, status = fit(x, np.array(scores), theta, count % 6 == 0,
                                          noise_sd=noise_sd, jitter=jitter)
        if status >= 0:
            fits.append([count, status, *np.exp(theta).tolist()])
        pool = disk(rng, POOL) @ q.T
        mean, sd = predict(x, pool, theta, lower, alpha)
        acq = mean - 2 * sd
        if refine:
            candidate, failures, used = refined_candidate(x, pool, theta, lower, alpha, acq)
            warnings += failures
            changed += used
        else:
            candidate = pool[np.argmin(acq)]
        points.append(candidate)
        scores.append(evaluate(candidate))
        if len(points) in CHECKPOINTS:
            visited = np.asarray(points)
            _, final_lower, final_alpha, _ = fit(
                visited, np.asarray(scores), theta, False, noise_sd=noise_sd, jitter=jitter
            )
            final_mean, _ = predict(visited, visited, theta, final_lower, final_alpha)
            recommendations[len(points)] = final_mean.tolist()
    return points, scores, dict(fits=fits, local_warnings=warnings,
                                local_improved=changed, seconds=time.perf_counter()-started,
                                recommendations=recommendations)

In [ ]:
%%writefile -a master_code/stage_b_budget.py

def run_coupled_cma(evaluate, q, seed, budget=BUDGET):
    """Use C^(1/2) Q z for matched draws, retaining the reference tell update."""
    state = CMAES([0., 0.], .4, popsize=POPULATION)
    rng = np.random.default_rng(seed)
    points, scores, states = [], [], []
    rejected = 0
    for _ in range(budget // POPULATION):
        state.C.update_eigensystem(state.counteval, state.params.lazy_gap_evals)
        basis = np.asarray(state.C.eigenbasis)
        root = (basis * np.sqrt(state.C.eigenvalues)) @ basis.T
        population = []
        while len(population) < POPULATION:
            point = np.asarray(state.xmean) + state.sigma * (root @ (q @ rng.standard_normal(2)))
            if np.linalg.norm(point) > 1:
                rejected += 1
                if rejected > 10000:
                    raise RuntimeError("CMA rejection cap")
                continue
            population.append(point.tolist())
        values = [evaluate(p) for p in population]
        state.tell(population, values)
        points.extend(population)
        scores.extend(values)
        covariance = np.array([[state.C[i][j] for j in range(2)] for i in range(2)])
        eigvals, vectors = np.linalg.eigh(covariance)
        major_axis = vectors[:, -1]
        states.append(dict(call=state.counteval, ratio=float(eigvals[-1]/eigvals[0]),
                           axis_error=float(np.degrees(np.arccos(np.clip(abs(major_axis @ q[:, 0]), 0, 1)))),
                           sigma=state.sigma, covariance=covariance.tolist(), mean=state.xmean,
                           search_sd=float(state.sigma * np.sqrt(eigvals.mean()))))
    assert state.counteval == budget
    return points, scores, dict(states=states, rejections=rejected)

### Execute and save the paired runs

The BO diagnostic measures the distance from each proposed point to its
nearest previously evaluated point, excluding the shared initial design.
The CMA-ES diagnostics record the search distribution's root mean square
coordinate SD and the true score gap between the third and fourth candidates.
That gap describes how readily noise can move a candidate across the parent
selection boundary. The true scores stay outside both optimisers.

In [ ]:
%%writefile -a master_code/stage_b_budget.py

def scores_for_history(valley, points, observations, recommendations):
    xy = np.asarray(points)
    y = np.asarray(observations)
    f = np.array([valley.clean(p) for p in xy])
    assert len(xy) == len(y) == BUDGET and np.isfinite(y).all()
    best = np.minimum.accumulate(f)
    checkpoints = []
    nearest = np.array([np.linalg.norm(xy[:i] - xy[i], axis=1).min()
                        for i in range(POPULATION, len(xy))])
    previous = POPULATION
    for t in CHECKPOINTS:
        observed_index = int(np.argmin(y[:t]))
        rule_scores = np.asarray(recommendations.get(t, y[:t]))
        rec_index = int(np.argmin(rule_scores))
        true_best_index = int(np.argmin(f[:t]))
        true_best_rule_rank = 1 + np.count_nonzero(rule_scores < rule_scores[true_best_index])
        true_best_observed_rank = 1 + np.count_nonzero(y[:t] < y[true_best_index])
        assert 0 <= rec_index < t
        checkpoints.append(dict(
            call=t, recommendation_index=rec_index + 1,
            best_observed_index=observed_index + 1, recommendation=float(f[rec_index]),
            best_visited=float(best[t - 1]), selection_loss=float(f[rec_index] - best[t - 1]),
            best_observed_gap=float(f[observed_index]),
            posterior_mean_gain=float(f[observed_index] - f[rec_index]),
            true_best_rule_rank=int(true_best_rule_rank),
            true_best_observed_rank=int(true_best_observed_rank),
            true_best_in_top5=int(true_best_rule_rank <= 5),
            best_observed_noise=float(y[observed_index] - f[observed_index]),
            nearest_distance=float(np.median(nearest[previous-POPULATION:t-POPULATION])),
        ))
        previous = t
    return dict(xy=xy.tolist(), observed=y.tolist(), clean=f.tolist(),
                checkpoints=checkpoints, repeat_calls=len(xy) - len(np.unique(xy, axis=0)))


def cma_parent_diagnostics(history, states):
    f, y = np.asarray(history["clean"]), np.asarray(history["observed"])
    mu = POPULATION // 2
    rows = []
    for start in range(0, BUDGET, POPULATION):
        end = start + POPULATION
        clean_order = np.argsort(f[start:end], kind="stable")
        noisy_order = np.argsort(y[start:end], kind="stable")
        replaced = len(set(noisy_order[:mu]) - set(clean_order[:mu]))
        ordered = f[start:end][clean_order]
        rows.append(dict(
            generation=end // POPULATION, call=end, parent_error=replaced / mu,
            clean_spread=float(ordered[-1] - ordered[0]),
            clean_parent_margin=float(ordered[mu] - ordered[mu - 1]),
            search_sd=states[end // POPULATION - 1]["search_sd"],
        ))
    return rows


def run_seed(seed):
    centre = rotate((.4, 0), random.Random(seed).uniform(0, math.tau))
    random_draws = disk(np.random.default_rng(seed + 10000), BUDGET)
    z = np.random.default_rng(np.random.SeedSequence([seed, 40117])).standard_normal(BUDGET)
    histories = []
    for bend in BENDS:
        for k in KINDS:
            random_base = {}
            for sigma in NOISE:
                cma_paths = {}
                for angle in ANGLES:
                    q = rotation(angle)
                    valley = Valley(centre, math.radians(angle), k, bend)

                    def oracle():
                        return noisy_oracle(valley, sigma, z)

                    cp, cy, ci = run_coupled_cma(oracle(), q, seed + 30000)
                    cma_paths[angle] = (cp, cy, ci)
                    initial = cp[:POPULATION]
                    rp = random_draws @ q.T
                    evaluate_random = oracle()
                    runs = [("Random", rp, [evaluate_random(p) for p in rp], {}),
                            ("CMA-ES", cp, cy, {})]
                    for name, isotropic in (("ARD BO", False), ("Isotropic BO", True)):
                        bp, by, info = run_bo(oracle(), q, initial, seed + 20000,
                                              isotropic, True, noise_sd=sigma)
                        np.testing.assert_array_equal(by[:POPULATION], cy[:POPULATION])
                        runs.append((name, bp, by, info))
                    for name, points, observations, info in runs:
                        row = scores_for_history(valley, points, observations,
                                                 info.pop("recommendations", {}))
                        np.testing.assert_allclose(
                            row["observed"], np.asarray(row["clean"]) + sigma * z,
                            rtol=0, atol=1e-13,
                        )
                        row.update(seed=seed, bend=bend, k=k, sigma=sigma, angle=angle,
                                   method=name, bo_diagnostics=info)
                        if name == "Random":
                            if sigma == 0:
                                random_base[angle] = row["clean"]
                            else:
                                np.testing.assert_array_equal(row["clean"], random_base[angle])
                        if name == "CMA-ES":
                            row["generations"] = cma_parent_diagnostics(row, ci["states"])
                            if sigma == 0:
                                assert all(g["parent_error"] == 0 for g in row["generations"])
                        histories.append(row)
                x0, y0, state0 = cma_paths[0]
                x45, y45, state45 = cma_paths[45]
                q = rotation(45)
                np.testing.assert_allclose(np.asarray(x0) @ q.T, x45, rtol=0, atol=1e-10)
                np.testing.assert_allclose(y0, y45, rtol=0, atol=1e-11)
                assert state0["rejections"] == state45["rejections"]
    return histories


def block_path(seed):
    return OUT / "blocks" / f"{seed}.json.gz"


def save_blocks():
    (OUT / "blocks").mkdir(parents=True, exist_ok=True)
    manifest = OUT / "settings.json"
    if manifest.exists():
        if json.loads(manifest.read_text(encoding="utf-8")) != SETTINGS:
            raise ValueError("Cached settings/software differ. Choose a new OUTPUT_ROOT.")
    else:
        manifest.write_text(json.dumps(SETTINGS, indent=2), encoding="utf-8")
    for i, seed in enumerate(SEEDS, 1):
        path = block_path(seed)
        if path.exists():
            print(f"[{i}/{len(SEEDS)}] Reusing block {seed}", flush=True)
            continue
        started = perf_counter()
        history = run_seed(seed)
        temp = path.with_suffix(".tmp")
        with gzip.open(temp, "wt", encoding="utf-8") as file:
            json.dump(history, file, allow_nan=False, separators=(",", ":"))
        temp.replace(path)
        print(f"[{i}/{len(SEEDS)}] Saved block {seed} ({perf_counter()-started:.1f} s)",
              flush=True)

In [ ]:
%%writefile -a master_code/stage_b_budget.py

save_blocks()

### Tables and paired budget comparisons

For each seed, take the **192-call value minus the 96-call value**. Negative
differences in the gap metrics mean improvement. On the raw scale,
the change in recommendation gap equals the change in best visited gap
plus the change in selection loss. The CSV contains all three components
and 95% pointwise paired bootstrap intervals over seeds. The individual
interval endpoints and the medians need not add up.

Use the true-best rank and top-five rate to see whether a good point was
available but poorly ranked. Ranks here use competition ranking, so ties
share a rank. The noise on the observation that won the raw-score ranking
is also recorded. More negative values indicate that the winning
observation received a larger favourable error. The median nearest
distance describes how locally BO queried
relative to its own previous locations. Together with the best visited
gap, it helps tell whether a change in where BO sampled translated into
better search. These trajectory comparisons describe behaviour, without
isolating a causal effect of any individual GP or CMA-ES update.

In [ ]:
%%writefile -a master_code/stage_b_budget.py

KEYS = ["seed", "bend", "k", "sigma", "angle", "method"]
metrics = ("recommendation", "best_visited", "selection_loss")
checkpoints, generations = [], []
for seed in SEEDS:
    with gzip.open(block_path(seed), "rt", encoding="utf-8") as file:
        for run in json.load(file):
            key = {name: run[name] for name in KEYS}
            checkpoints.extend({**key, **r, "repeat_calls_in_full_run": run["repeat_calls"]}
                               for r in run["checkpoints"])
            generations.extend({**key, **r} for r in run.get("generations", []))
scores = pd.DataFrame(checkpoints)
ranks = pd.DataFrame(generations)
expected = len(SEEDS)*len(BENDS)*len(KINDS)*len(NOISE)*len(ANGLES)*len(METHODS)
assert len(scores) == expected * len(CHECKPOINTS)
assert not scores.duplicated(KEYS + ["call"]).any()
assert np.allclose(scores.recommendation - scores.best_visited, scores.selection_loss)

scores.to_csv(OUT / "checkpoints.csv", index=False)
ranks.to_csv(OUT / "cma_parent_diagnostics.csv", index=False)
summary = (scores.groupby(KEYS[1:] + ["call"], as_index=False)
           [list(metrics) + ["best_observed_gap", "posterior_mean_gain"]].median())
summary.to_csv(OUT / "median_gaps.csv", index=False)
rank_summary = (ranks.groupby(["bend", "k", "sigma", "angle", "generation", "call"],
                              as_index=False)
                .agg(mean_parent_error=("parent_error", "mean"),
                     median_clean_spread=("clean_spread", "median"),
                     median_parent_margin=("clean_parent_margin", "median"),
                     median_search_sd=("search_sd", "median")))
rank_summary.to_csv(OUT / "cma_parent_summary.csv", index=False)

In [ ]:
%%writefile -a master_code/stage_b_budget.py

GROUPS = ["bend", "k", "sigma", "angle", "method"]
bootstrap_rng = np.random.default_rng(4569)
resamples = (bootstrap_rng.integers(0, len(SEEDS), size=(10000, len(SEEDS)))
             if len(SEEDS) > 1 else None)
changes = []
for key, group in scores.groupby(GROUPS):
    earlier = group[group["call"] == 96].set_index("seed").loc[SEEDS, list(metrics)]
    later = group[group["call"] == 192].set_index("seed").loc[SEEDS, list(metrics)]
    delta = (later - earlier).to_numpy()
    assert np.all(delta[:, 1] <= 0)
    np.testing.assert_allclose(delta[:, 0], delta[:, 1] + delta[:, 2], atol=1e-12)
    bounds = (np.quantile(delta[resamples].mean(axis=1), [.025, .975], axis=0)
              if resamples is not None else np.full((2, len(metrics)), np.nan))
    for j, metric in enumerate(metrics):
        changes.append(dict(zip(GROUPS, key), metric=metric, n=len(SEEDS),
                            mean_change=delta[:, j].mean(), low_95=bounds[0, j],
                            high_95=bounds[1, j], fraction_improved=np.mean(delta[:, j] < 0)))
budget_effects = pd.DataFrame(changes)
budget_effects.to_csv(OUT / "paired_changes_192_minus_96.csv", index=False)

behaviour = (scores.groupby(GROUPS + ["call"], as_index=False)
             .agg(median_nearest_distance=("nearest_distance", "median"),
                  median_true_best_rank=("true_best_rule_rank", "median"),
                  median_true_best_observed_rank=("true_best_observed_rank", "median"),
                  true_best_top5_rate=("true_best_in_top5", "mean"),
                  median_best_observed_noise=("best_observed_noise", "median"),
                  median_posterior_mean_gain=("posterior_mean_gain", "median")))
behaviour.to_csv(OUT / "search_and_selection_summary.csv", index=False)

example = summary[(summary.sigma == max(NOISE)) & (summary.bend == .75)
                  & (summary.angle == 45)]
display(example[["method", "call", *metrics]].style.format(
    {m: "{:.3e}" for m in metrics}))
focus = budget_effects[(budget_effects.sigma == max(NOISE))
                       & (budget_effects.metric == "recommendation")]
display(focus[["bend", "angle", "method", "mean_change", "low_95", "high_95"]]
        .style.format({col: "{:.3e}" for col in ("mean_change", "low_95", "high_95")}))

### Plot the budget comparison and search diagnostics

The main plots show the strongest noise level. To display a different level,
change `PLOT_SIGMA` to one of the values in `NOISE`. The BO diagnostic
shows clean and high-noise trajectories at the rotated orientation to keep
four curves per panel. The CMA-ES diagnostic shows the aligned orientation
because its paired rotated run has the same ranking behaviour.

In [ ]:
%%writefile -a master_code/stage_b_budget.py

PLOT_SIGMA = max(NOISE)
for metric, label in METRIC_LABELS.items():
    fig, axes = figure_grid(len(BENDS), len(ANGLES),
                            f"{label} with noise SD {noise_label(PLOT_SIGMA)}",
                            "Objective calls", f"Median {label.lower()}", method_handles())
    for i, bend in enumerate(BENDS):
        for j, angle in enumerate(ANGLES):
            ax = axes[i, j]
            panel = summary[(summary.bend == bend) & (summary.angle == angle)
                            & (summary.sigma == PLOT_SIGMA)]
            for method in METHODS:
                values = panel[panel.method == method].set_index("call")[metric].reindex(CHECKPOINTS)
                ax.plot(CHECKPOINTS, metric_scale(ax, metric, values), color=COLORS[method],
                        marker=MARKERS[method], ms=5)
            ax.set_title(f"{SHAPES[bend]}, {angle}°")
            ax.set_xticks(CHECKPOINTS)
    if metric == "selection_loss":
        axes[0, 0].set_ylim(bottom=0)
    finish_figure(fig, OUT / f"{metric}-budget-{PLOT_SIGMA:g}.png")

In [ ]:
%%writefile -a master_code/stage_b_budget.py

def behaviour_grid(title, row_labels, xlabel, handles):
    fig, axes = figure_grid(2, len(BENDS), title, xlabel, "", handles, row_labels)
    for j, bend in enumerate(BENDS):
        axes[0, j].set_title(SHAPES[bend])
    return fig, axes


rank_colors = plt.get_cmap("viridis")(np.linspace(.14, .82, len(NOISE)))
handles = [Line2D([], [], label=noise_label(s), color=c) for s, c in zip(NOISE, rank_colors)]
fig, axes = behaviour_grid("CMA-ES parent selection and search scale (0°)",
                           ("Mean parent error", "Median search SD"),
                           "Objective calls at generation end", handles)
for j, bend in enumerate(BENDS):
    for sigma, color in zip(NOISE, rank_colors):
        panel = rank_summary[(rank_summary.bend == bend) & (rank_summary.angle == 0)
                             & (rank_summary.sigma == sigma)]
        panel = panel.sort_values("call")
        axes[0, j].plot(panel["call"], panel.mean_parent_error, color=color)
        axes[1, j].plot(panel["call"], panel.median_search_sd, color=color)
    axes[0, j].axhline(.5, color=".55", ls="--", lw=.9)
    axes[0, j].set_ylim(-.02, 1)
    axes[1, j].set_yscale("log")
    for ax in axes[:, j]:
        ax.axvline(96, color=".7", ls=":", lw=1)
finish_figure(fig, OUT / "cma-search-diagnostics.png")

BO_METHODS = ("ARD BO", "Isotropic BO")
noise_styles = ((0., "-"), (max(NOISE), "--"))
handles = method_handles(BO_METHODS) + [
    Line2D([], [], color="black", ls=ls, label=f"Noise SD {noise_label(sigma)}")
    for sigma, ls in noise_styles]
fig, axes = behaviour_grid("BO sampling locality and finalist coverage (45°)",
                           ("Median nearest distance", "True best ranked in top five"),
                           "Objective calls at checkpoint", handles)
for j, bend in enumerate(BENDS):
    for method in BO_METHODS:
        for sigma, ls in noise_styles:
            panel = behaviour[(behaviour.bend == bend) & (behaviour.angle == 45)
                              & (behaviour.method == method) & (behaviour.sigma == sigma)]
            panel = panel.sort_values("call")
            for i, field in enumerate(("median_nearest_distance", "true_best_top5_rate")):
                axes[i, j].plot(panel["call"], panel[field], color=COLORS[method],
                                ls=ls, marker=MARKERS[method], ms=4)
    axes[1, j].set_ylim(-.03, 1.03)
    for ax in axes[:, j]:
        ax.set_xticks(CHECKPOINTS)
finish_figure(fig, OUT / "bo-search-diagnostics.png")

Read the budget contrasts alongside the recommendation and best-visited
curves. If a method keeps finding better points and its selection loss
grows, a larger search budget has improved the candidate set without
necessarily improving its return value. If both true gaps flatten, check
the CMA-ES parent diagnostics or the BO query distances to see how the
search behaved. The results apply to the selected high-conditioning
geometries and the observation noise used here.

In [ ]:
run_stage("stage_b_budget")

## Stage C - fixed repeat allocations

Source reused from `stage_c_replication.ipynb`. The configuration above controls execution. Code cells in this section prepare the stage script; the final cell runs it when selected.

In [ ]:
%%writefile master_code/stage_c_fixed.py
from pathlib import Path
import os

MASTER_PROFILE = globals().get("MASTER_PROFILE", os.environ.get("OPT_PROFILE", "smoke"))
MASTER_ROOT = Path(globals().get("MASTER_ROOT", os.environ.get("OPT_RESULTS_ROOT", "optimisation_results"))).resolve()
MASTER_INCLUDE_TRANSFER = globals().get("MASTER_INCLUDE_TRANSFER", os.environ.get("OPT_INCLUDE_TRANSFER", "1") == "1")
if MASTER_PROFILE not in ("full", "smoke"):
    raise ValueError("Choose full or smoke.")

In [ ]:
%%writefile -a master_code/stage_c_fixed.py

import gzip
import json
import math
import os
import random
import runpy
import sys
from dataclasses import dataclass
from importlib.metadata import distribution, version
from math import cos, sin, hypot, tau
from pathlib import Path
from time import perf_counter

for variable in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS"):
    os.environ[variable] = "1"

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.linalg import cholesky, cho_solve, solve_triangular
from scipy.optimize import minimize
from scipy.spatial.distance import cdist

pure_cma = distribution("cma").locate_file("cma/more_algorithms/purecma.py")
CMAES = runpy.run_path(str(pure_cma))["CMAES"]

PROFILE = MASTER_PROFILE
OUTPUT_ROOT = MASTER_ROOT / "stage_c_fixed"
BUDGET, POPULATION, POOL = 96, 6, 1024
SEARCH_PREFIX, REPEAT_START = 72, 48
FINALISTS, PER_FINALIST = 4, 6
JITTER, LOG_FLOOR = 1e-7, 1e-14
METHODS = ("CMA-ES", "ARD BO", "Isotropic BO")
BASELINE, VERIFY, LATE = "Search only", "Finalist verification", "Late repeats"
POLICIES = (BASELINE, VERIFY, LATE)
if PROFILE == "full":
    SEEDS = list(range(9600, 9624))
    BENDS, ANGLES, NOISE = (0., .75), (0, 45), (1e-3, 1e-2)
elif PROFILE == "smoke":
    SEEDS = [9600]
    BENDS, ANGLES, NOISE = (.75,), (45,), (1e-2,)
else:
    raise ValueError("Use 'smoke' or 'full'.")
assert SEARCH_PREFIX + FINALISTS * PER_FINALIST == BUDGET
assert SEARCH_PREFIX + (SEARCH_PREFIX - REPEAT_START) == BUDGET
assert all(n % POPULATION == 0 for n in (REPEAT_START, SEARCH_PREFIX, BUDGET))

SOFTWARE = {"python": sys.version.split()[0], **{
    package: version(package) for package in ("numpy", "scipy", "pandas", "matplotlib", "cma")
}}
SETTINGS = dict(schema=1, seeds=SEEDS, bends=list(BENDS), angles=list(ANGLES),
                noise=list(NOISE), k=100, budget=BUDGET, prefix=SEARCH_PREFIX,
                repeat_start=REPEAT_START, finalists=FINALISTS, per_finalist=PER_FINALIST,
                pool=POOL, population=POPULATION, jitter=JITTER, software=SOFTWARE)
OUT = OUTPUT_ROOT / PROFILE
print(f"{PROFILE}: {len(SEEDS)} blocks under {OUT}")
print(SOFTWARE)

### Objective and search machinery

The valley, normalisation, GP and acquisition refinement are from Stage B.
Search noise is paired by proposal number. The deliberately repeated
measurements use an independent stream and never reuse a search noise draw.
Within a policy each objective call has an independent perturbation.
The same tapes and search seeds are paired across orientations and policies.

In [ ]:
%%writefile -a master_code/stage_c_fixed.py

def rotate(point, angle):
    x, y = point
    return cos(angle) * x - sin(angle) * y, sin(angle) * x + cos(angle) * y


@dataclass(frozen=True)
class Valley:
    centre: tuple
    angle: float
    curvature_ratio: float = 100.0
    bend: float = 0.0

    @property
    def normaliser(self):
        # Under uniform disk sampling E[X^2] = 1/4 and E[X^4] = 1/8.
        # Terms odd in either coordinate integrate to zero.
        a, b = self.centre
        if self.bend == 0:
            k = self.curvature_ratio
            return (1 + k) / 4 + a*a + k*b*b
        eu2 = .25 + a*a
        ev2 = .25 + b*b
        eu4 = .125 + 1.5*a*a + a**4
        return eu2 + self.curvature_ratio * (
            ev2 + 2*self.bend*b*eu2 + self.bend**2*eu4
        )

    def canonical_gap(self, x, y):
        a, b = self.centre
        u, v = x - a, y - b
        return (u**2 + self.curvature_ratio*(v - self.bend*u**2)**2) / self.normaliser

    def clean(self, point):
        if hypot(*point) > 1.0 + 1e-12:
            raise ValueError("Point outside the unit disk")
        x, y = rotate(point, -self.angle)
        return self.canonical_gap(x, y)

    def observe(self, point, sigma, rng):
        return self.clean(point) + sigma * rng.gauss(0.0, 1.0)


def rotation(degree):
    angle = math.radians(degree)
    return np.array([rotate((1, 0), angle), rotate((0, 1), angle)]).T


def disk(rng, size):
    u = rng.random((size, 2))
    angle = 2 * np.pi * u[:, 0]
    return np.sqrt(u[:, 1:]) * np.column_stack((np.cos(angle), np.sin(angle)))

In [ ]:
%%writefile -a master_code/stage_c_fixed.py

def covariance(x, y, theta):
    lengths = np.exp(theta[1:])
    return np.exp(theta[0] - .5 * cdist(x / lengths, y / lengths, "sqeuclidean"))


def fit(x, values, theta, refit, noise_sd=0.0, jitter=JITTER):
    output_scale = values.std() or 1
    y = (values - values.mean()) / output_scale
    ident = np.eye(len(x))

    def factor(parameters):
        signal = covariance(x, x, parameters)
        lower = cholesky(signal + (jitter + (noise_sd/output_scale)**2) * ident, lower=True)
        return signal, lower, cho_solve((lower, True), y)

    def likelihood(parameters):
        signal, lower, alpha = factor(parameters)
        loss = .5 * (y @ alpha) + np.log(np.diag(lower)).sum()
        weights = .5 * (cho_solve((lower, True), ident) - np.outer(alpha, alpha)) * signal
        gradient = [weights.sum()]
        if len(parameters) == 2:
            gradient.append((weights * cdist(x, x, "sqeuclidean")).sum() / np.exp(2*parameters[1]))
        else:
            for axis, log_length in enumerate(parameters[1:]):
                diff = x[:, axis, None] - x[:, axis]
                gradient.append((weights * diff**2).sum() / np.exp(2*log_length))
        return loss, np.array(gradient)

    status = -1
    if refit:
        bounds = [(-6, 6)] + [(math.log(.02), math.log(2))] * (len(theta)-1)
        result = minimize(likelihood, theta, jac=True, bounds=bounds, method="L-BFGS-B",
                          options={"maxiter": 50})
        theta, status = result.x, int(result.status)
    _, lower, alpha = factor(theta)
    return theta, lower, alpha, status


def predict(x, candidates, theta, lower, alpha):
    cross = covariance(x, candidates, theta)
    projected = solve_triangular(lower, cross, lower=True, check_finite=False)
    variance = np.maximum(0, np.exp(theta[0]) - np.einsum("ij,ij->j", projected, projected))
    return cross.T @ alpha, np.sqrt(variance)

In [ ]:
%%writefile -a master_code/stage_c_fixed.py

def disk_constraint(point):
    return 1 - point @ point


def disk_jacobian(point):
    return -2 * point


def refined_candidate(x, pool, theta, lower, alpha, pool_lcb):
    """Refine two promising pool candidates with the same fitted LCB."""
    precision = np.broadcast_to(np.exp(-2 * theta[1:]), (2,))
    amplitude = np.exp(theta[0])

    def lcb(p):
        delta = x - p
        kernel = amplitude * np.exp(-.5 * ((delta ** 2) @ precision))
        derivative = kernel[:, None] * delta * precision
        projected = solve_triangular(lower, kernel, lower=True, check_finite=False)
        weights = solve_triangular(lower.T, projected, lower=False, check_finite=False)
        # A tiny variance floor avoids an undefined derivative at zero.
        variance = amplitude - projected @ projected
        sd = np.sqrt(max(variance, 1e-20))
        mean_gradient = derivative.T @ alpha
        gradient = mean_gradient + 2 * (derivative.T @ weights) / sd if variance > 1e-20 else mean_gradient
        return kernel @ alpha - 2 * sd, gradient

    best = pool[int(np.argmin(pool_lcb))]
    best_lcb = float(np.min(pool_lcb))
    starts = [best]
    distant = np.linalg.norm(pool - best, axis=1) >= .05
    if distant.any():
        starts.append(pool[np.argmin(np.where(distant, pool_lcb, np.inf))])
    warnings, improved = 0, False
    for start in starts:
        result = minimize(lcb, start, jac=True, method="SLSQP",
                          constraints=[{"type": "ineq", "fun": disk_constraint,
                                        "jac": disk_jacobian}],
                          options={"maxiter": 25, "ftol": 1e-10})
        warnings += int(not result.success)
        if not np.all(np.isfinite(result.x)) or np.linalg.norm(result.x) > 1 + 1e-8:
            continue
        candidate = result.x / max(1., np.linalg.norm(result.x))
        # Compare with the stable triangular-factor prediction used for the pool.
        mean, sd = predict(x, candidate[None, :], theta, lower, alpha)
        score = float((mean - 2 * sd)[0])
        if score < best_lcb - 1e-12:
            best, best_lcb, improved = candidate, score, True
    return best, warnings, int(improved)

In [ ]:
%%writefile -a master_code/stage_c_fixed.py

def run_coupled_cma(evaluate, q, seed, budget=BUDGET):
    """Use C^(1/2) Q z for matched draws, retaining the reference tell update."""
    state = CMAES([0., 0.], .4, popsize=POPULATION)
    rng = np.random.default_rng(seed)
    points, scores, states = [], [], []
    rejected = 0
    for _ in range(budget // POPULATION):
        state.C.update_eigensystem(state.counteval, state.params.lazy_gap_evals)
        basis = np.asarray(state.C.eigenbasis)
        root = (basis * np.sqrt(state.C.eigenvalues)) @ basis.T
        population = []
        while len(population) < POPULATION:
            point = np.asarray(state.xmean) + state.sigma * (root @ (q @ rng.standard_normal(2)))
            if np.linalg.norm(point) > 1:
                rejected += 1
                if rejected > 10000:
                    raise RuntimeError("CMA rejection cap")
                continue
            population.append(point.tolist())
        values = [evaluate(p) for p in population]
        state.tell(population, values)
        points.extend(population)
        scores.extend(values)
        covariance = np.array([[state.C[i][j] for j in range(2)] for i in range(2)])
        eigvals, vectors = np.linalg.eigh(covariance)
        major_axis = vectors[:, -1]
        states.append(dict(call=state.counteval, ratio=float(eigvals[-1]/eigvals[0]),
                           axis_error=float(np.degrees(np.arccos(np.clip(abs(major_axis @ q[:, 0]), 0, 1)))),
                           sigma=state.sigma, covariance=covariance.tolist(), mean=state.xmean))
    assert state.counteval == budget
    return points, scores, dict(states=states, rejections=rejected)

In [ ]:
%%writefile -a master_code/stage_c_fixed.py

class Evaluator:
    def __init__(self, valley, sigma, search_noise, repeat_noise):
        self.valley, self.sigma = valley, sigma
        self.search_noise, self.repeat_noise = search_noise, repeat_noise
        self.search_count = self.repeat_count = 0
        self.trace = []

    def measure(self, point, repeat=False):
        if repeat:
            noise = self.repeat_noise[self.repeat_count]
            self.repeat_count += 1
        else:
            noise = self.search_noise[self.search_count]
            self.search_count += 1
        value = float(self.valley.clean(point) + self.sigma * noise)
        self.trace.append(dict(point=np.asarray(point).tolist(), observed=value, repeat=repeat))
        return value


def posterior_scores(points, values, theta, noise_sd, candidates):
    x = np.asarray(points)
    _, lower, alpha, _ = fit(x, np.asarray(values), theta, False, noise_sd=noise_sd)
    means, _ = predict(x, np.asarray(candidates), theta, lower, alpha)
    return means


def bo_search(evaluator, q, initial, seed, isotropic, late):
    rng = np.random.default_rng(seed)
    points = list(initial)
    training_x = list(initial)
    training_y = [evaluator.measure(p) for p in initial]
    theta = np.log([1, .4] if isotropic else [1, .4, .4])
    checkpoints = {}
    limit = SEARCH_PREFIX if late else BUDGET
    while len(points) < limit:
        x = np.asarray(training_x)
        theta, lower, alpha, _ = fit(
            x, np.asarray(training_y), theta, len(x) % 6 == 0, noise_sd=evaluator.sigma
        )
        pool = disk(rng, POOL) @ q.T
        mean, sd = predict(x, pool, theta, lower, alpha)
        candidate, _, _ = refined_candidate(x, pool, theta, lower, alpha, mean - 2 * sd)
        points.append(candidate)
        training_x.append(candidate)
        training_y.append(evaluator.measure(candidate))
        if late and len(points) > REPEAT_START:
            training_x.append(candidate)
            training_y.append(evaluator.measure(candidate, repeat=True))
        if len(points) in (SEARCH_PREFIX, limit):
            scores = posterior_scores(training_x, training_y, theta, evaluator.sigma, points)
            checkpoints[len(points)] = dict(scores=scores, theta=theta.copy())
    return dict(points=np.asarray(points), checkpoints=checkpoints, evaluator=evaluator)


def cma_search(evaluator, q, seed, late):
    def evaluate_candidate(point):
        y = evaluator.measure(point)
        if late and evaluator.search_count > REPEAT_START:
            y = (y + evaluator.measure(point, repeat=True)) / 2
        return y

    limit = SEARCH_PREFIX if late else BUDGET
    points, scores, _ = run_coupled_cma(evaluate_candidate, q, seed, budget=limit)
    checkpoints = {n: {"scores": np.asarray(scores[:n])} for n in (SEARCH_PREFIX, limit)}
    return dict(points=np.asarray(points), checkpoints=checkpoints, evaluator=evaluator,
                update_scores=np.asarray(scores))


def distinct_top(points, scores):
    indices, seen = [], set()
    for index in np.argsort(scores, kind="stable"):
        key = tuple(points[index])
        if key not in seen:
            indices.append(int(index))
            seen.add(key)
        if len(indices) == FINALISTS:
            return np.asarray(indices)
    raise ValueError("Fewer distinct visited locations than finalists.")


def verify_finalists(baseline, evaluator, method):
    points = baseline["points"][:SEARCH_PREFIX]
    state = baseline["checkpoints"][SEARCH_PREFIX]
    indices = distinct_top(points, state["scores"])
    finalists = points[indices]
    fresh = np.array([[evaluator.measure(p, repeat=True) for _ in range(PER_FINALIST)]
                      for p in finalists])
    chosen = int(np.argmin(fresh.mean(axis=1)))
    result = dict(indices=indices.tolist(), chosen_index=int(indices[chosen]))
    if method != "CMA-ES":
        # The same observations support this secondary recommendation.
        train_x = np.vstack([points, np.repeat(finalists, PER_FINALIST, axis=0)])
        search_y = [row["observed"] for row in baseline["evaluator"].trace[:SEARCH_PREFIX]]
        train_y = np.concatenate([search_y, fresh.ravel()])
        gp_scores = posterior_scores(train_x, train_y, state["theta"],
                                     evaluator.sigma, finalists)
        result["gp_chosen_index"] = int(indices[np.argmin(gp_scores)])
    return result

### Run and save one block at a time

The outcome and diagnostics use the clean objective only after each
policy has made its recommendation. For verification, selection loss
splits into two parts. Shortlist loss measures the gap between the best
finalist and the best point visited. Final ranking loss measures the
gap between the returned point and the best finalist.

In [ ]:
%%writefile -a master_code/stage_c_fixed.py

def assess(clean, chosen, **labels):
    best = float(np.min(clean))
    recommendation = float(clean[chosen])
    return dict(**labels, recommendation=recommendation, best_visited=best,
                selection_loss=recommendation - best, chosen_index=int(chosen))


def parent_rows(run, valley, **labels):
    clean = np.array([valley.clean(p) for p in run["points"]])
    errors = []
    mu = POPULATION // 2
    for start in range(0, len(clean), POPULATION):
        end = start + POPULATION
        clean_top = set(np.argsort(clean[start:end], kind="stable")[:mu])
        selected = set(np.argsort(run["update_scores"][start:end], kind="stable")[:mu])
        errors.append(dict(**labels, generation=end // POPULATION,
                           parent_error=len(selected - clean_top) / mu))
    return errors


def run_block(seed):
    centre = rotate((.4, 0), random.Random(seed).uniform(0, tau))
    search_rng = np.random.default_rng(np.random.SeedSequence([seed, 40117]))
    search_noise = search_rng.standard_normal(BUDGET)
    repeat_rng = np.random.default_rng(np.random.SeedSequence([seed, 73093]))
    repeat_noise = repeat_rng.standard_normal(BUDGET - SEARCH_PREFIX)
    results, checks, parents, histories = [], [], [], []
    for bend in BENDS:
        for sigma in NOISE:
            for angle in ANGLES:
                q = rotation(angle)
                valley = Valley(centre, math.radians(angle), 100, bend)
                initial = None
                for method in METHODS:
                    labels = dict(seed=seed, bend=bend, sigma=sigma, angle=angle,
                                  method=method)
                    runs = {}
                    for policy in (BASELINE, LATE):
                        meter = Evaluator(valley, sigma, search_noise, repeat_noise)
                        late = policy == LATE
                        if method == "CMA-ES":
                            run = cma_search(meter, q, seed + 30000, late)
                            initial = run["points"][:POPULATION]
                        else:
                            run = bo_search(meter, q, initial, seed + 20000,
                                            method == "Isotropic BO", late)
                        limit = SEARCH_PREFIX if late else BUDGET
                        assert (meter.search_count, meter.repeat_count) == (limit, BUDGET - limit)
                        scores = run["checkpoints"][limit]["scores"]
                        clean = np.array([valley.clean(p) for p in run["points"]])
                        row = assess(clean, np.argmin(scores), **labels, policy=policy,
                                     search_calls=limit, repeat_calls=BUDGET-limit)
                        results.append(row)
                        histories.append(dict(**labels, policy=policy, trace=meter.trace))
                        if method == "CMA-ES":
                            parents.extend(parent_rows(run, valley, **labels, policy=policy))
                        runs[policy] = run
                    baseline, late = runs[BASELINE], runs[LATE]
                    np.testing.assert_array_equal(baseline["points"][:REPEAT_START],
                                                  late["points"][:REPEAT_START])
                    assert (baseline["evaluator"].trace[:REPEAT_START] ==
                            late["evaluator"].trace[:REPEAT_START])
                    if method == "CMA-ES":
                        first_observations = [row["observed"] for row in
                                              baseline["evaluator"].trace[:POPULATION]]
                    else:
                        assert first_observations == [row["observed"] for row in
                                                      baseline["evaluator"].trace[:POPULATION]]

                    meter = Evaluator(valley, sigma, search_noise, repeat_noise)
                    verified = verify_finalists(baseline, meter, method)
                    assert meter.repeat_count == BUDGET - SEARCH_PREFIX
                    assert meter.search_count == 0
                    prefix = baseline["points"][:SEARCH_PREFIX]
                    clean = np.array([valley.clean(p) for p in prefix])
                    row = assess(clean, verified["chosen_index"], **labels, policy=VERIFY,
                                 search_calls=SEARCH_PREFIX, repeat_calls=meter.repeat_count)
                    results.append(row)
                    best_finalist = float(np.min(clean[verified["indices"]]))
                    prefix_chosen = int(np.argmin(baseline["checkpoints"][SEARCH_PREFIX]["scores"]))
                    checks.append(dict(**labels,
                        shortlist_loss=best_finalist - row["best_visited"],
                        final_ranking_loss=row["recommendation"] - best_finalist,
                        prefix_recommendation=float(clean[prefix_chosen]),
                        verified_recommendation=row["recommendation"],
                        gp_verified_recommendation=(float(clean[verified["gp_chosen_index"]])
                                                    if "gp_chosen_index" in verified else None)))
                    histories.append(dict(**labels, policy=VERIFY, **verified,
                                          trace=meter.trace))
    return dict(results=results, verification=checks, parents=parents, histories=histories)


def block_path(seed):
    return OUT / "blocks" / f"{seed}.json.gz"


def save_blocks():
    (OUT / "blocks").mkdir(parents=True, exist_ok=True)
    manifest = OUT / "settings.json"
    if manifest.exists():
        if json.loads(manifest.read_text(encoding="utf-8")) != SETTINGS:
            raise ValueError("Cached settings or software differ. Choose a new OUTPUT_ROOT.")
    else:
        manifest.write_text(json.dumps(SETTINGS, indent=2), encoding="utf-8")
    for i, seed in enumerate(SEEDS, 1):
        path = block_path(seed)
        if path.exists():
            print(f"[{i}/{len(SEEDS)}] Reusing block {seed}", flush=True)
            continue
        started = perf_counter()
        block = run_block(seed)
        temp = path.with_suffix(".tmp")
        with gzip.open(temp, "wt", encoding="utf-8") as file:
            json.dump(block, file, allow_nan=False, separators=(",", ":"))
        temp.replace(path)
        print(f"[{i}/{len(SEEDS)}] Saved block {seed} ({perf_counter()-started:.1f} s)",
              flush=True)

In [ ]:
%%writefile -a master_code/stage_c_fixed.py

save_blocks()

### Analyse the results

The primary effect is the within-seed $\log_{10}$ ratio of true recommendation
gaps for each repeat policy versus 96 search calls. Negative favours the
repeat policy. For example, $-0.301$ means half the gap on a geometric mean
basis. The table also gives absolute gaps and the proportion of seeds
improved. Report the primary setting first and treat the other settings
and diagnostics as sensitivity analyses. Bootstrap intervals are pointwise
and the one-seed smoke profile suppresses them.

On the raw scale, the change in returned gap is the sum of the change in
best visited gap and the change in selection loss. To judge late repeats
for CMA-ES, compare its parent errors with the baseline at the same
generation numbers (9--12). That reveals whether rankings improved, while
the 96-call comparison includes the cost of completing four fewer updates.

In [ ]:
%%writefile -a master_code/stage_c_fixed.py

result_rows, check_rows, parent_data = [], [], []
for seed in SEEDS:
    with gzip.open(block_path(seed), "rt", encoding="utf-8") as file:
        block = json.load(file)
    result_rows.extend(block["results"])
    check_rows.extend(block["verification"])
    parent_data.extend(block["parents"])
results = pd.DataFrame(result_rows)
verification = pd.DataFrame(check_rows)
parents = pd.DataFrame(parent_data)
geometry = ["bend", "sigma", "angle", "method"]
metrics = ["recommendation", "best_visited", "selection_loss"]
assert len(results) == len(SEEDS)*len(BENDS)*len(NOISE)*len(ANGLES)*len(METHODS)*len(POLICIES)
assert not results.duplicated(["seed", *geometry, "policy"]).any()
assert (results.search_calls + results.repeat_calls == BUDGET).all()
matched = results[results.policy == VERIFY].merge(
    verification, on=["seed", *geometry], validate="one_to_one")
np.testing.assert_allclose(matched.selection_loss,
                           matched.shortlist_loss + matched.final_ranking_loss, atol=1e-14)

results.to_csv(OUT / "run_gaps.csv", index=False)
verification.to_csv(OUT / "verification_diagnostics.csv", index=False)
parents.to_csv(OUT / "cma_parent_errors.csv", index=False)
medians = results.groupby([*geometry, "policy"], as_index=False)[metrics].median()
medians.to_csv(OUT / "median_gaps.csv", index=False)
ranks = parents.groupby(["bend", "sigma", "angle", "policy", "generation"],
                        as_index=False).parent_error.mean()
ranks.to_csv(OUT / "cma_parent_summary.csv", index=False)

bootstrap_rng = np.random.default_rng(4670)
draws = (bootstrap_rng.integers(0, len(SEEDS), size=(10000, len(SEEDS)))
         if len(SEEDS) > 1 else None)


def effect(values, **labels):
    bounds = (np.quantile(values[draws].mean(axis=1), [.025, .975]) if draws is not None
              else (np.nan, np.nan))
    return dict(**labels, n=len(values), mean=values.mean(), low_95=bounds[0],
                high_95=bounds[1], fraction_improved=np.mean(values < 0))


effects = []
for key, group in results.groupby(geometry):
    labels = dict(zip(geometry, key))
    by_policy = {p: table.set_index("seed").loc[SEEDS, metrics].to_numpy()
                 for p, table in group.groupby("policy")}
    for policy, reference in ((VERIFY, BASELINE), (LATE, BASELINE), (VERIFY, LATE)):
        a, b = by_policy[policy], by_policy[reference]
        delta = a - b
        np.testing.assert_allclose(delta[:, 0], delta[:, 1] + delta[:, 2], atol=1e-14)
        for j, metric in enumerate(metrics):
            effects.append(effect(delta[:, j], **labels, policy=policy, reference=reference,
                                  metric=metric, scale="raw"))
        log_ratio = np.log10(np.maximum(a[:, 0], LOG_FLOOR)) - np.log10(
            np.maximum(b[:, 0], LOG_FLOOR))
        effects.append(effect(log_ratio, **labels, policy=policy, reference=reference,
                              metric="recommendation", scale="log10"))
paired = pd.DataFrame(effects)
paired.to_csv(OUT / "paired_policy_effects.csv", index=False)

# Within-prefix verification effect and secondary BO reranking.
within_prefix = []
for key, group in verification.groupby(geometry):
    labels = dict(zip(geometry, key))
    group = group.set_index("seed").loc[SEEDS]
    a = group.verified_recommendation.to_numpy()
    b = group.prefix_recommendation.to_numpy()
    within_prefix.append(effect(a - b, **labels, comparison="Verification vs. 72-call prefix"))
    if labels["method"] != "CMA-ES":
        gp = group.gp_verified_recommendation.to_numpy()
        within_prefix.append(effect(gp - a, **labels, comparison="GP vs. fresh-mean reranking"))
pd.DataFrame(within_prefix).to_csv(OUT / "verification_paired_effects.csv", index=False)

def primary(data):
    return data[(data.bend == .75) & (data.angle == 45) & (data.sigma == 1e-2)]


display(primary(medians)[["method", "policy", *metrics]]
        .style.format({m: "{:.3e}" for m in metrics}))
selected = primary(paired)
selected = selected[(selected.scale == "log10") & (selected.reference == BASELINE)]
display(selected[["method", "policy", "mean", "low_95", "high_95", "fraction_improved"]]
        .round(3))
parts = (primary(verification).groupby("method", as_index=False)
         [["shortlist_loss", "final_ranking_loss"]].mean())
display(parts.style.format({m: "{:.3e}" for m in ("shortlist_loss", "final_ranking_loss")}))

### Figures for the primary setting

The first figure shows the paired recommendation effect. The second shows
how much of verification's mean selection loss came from shortlisting and
final ranking. The parent plot shows how double evaluations affected the
CMA-ES updates at matched generation numbers.

In [ ]:
%%writefile -a master_code/stage_c_fixed.py

colors = {VERIFY: "#0072b2", LATE: "#d55e00"}
fig, ax = plt.subplots(figsize=(7.5, 4.1), layout="constrained")
for i, method in enumerate(METHODS):
    for policy, offset in ((VERIFY, -.12), (LATE, .12)):
        row = selected[(selected.method == method) & (selected.policy == policy)].iloc[0]
        y = i + offset
        if np.isfinite(row.low_95):
            ax.hlines(y, row.low_95, row.high_95, color=colors[policy], lw=1.8)
        ax.scatter(row["mean"], y, color=colors[policy], label=policy if i == 0 else None)
ax.axvline(0, color=".5", ls="--", lw=1)
ax.set_yticks(range(len(METHODS)), METHODS)
ax.invert_yaxis()
ax.set_xlabel("Mean paired log10 ratio of true recommendation gaps")
ax.set_title("Repeat policies versus 96 search calls")
ax.legend( ncol=2, frameon=False)
fig.savefig(OUT / "paired-recommendation-effects.png", dpi=220)
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(7.5, 4.1), layout="constrained")
ax.bar(parts.method, parts.shortlist_loss, color="#0072b2", label="Shortlist loss")
ax.bar(parts.method, parts.final_ranking_loss, bottom=parts.shortlist_loss,
       color="#e69f00", label="Final ranking loss")
ax.set_ylabel("Mean true gap lost")
ax.set_title("Where finalist verification lost solution quality")
ax.legend(ncol=2, frameon=False)
fig.savefig(OUT / "verification-loss-components.png", dpi=220)
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(7.5, 3.8), layout="constrained")
for policy, color in ((BASELINE, "#666666"), (LATE, colors[LATE])):
    panel = primary(ranks[ranks.policy == policy]).sort_values("generation")
    ax.plot(panel.generation, panel.parent_error, marker="o", ms=3, color=color, label=policy)
ax.axvline(REPEAT_START / POPULATION + .5, color=".5", ls=":", lw=1)
ax.set(ylim=(0, 1), xlabel="CMA-ES generation", ylabel="Mean parent replacement rate",
       title="Parent rankings before and after late repeats")
ax.legend( ncol=2, frameon=False)
fig.savefig(OUT / "cma-late-parent-rankings.png", dpi=220)
plt.show()
plt.close(fig)

A favourable finalist effect against the 72-call prefix answers whether
verification improved selection on the same search history. The 96-call
contrast answers whether that gain paid for the 24 search proposals it
replaced. A lower CMA-ES rank error can coexist with a worse returned
solution if the budget permits too few updates. The run CSV and paired
contrasts preserve those distinctions.

In [ ]:
run_stage("stage_c_fixed")

## Stage C - adaptive allocation

Source reused from `stage_c_dynamic_evaluation.ipynb`. The configuration above controls execution. Code cells in this section prepare the stage script; the final cell runs it when selected.

In [ ]:
%%writefile master_code/stage_c_dynamic.py
from pathlib import Path
import os

MASTER_PROFILE = globals().get("MASTER_PROFILE", os.environ.get("OPT_PROFILE", "smoke"))
MASTER_ROOT = Path(globals().get("MASTER_ROOT", os.environ.get("OPT_RESULTS_ROOT", "optimisation_results"))).resolve()
MASTER_INCLUDE_TRANSFER = globals().get("MASTER_INCLUDE_TRANSFER", os.environ.get("OPT_INCLUDE_TRANSFER", "1") == "1")
if MASTER_PROFILE not in ("full", "smoke"):
    raise ValueError("Choose full or smoke.")

In [ ]:
%%writefile -a master_code/stage_c_dynamic.py

import gzip
import json
import math
import os
import random
import runpy
import sys
from dataclasses import dataclass
from importlib.metadata import distribution, version
from math import cos, sin, hypot, tau
from pathlib import Path
from time import perf_counter

for variable in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS"):
    os.environ[variable] = "1"

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.linalg import cholesky, cho_solve, solve_triangular
from scipy.optimize import minimize
from scipy.spatial.distance import cdist

pure_cma = distribution("cma").locate_file("cma/more_algorithms/purecma.py")
CMAES = runpy.run_path(str(pure_cma))["CMAES"]

PROFILE = MASTER_PROFILE
OUTPUT_ROOT = MASTER_ROOT / "stage_c_dynamic"
BUDGET, POPULATION, POOL = 96, 6, 1024
SEARCH_PREFIX, REPEAT_START = 72, 48
FINALISTS, PER_FINALIST = 4, 6
JITTER, LOG_FLOOR = 1e-7, 1e-14
METHODS = ("CMA-ES", "ARD BO", "Isotropic BO")
BASELINE, VERIFY, LATE = "Search only", "Finalist verification", "Late repeats"
CUTOFF, KG, COVERAGE = "Cutoff repeats", "Knowledge gradient", "Coverage shortlist"
MAX_CUTOFF_REPEATS, KG_NEW_POINTS, SHORTLIST_DRAWS = 12, 24, 512
SHORTLIST_TOLERANCE = .10
KG_NODES, KG_WEIGHTS = np.polynomial.hermite.hermgauss(24)
KG_NODES *= np.sqrt(2)
KG_WEIGHTS /= np.sqrt(np.pi)
METHOD_POLICIES = {
    "CMA-ES": (BASELINE, VERIFY, LATE, CUTOFF),
    "ARD BO": (BASELINE, VERIFY, LATE, KG, COVERAGE),
    "Isotropic BO": (BASELINE, VERIFY, LATE, KG, COVERAGE),
}
if PROFILE == "full":
    SEEDS = list(range(9800, 9824))
    BENDS, ANGLES, NOISE = (0., .75), (0, 45), (1e-3, 1e-2)
elif PROFILE == "smoke":
    SEEDS = [9800]
    BENDS, ANGLES, NOISE = (.75,), (45,), (1e-2,)
else:
    raise ValueError("Use 'smoke' or 'full'.")
assert SEARCH_PREFIX + FINALISTS * PER_FINALIST == BUDGET
assert SEARCH_PREFIX + (SEARCH_PREFIX - REPEAT_START) == BUDGET
assert all(n % POPULATION == 0 for n in (REPEAT_START, SEARCH_PREFIX, BUDGET))

SOFTWARE = {"python": sys.version.split()[0], **{
    package: version(package) for package in ("numpy", "scipy", "pandas", "matplotlib", "cma")
}}
SETTINGS = dict(schema=1, seeds=SEEDS, bends=list(BENDS), angles=list(ANGLES),
                noise=list(NOISE), k=100, budget=BUDGET, prefix=SEARCH_PREFIX,
                repeat_start=REPEAT_START, finalists=FINALISTS, per_finalist=PER_FINALIST,
                pool=POOL, population=POPULATION, jitter=JITTER, software=SOFTWARE,
                cutoff_cap=MAX_CUTOFF_REPEATS, kg_new_points=KG_NEW_POINTS,
                shortlist_draws=SHORTLIST_DRAWS, shortlist_tolerance=SHORTLIST_TOLERANCE)
OUT = OUTPUT_ROOT / PROFILE
print(f"{PROFILE}: {len(SEEDS)} blocks under {OUT}")
print(SOFTWARE)

In [ ]:
%%writefile -a master_code/stage_c_dynamic.py

def rotate(point, angle):
    x, y = point
    return cos(angle) * x - sin(angle) * y, sin(angle) * x + cos(angle) * y


@dataclass(frozen=True)
class Valley:
    centre: tuple
    angle: float
    curvature_ratio: float = 100.0
    bend: float = 0.0

    @property
    def normaliser(self):
        # Under uniform disk sampling E[X^2] = 1/4 and E[X^4] = 1/8.
        # Terms odd in either coordinate integrate to zero.
        a, b = self.centre
        if self.bend == 0:
            k = self.curvature_ratio
            return (1 + k) / 4 + a*a + k*b*b
        eu2 = .25 + a*a
        ev2 = .25 + b*b
        eu4 = .125 + 1.5*a*a + a**4
        return eu2 + self.curvature_ratio * (
            ev2 + 2*self.bend*b*eu2 + self.bend**2*eu4
        )

    def canonical_gap(self, x, y):
        a, b = self.centre
        u, v = x - a, y - b
        return (u**2 + self.curvature_ratio*(v - self.bend*u**2)**2) / self.normaliser

    def clean(self, point):
        if hypot(*point) > 1.0 + 1e-12:
            raise ValueError("Point outside the unit disk")
        x, y = rotate(point, -self.angle)
        return self.canonical_gap(x, y)

    def observe(self, point, sigma, rng):
        return self.clean(point) + sigma * rng.gauss(0.0, 1.0)


def rotation(degree):
    angle = math.radians(degree)
    return np.array([rotate((1, 0), angle), rotate((0, 1), angle)]).T


def disk(rng, size):
    u = rng.random((size, 2))
    angle = 2 * np.pi * u[:, 0]
    return np.sqrt(u[:, 1:]) * np.column_stack((np.cos(angle), np.sin(angle)))

In [ ]:
%%writefile -a master_code/stage_c_dynamic.py

def covariance(x, y, theta):
    lengths = np.exp(theta[1:])
    return np.exp(theta[0] - .5 * cdist(x / lengths, y / lengths, "sqeuclidean"))


def fit(x, values, theta, refit, noise_sd=0.0, jitter=JITTER):
    output_scale = values.std() or 1
    y = (values - values.mean()) / output_scale
    ident = np.eye(len(x))

    def factor(parameters):
        signal = covariance(x, x, parameters)
        lower = cholesky(signal + (jitter + (noise_sd/output_scale)**2) * ident, lower=True)
        return signal, lower, cho_solve((lower, True), y)

    def likelihood(parameters):
        signal, lower, alpha = factor(parameters)
        loss = .5 * (y @ alpha) + np.log(np.diag(lower)).sum()
        weights = .5 * (cho_solve((lower, True), ident) - np.outer(alpha, alpha)) * signal
        gradient = [weights.sum()]
        if len(parameters) == 2:
            gradient.append((weights * cdist(x, x, "sqeuclidean")).sum() / np.exp(2*parameters[1]))
        else:
            for axis, log_length in enumerate(parameters[1:]):
                diff = x[:, axis, None] - x[:, axis]
                gradient.append((weights * diff**2).sum() / np.exp(2*log_length))
        return loss, np.array(gradient)

    status = -1
    if refit:
        bounds = [(-6, 6)] + [(math.log(.02), math.log(2))] * (len(theta)-1)
        result = minimize(likelihood, theta, jac=True, bounds=bounds, method="L-BFGS-B",
                          options={"maxiter": 50})
        theta, status = result.x, int(result.status)
    _, lower, alpha = factor(theta)
    return theta, lower, alpha, status


def predict(x, candidates, theta, lower, alpha):
    cross = covariance(x, candidates, theta)
    projected = solve_triangular(lower, cross, lower=True, check_finite=False)
    variance = np.maximum(0, np.exp(theta[0]) - np.einsum("ij,ij->j", projected, projected))
    return cross.T @ alpha, np.sqrt(variance)

In [ ]:
%%writefile -a master_code/stage_c_dynamic.py

def disk_constraint(point):
    return 1 - point @ point


def disk_jacobian(point):
    return -2 * point


def refined_candidate(x, pool, theta, lower, alpha, pool_lcb):
    """Refine two promising pool candidates with the same fitted LCB."""
    precision = np.broadcast_to(np.exp(-2 * theta[1:]), (2,))
    amplitude = np.exp(theta[0])

    def lcb(p):
        delta = x - p
        kernel = amplitude * np.exp(-.5 * ((delta ** 2) @ precision))
        derivative = kernel[:, None] * delta * precision
        projected = solve_triangular(lower, kernel, lower=True, check_finite=False)
        weights = solve_triangular(lower.T, projected, lower=False, check_finite=False)
        # A tiny variance floor avoids an undefined derivative at zero.
        variance = amplitude - projected @ projected
        sd = np.sqrt(max(variance, 1e-20))
        mean_gradient = derivative.T @ alpha
        gradient = mean_gradient + 2 * (derivative.T @ weights) / sd if variance > 1e-20 else mean_gradient
        return kernel @ alpha - 2 * sd, gradient

    best = pool[int(np.argmin(pool_lcb))]
    best_lcb = float(np.min(pool_lcb))
    starts = [best]
    distant = np.linalg.norm(pool - best, axis=1) >= .05
    if distant.any():
        starts.append(pool[np.argmin(np.where(distant, pool_lcb, np.inf))])
    warnings, improved = 0, False
    for start in starts:
        result = minimize(lcb, start, jac=True, method="SLSQP",
                          constraints=[{"type": "ineq", "fun": disk_constraint,
                                        "jac": disk_jacobian}],
                          options={"maxiter": 25, "ftol": 1e-10})
        warnings += int(not result.success)
        if not np.all(np.isfinite(result.x)) or np.linalg.norm(result.x) > 1 + 1e-8:
            continue
        candidate = result.x / max(1., np.linalg.norm(result.x))
        # Compare with the stable triangular-factor prediction used for the pool.
        mean, sd = predict(x, candidate[None, :], theta, lower, alpha)
        score = float((mean - 2 * sd)[0])
        if score < best_lcb - 1e-12:
            best, best_lcb, improved = candidate, score, True
    return best, warnings, int(improved)

### Shared search machinery

The GP, LCB refinement and CMA-ES updates are inherited from Stage C.
Common random draws pair new observations by proposal index and repeat
observations by repeat index, while each policy uses independent
observation errors within its own run. The original baseline keeps the
Stage C scheduling and recommendation rules.

In [ ]:
%%writefile -a master_code/stage_c_dynamic.py

def run_coupled_cma(evaluate, q, seed, budget=BUDGET, can_start=None,
                    evaluate_population=None):
    """Use C^(1/2) Q z for matched draws, retaining the reference tell update."""
    state = CMAES([0., 0.], .4, popsize=POPULATION)
    rng = np.random.default_rng(seed)
    points, scores, states = [], [], []
    rejected = 0
    while len(points) + POPULATION <= budget and (can_start is None or can_start()):
        state.C.update_eigensystem(state.counteval, state.params.lazy_gap_evals)
        basis = np.asarray(state.C.eigenbasis)
        root = (basis * np.sqrt(state.C.eigenvalues)) @ basis.T
        population = []
        while len(population) < POPULATION:
            point = np.asarray(state.xmean) + state.sigma * (root @ (q @ rng.standard_normal(2)))
            if np.linalg.norm(point) > 1:
                rejected += 1
                if rejected > 10000:
                    raise RuntimeError("CMA rejection cap")
                continue
            population.append(point.tolist())
        values = ([evaluate(p) for p in population] if evaluate_population is None
                  else evaluate_population(population))
        state.tell(population, values)
        points.extend(population)
        scores.extend(values)
        covariance = np.array([[state.C[i][j] for j in range(2)] for i in range(2)])
        eigvals, vectors = np.linalg.eigh(covariance)
        major_axis = vectors[:, -1]
        states.append(dict(call=state.counteval, ratio=float(eigvals[-1]/eigvals[0]),
                           axis_error=float(np.degrees(np.arccos(np.clip(abs(major_axis @ q[:, 0]), 0, 1)))),
                           sigma=state.sigma, covariance=covariance.tolist(), mean=state.xmean))
    assert state.counteval == len(points)
    if can_start is None:
        assert state.counteval == budget
    return points, scores, dict(states=states, rejections=rejected)

In [ ]:
%%writefile -a master_code/stage_c_dynamic.py

class Evaluator:
    def __init__(self, valley, sigma, search_noise, repeat_noise):
        self.valley, self.sigma = valley, sigma
        self.search_noise, self.repeat_noise = search_noise, repeat_noise
        self.search_count = self.repeat_count = 0
        self.trace = []

    def measure(self, point, repeat=False):
        if repeat:
            noise = self.repeat_noise[self.repeat_count]
            self.repeat_count += 1
        else:
            noise = self.search_noise[self.search_count]
            self.search_count += 1
        value = float(self.valley.clean(point) + self.sigma * noise)
        self.trace.append(dict(point=np.asarray(point).tolist(), observed=value, repeat=repeat))
        return value


def posterior_scores(points, values, theta, noise_sd, candidates):
    x = np.asarray(points)
    _, lower, alpha, _ = fit(x, np.asarray(values), theta, False, noise_sd=noise_sd)
    means, _ = predict(x, np.asarray(candidates), theta, lower, alpha)
    return means


def bo_search(evaluator, q, initial, seed, isotropic, late):
    rng = np.random.default_rng(seed)
    points = list(initial)
    training_x = list(initial)
    training_y = [evaluator.measure(p) for p in initial]
    theta = np.log([1, .4] if isotropic else [1, .4, .4])
    checkpoints = {}
    limit = SEARCH_PREFIX if late else BUDGET
    while len(points) < limit:
        x = np.asarray(training_x)
        theta, lower, alpha, _ = fit(
            x, np.asarray(training_y), theta, len(x) % 6 == 0, noise_sd=evaluator.sigma
        )
        pool = disk(rng, POOL) @ q.T
        mean, sd = predict(x, pool, theta, lower, alpha)
        candidate, _, _ = refined_candidate(x, pool, theta, lower, alpha, mean - 2 * sd)
        points.append(candidate)
        training_x.append(candidate)
        training_y.append(evaluator.measure(candidate))
        if late and len(points) > REPEAT_START:
            training_x.append(candidate)
            training_y.append(evaluator.measure(candidate, repeat=True))
        if len(points) in (SEARCH_PREFIX, limit):
            scores = posterior_scores(training_x, training_y, theta, evaluator.sigma, points)
            checkpoints[len(points)] = dict(scores=scores, theta=theta.copy())
    return dict(points=np.asarray(points), checkpoints=checkpoints, evaluator=evaluator)


def cma_search(evaluator, q, seed, late):
    def evaluate_candidate(point):
        y = evaluator.measure(point)
        if late and evaluator.search_count > REPEAT_START:
            y = (y + evaluator.measure(point, repeat=True)) / 2
        return y

    limit = SEARCH_PREFIX if late else BUDGET
    points, scores, _ = run_coupled_cma(evaluate_candidate, q, seed, budget=limit)
    checkpoints = {n: {"scores": np.asarray(scores[:n])} for n in (SEARCH_PREFIX, limit)}
    return dict(points=np.asarray(points), checkpoints=checkpoints, evaluator=evaluator,
                update_scores=np.asarray(scores))


def distinct_top(points, scores):
    indices, seen = [], set()
    for index in np.argsort(scores, kind="stable"):
        key = tuple(points[index])
        if key not in seen:
            indices.append(int(index))
            seen.add(key)
        if len(indices) == FINALISTS:
            return np.asarray(indices)
    raise ValueError("Fewer distinct visited locations than finalists.")


def verify_finalists(baseline, evaluator, method, indices=None):
    points = baseline["points"][:SEARCH_PREFIX]
    state = baseline["checkpoints"][SEARCH_PREFIX]
    if indices is None:
        indices = distinct_top(points, state["scores"])
    finalists = points[indices]
    fresh = np.array([[evaluator.measure(p, repeat=True) for _ in range(PER_FINALIST)]
                      for p in finalists])
    chosen = int(np.argmin(fresh.mean(axis=1)))
    result = dict(indices=indices.tolist(), chosen_index=int(indices[chosen]))
    if method != "CMA-ES":
        # The same observations support this secondary recommendation.
        train_x = np.vstack([points, np.repeat(finalists, PER_FINALIST, axis=0)])
        search_y = [row["observed"] for row in baseline["evaluator"].trace[:SEARCH_PREFIX]]
        train_y = np.concatenate([search_y, fresh.ravel()])
        gp_scores = posterior_scores(train_x, train_y, state["theta"],
                                     evaluator.sigma, finalists)
        result["gp_chosen_index"] = int(indices[np.argmin(gp_scores)])
    return result

### Dynamic CMA-ES

The trigger uses only first measurements from the current generation.
Repeat counts before updates and terminal leftover calls are recorded
separately. Candidate ranking errors are assessed later using the true
objective and the actual scores supplied to CMA-ES.

In [ ]:
%%writefile -a master_code/stage_c_dynamic.py

def cutoff_cma(meter, q, seed):
    archived_sums, archived_counts = [], []

    def remaining():
        return BUDGET - meter.search_count - meter.repeat_count

    def score_population(population):
        sums = np.array([meter.measure(p) for p in population])
        counts = np.ones(POPULATION, dtype=int)
        if (meter.search_count > REPEAT_START and remaining() >= 2
                and meter.repeat_count + 2 <= MAX_CUTOFF_REPEATS):
            order = np.argsort(sums, kind="stable")
            boundary = order[POPULATION // 2 - 1:POPULATION // 2 + 1]
            if sums[boundary[1]] - sums[boundary[0]] <= np.sqrt(2) * meter.sigma:
                for idx in boundary:
                    sums[idx] += meter.measure(population[idx], repeat=True)
                    counts[idx] += 1
        archived_sums.extend(sums)
        archived_counts.extend(counts)
        return sums / counts

    points, update_scores, _ = run_coupled_cma(
        meter.measure, q, seed, can_start=lambda: remaining() >= POPULATION,
        evaluate_population=score_population,
    )
    assert len(points) >= 84
    update_repeats = meter.repeat_count
    top_two = np.argsort(np.asarray(archived_sums) / archived_counts, kind="stable")[:2]
    for i in range(remaining()):
        idx = top_two[i % len(top_two)]
        archived_sums[idx] += meter.measure(points[idx], repeat=True)
        archived_counts[idx] += 1
    scores = np.asarray(archived_sums) / archived_counts
    return dict(points=np.asarray(points), scores=scores, evaluator=meter,
                update_scores=np.asarray(update_scores), update_repeats=update_repeats)

### BO with dynamic information allocation

Candidate measurements include all distinct visited points and up to 24
new points. The new set contains the refined LCB proposal and the strongest
pool candidates under LCB. For each possible measurement, Gaussian
conditioning gives a distribution of updated posterior means. We integrate
the best updated mean with 24-point Gauss--Hermite quadrature. A proposed new
location becomes eligible for the recommendation only in the scenario
where it is evaluated. This is a finite-candidate approximation to the
visited-set knowledge gradient. Its comparison with refined LCB evaluates
the entire decision rule, including the restricted new-candidate set.

In [ ]:
%%writefile -a master_code/stage_c_dynamic.py

def distinct_indices(points):
    return np.sort(np.unique(points, axis=0, return_index=True)[1])


def posterior_joint(x, candidates, theta, lower, alpha):
    cross = covariance(x, candidates, theta)
    solved = solve_triangular(lower, cross, lower=True, check_finite=False)
    cov = covariance(candidates, candidates, theta) - solved.T @ solved
    return cross.T @ alpha, (cov + cov.T) / 2


def knowledge_gradient_choice(x, y, theta, lower, alpha, old, new, noise_sd):
    candidates = np.vstack([old, new])
    means, cov = posterior_joint(x, candidates, theta, lower, alpha)
    noise_standardised = noise_sd / (np.std(y) or 1)
    observation_sd = np.sqrt(np.maximum(np.diag(cov), 0) + noise_standardised**2)
    slope = cov / observation_sd[None, :]
    n_old = len(old)
    fantasies = means[:n_old, None, None] + slope[:n_old, :, None] * KG_NODES
    best = fantasies.min(axis=0)
    if len(new):
        new_fantasy = means[n_old:, None] + np.diag(slope)[n_old:, None] * KG_NODES
        best[n_old:] = np.minimum(best[n_old:], new_fantasy)
    return int(np.argmin(best @ KG_WEIGHTS))


def kg_bo(baseline, meter, q, seed):
    prefix = baseline["points"][:SEARCH_PREFIX]
    meter.trace = list(baseline["evaluator"].trace[:SEARCH_PREFIX])
    meter.search_count = SEARCH_PREFIX
    theta = baseline["checkpoints"][SEARCH_PREFIX]["theta"].copy()
    rng = np.random.default_rng(seed)
    for _ in range(POPULATION, SEARCH_PREFIX):
        disk(rng, POOL)
    while meter.search_count + meter.repeat_count < BUDGET:
        x = np.asarray([r["point"] for r in meter.trace])
        y = np.asarray([r["observed"] for r in meter.trace])
        theta, lower, alpha, _ = fit(x, y, theta, len(y) % 6 == 0, noise_sd=meter.sigma)
        old = x[distinct_indices(x)]
        pool = disk(rng, POOL) @ q.T
        mu, sd = predict(x, pool, theta, lower, alpha)
        lcb = mu - 2 * sd
        refined, _, _ = refined_candidate(x, pool, theta, lower, alpha, lcb)
        seen = {tuple(p) for p in old}
        new = []
        for point in (refined, *pool[np.argsort(lcb, kind="stable")[:KG_NEW_POINTS]]):
            if tuple(point) not in seen and len(new) < KG_NEW_POINTS:
                new.append(point)
                seen.add(tuple(point))
        new = np.asarray(new).reshape(-1, old.shape[1])
        chosen = knowledge_gradient_choice(x, y, theta, lower, alpha, old, new, meter.sigma)
        if chosen < len(old):
            meter.measure(old[chosen], repeat=True)
        else:
            meter.measure(new[chosen - len(old)])
    x = np.asarray([r["point"] for r in meter.trace])
    y = np.asarray([r["observed"] for r in meter.trace])
    visited = x[distinct_indices(x)]
    scores = posterior_scores(x, y, theta, meter.sigma, visited)
    np.testing.assert_array_equal(x[:SEARCH_PREFIX], prefix)
    return dict(points=visited, scores=scores, evaluator=meter)


def coverage_shortlist(baseline, sigma, seed):
    x = baseline["points"][:SEARCH_PREFIX]
    y = np.array([r["observed"] for r in baseline["evaluator"].trace[:SEARCH_PREFIX]])
    theta = baseline["checkpoints"][SEARCH_PREFIX]["theta"]
    indices = distinct_indices(x)
    _, lower, alpha, _ = fit(x, y, theta, False, noise_sd=sigma)
    mu, cov = posterior_joint(x, x[indices], theta, lower, alpha)
    eigenvalues, eigenvectors = np.linalg.eigh(cov)
    rng = np.random.default_rng(seed)
    draws = mu[:, None] + (eigenvectors * np.sqrt(np.maximum(eigenvalues, 0))) @ (
        rng.standard_normal((len(indices), SHORTLIST_DRAWS)))
    epsilon = SHORTLIST_TOLERANCE * sigma / (y.std() or 1)
    near_best = draws <= draws.min(axis=0) + epsilon
    uncovered = np.ones(SHORTLIST_DRAWS, dtype=bool)
    chosen = []
    for _ in range(FINALISTS):
        gain = near_best[:, uncovered].sum(axis=1)
        gain[chosen] = -1
        tied = np.flatnonzero(gain == gain.max())
        idx = int(tied[np.argmin(mu[tied])])
        chosen.append(idx)
        uncovered &= ~near_best[idx]
    return indices[chosen]

### Run paired experiments

Store every measurement and count its budget use. The best visited point,
recommendation gap, and shortlist and parent diagnostics are scored after
each recommendation has been made. The same 72-call history supplies both
shortlist policies. A seed block can be resumed from disk.

In [ ]:
%%writefile -a master_code/stage_c_dynamic.py

def assess(clean, chosen, **labels):
    best = float(np.min(clean))
    recommendation = float(clean[chosen])
    return dict(**labels, recommendation=recommendation, best_visited=best,
                selection_loss=recommendation - best, chosen_index=int(chosen))


def parent_rows(run, valley, **labels):
    clean = np.array([valley.clean(p) for p in run["points"]])
    errors = []
    mu = POPULATION // 2
    for start in range(0, len(clean), POPULATION):
        end = start + POPULATION
        clean_top = set(np.argsort(clean[start:end], kind="stable")[:mu])
        selected = set(np.argsort(run["update_scores"][start:end], kind="stable")[:mu])
        errors.append(dict(**labels, generation=end // POPULATION,
                           parent_error=len(selected - clean_top) / mu))
    return errors

In [ ]:
%%writefile -a master_code/stage_c_dynamic.py

def run_block(seed):
    centre = rotate((.4, 0), random.Random(seed).uniform(0, tau))
    search_noise = np.random.default_rng(
        np.random.SeedSequence([seed, 40117])).standard_normal(BUDGET)
    repeat_noise = np.random.default_rng(
        np.random.SeedSequence([seed, 73093])).standard_normal(BUDGET)
    results, checks, parents, histories = [], [], [], []

    def meter_for(valley, sigma):
        return Evaluator(valley, sigma, search_noise, repeat_noise)

    def record_search(run, valley, labels, policy, scores):
        meter = run["evaluator"]
        assert meter.search_count + meter.repeat_count == BUDGET
        clean = np.array([valley.clean(p) for p in run["points"]])
        repeats_before_update = run.get("update_repeats", meter.repeat_count)
        results.append(assess(clean, np.argmin(scores), **labels, policy=policy,
                              search_calls=meter.search_count, repeat_calls=meter.repeat_count,
                              repeats_before_update=repeats_before_update))
        histories.append(dict(**labels, policy=policy, trace=meter.trace))
        if labels["method"] == "CMA-ES":
            parents.extend(parent_rows(run, valley, **labels, policy=policy))

    for bend in BENDS:
        for sigma in NOISE:
            for angle in ANGLES:
                q = rotation(angle)
                valley = Valley(centre, math.radians(angle), 100, bend)
                initial = None
                for method in METHODS:
                    labels = dict(seed=seed, bend=bend, sigma=sigma, angle=angle, method=method)
                    is_cma = method == "CMA-ES"
                    runs = {}
                    for policy in (BASELINE, LATE):
                        meter = meter_for(valley, sigma)
                        late = policy == LATE
                        if is_cma:
                            run = cma_search(meter, q, seed + 30000, late)
                            initial = run["points"][:POPULATION]
                        else:
                            run = bo_search(meter, q, initial, seed + 20000,
                                            method == "Isotropic BO", late)
                        limit = SEARCH_PREFIX if late else BUDGET
                        assert (meter.search_count, meter.repeat_count) == (limit, BUDGET-limit)
                        record_search(run, valley, labels, policy,
                                      run["checkpoints"][limit]["scores"])
                        runs[policy] = run
                    baseline, late = runs[BASELINE], runs[LATE]
                    prefix = baseline["evaluator"].trace
                    assert prefix[:REPEAT_START] == late["evaluator"].trace[:REPEAT_START]

                    meter = meter_for(valley, sigma)
                    if is_cma:
                        dynamic = cutoff_cma(meter, q, seed + 30000)
                        policy = CUTOFF
                        assert prefix[:REPEAT_START] == meter.trace[:REPEAT_START]
                    else:
                        dynamic = kg_bo(baseline, meter, q, seed + 20000)
                        policy = KG
                        assert prefix[:SEARCH_PREFIX] == meter.trace[:SEARCH_PREFIX]
                    record_search(dynamic, valley, labels, policy, dynamic["scores"])

                    points = baseline["points"][:SEARCH_PREFIX]
                    clean = np.array([valley.clean(p) for p in points])
                    prefix_best = np.argmin(baseline["checkpoints"][SEARCH_PREFIX]["scores"])
                    for policy in ((VERIFY,) if is_cma else (VERIFY, COVERAGE)):
                        indices = None
                        if policy == COVERAGE:
                            indices = coverage_shortlist(baseline, sigma, seed + 50000)
                        meter = meter_for(valley, sigma)
                        verified = verify_finalists(baseline, meter, method, indices=indices)
                        assert meter.repeat_count == BUDGET - SEARCH_PREFIX
                        row = assess(clean, verified["chosen_index"], **labels, policy=policy,
                                     search_calls=SEARCH_PREFIX, repeat_calls=meter.repeat_count,
                                     repeats_before_update=0)
                        results.append(row)
                        best_finalist = float(np.min(clean[verified["indices"]]))
                        checks.append(dict(**labels, policy=policy,
                            shortlist_loss=best_finalist - row["best_visited"],
                            final_ranking_loss=row["recommendation"] - best_finalist,
                            prefix_recommendation=float(clean[prefix_best])))
                        histories.append(dict(**labels, policy=policy, **verified,
                                              trace=prefix[:SEARCH_PREFIX] + meter.trace))
    return dict(results=results, verification=checks, parents=parents, histories=histories)

In [ ]:
%%writefile -a master_code/stage_c_dynamic.py

def block_path(seed):
    return OUT / "blocks" / f"{seed}.json.gz"


def save_blocks():
    (OUT / "blocks").mkdir(parents=True, exist_ok=True)
    manifest = OUT / "settings.json"
    if manifest.exists():
        if json.loads(manifest.read_text(encoding="utf-8")) != SETTINGS:
            raise ValueError("Cached settings or software differ. Choose a new OUTPUT_ROOT.")
    else:
        manifest.write_text(json.dumps(SETTINGS, indent=2), encoding="utf-8")
    for i, seed in enumerate(SEEDS, 1):
        path = block_path(seed)
        if path.exists():
            print(f"[{i}/{len(SEEDS)}] Reusing block {seed}", flush=True)
            continue
        started = perf_counter()
        block = run_block(seed)
        temp = path.with_suffix(".tmp")
        with gzip.open(temp, "wt", encoding="utf-8") as file:
            json.dump(block, file, allow_nan=False, separators=(",", ":"))
        temp.replace(path)
        print(f"[{i}/{len(SEEDS)}] Saved block {seed} ({perf_counter()-started:.1f} s)",
              flush=True)

In [ ]:
%%writefile -a master_code/stage_c_dynamic.py

save_blocks()

### Read the results

Negative values of the mean paired $\log_{10}$ gap ratio favour the new
policy. The 95% intervals resample complete seeds and are pointwise.
Compare all alternatives with search only and then use the direct
policy contrasts to distinguish the effect of adaptive allocation from
the effect of merely repeating points. Shortlist coverage is assessed
before fresh measurements. Keep the results for each kernel separate.

In [ ]:
%%writefile -a master_code/stage_c_dynamic.py

records, checks, rank_data = [], [], []
for seed in SEEDS:
    with gzip.open(block_path(seed), "rt", encoding="utf-8") as file:
        block = json.load(file)
    records.extend(block["results"])
    checks.extend(block["verification"])
    rank_data.extend(block["parents"])
results = pd.DataFrame(records)
verification = pd.DataFrame(checks)
parents = pd.DataFrame(rank_data)
group_keys = ["bend", "sigma", "angle", "method"]
metrics = ["recommendation", "best_visited", "selection_loss"]
expected = len(SEEDS)*len(BENDS)*len(NOISE)*len(ANGLES)*sum(map(len, METHOD_POLICIES.values()))
assert len(results) == expected
assert not results.duplicated(["seed", *group_keys, "policy"]).any()
assert (results.search_calls + results.repeat_calls == BUDGET).all()
matched = verification.merge(results, on=["seed", *group_keys, "policy"], validate="one_to_one")
np.testing.assert_allclose(matched.selection_loss, matched.shortlist_loss + matched.final_ranking_loss)
results.to_csv(OUT / "run_gaps.csv", index=False)
verification.to_csv(OUT / "verification_diagnostics.csv", index=False)
parents.to_csv(OUT / "cma_parent_errors.csv", index=False)

def primary(table):
    return table[(table.bend == .75) & (table.sigma == 1e-2) & (table.angle == 45)]


seed_rng = np.random.default_rng(4670)
draws = (seed_rng.integers(0, len(SEEDS), (10000, len(SEEDS)))
         if len(SEEDS) > 1 else None)

def paired_row(delta, **labels):
    bounds = (np.quantile(delta[draws].mean(axis=1), [.025, .975])
              if draws is not None else [np.nan, np.nan])
    return dict(**labels, mean=delta.mean(), low_95=bounds[0], high_95=bounds[1],
                fraction_improved=np.mean(delta < 0), n=len(delta))


paired = []
for key, group in results.groupby(group_keys):
    labels = dict(zip(group_keys, key))
    method = labels["method"]
    by_policy = {p: t.set_index("seed").loc[SEEDS, metrics].to_numpy()
                 for p, t in group.groupby("policy")}
    pairs = [(p, BASELINE) for p in METHOD_POLICIES[method] if p != BASELINE]
    pairs += [(CUTOFF, LATE)] if method == "CMA-ES" else [
        (KG, LATE), (KG, VERIFY), (COVERAGE, VERIFY)]
    for policy, reference in pairs:
        a, b = by_policy[policy], by_policy[reference]
        delta = a - b
        np.testing.assert_allclose(delta[:, 0], delta[:, 1] + delta[:, 2], atol=1e-14)
        for j, metric in enumerate(metrics):
            paired.append(paired_row(delta[:, j], **labels, policy=policy,
                                     reference=reference, metric=metric, scale="raw"))
        log_ratio = np.log10(np.maximum(a[:, 0], LOG_FLOOR)) - np.log10(
            np.maximum(b[:, 0], LOG_FLOOR))
        paired.append(paired_row(log_ratio, **labels, policy=policy, reference=reference,
                                 metric="recommendation", scale="log10"))
effects = pd.DataFrame(paired)
effects.to_csv(OUT / "paired_policy_effects.csv", index=False)

coverage_effects = []
for key, group in verification[verification.method != "CMA-ES"].groupby(group_keys):
    scores = group.pivot(index="seed", columns="policy", values="shortlist_loss").loc[SEEDS]
    coverage_effects.append(paired_row((scores[COVERAGE] - scores[VERIFY]).to_numpy(),
                                       **dict(zip(group_keys, key))))
coverage_effects = pd.DataFrame(coverage_effects)
coverage_effects.to_csv(OUT / "paired_shortlist_effects.csv", index=False)

medians = results.groupby([*group_keys, "policy"], as_index=False)[
    metrics + ["search_calls", "repeat_calls", "repeats_before_update"]].median()
medians.to_csv(OUT / "median_gaps_and_allocation.csv", index=False)
display(primary(medians).drop(columns=["bend", "sigma", "angle"]).style.format(
    {m: "{:.3e}" for m in metrics}))
selected = primary(effects)
selected = selected[selected.scale == "log10"]
display(selected[["method", "policy", "reference", "mean", "low_95", "high_95",
                  "fraction_improved"]].round(3))
display(primary(coverage_effects)[["method", "mean", "low_95", "high_95"]].style.format(
    {name: "{:.3e}" for name in ("mean", "low_95", "high_95")}))

In [ ]:
%%writefile -a master_code/stage_c_dynamic.py

focus = selected[(selected.reference == BASELINE) & selected.policy.isin((CUTOFF, KG, COVERAGE))]
fig, ax = plt.subplots(figsize=(8, 4.8), layout="constrained")
for index, row in enumerate(focus.itertuples(index=False)):
    color = "#b45309" if row.method == "CMA-ES" else "#0072b2"
    if np.isfinite(row.low_95):
        ax.hlines(index, row.low_95, row.high_95, color=color)
    ax.scatter(row.mean, index, color=color)
ax.set_yticks(range(len(focus)), [f"{r.method} · {r.policy}" for r in focus.itertuples()])
ax.axvline(0, color=".5", ls="--")
ax.invert_yaxis()
ax.set(title="Adaptive policies versus search only at 96 calls",
       xlabel="Mean paired log10 ratio of true recommendation gaps")
fig.savefig(OUT / "dynamic-recommendation-effects.png", dpi=220)
plt.show()
plt.close(fig)

rank_summary = (parents.groupby(["bend", "sigma", "angle", "policy", "generation"],
                                as_index=False).parent_error.mean())
rank_summary.to_csv(OUT / "cma_parent_summary.csv", index=False)
fig, ax = plt.subplots(figsize=(7.2, 4), layout="constrained")
for policy, color in ((BASELINE, ".45"), (LATE, "#0072b2"), (CUTOFF, "#b45309")):
    panel = primary(rank_summary[rank_summary.policy == policy]).sort_values("generation")
    ax.plot(panel.generation, panel.parent_error, marker="o", ms=3, color=color, label=policy)
ax.axvline(8.5, color=".6", ls=":")
ax.set(ylim=(0, 1), title="CMA-ES parent ranking errors",
       xlabel="Generation", ylabel="Mean parent replacement rate")
ax.legend( ncol=3, frameon=False)
fig.savefig(OUT / "dynamic-cma-parent-errors.png", dpi=220)
plt.show()
plt.close(fig)

The recommendation contrast determines whether dynamic allocation paid off
at a fixed budget. Use the allocation counts to see how often it chose a
repeat. Use the best visited and selection gaps to distinguish better
search from better recommendation. A lower parent replacement rate alone
does not establish an advantage, because strategies complete different
numbers of generations. Compare common generations to study the rankings.

In [ ]:
run_stage("stage_c_dynamic")

## Stage C - recommendation and switch timing

Source reused from `stage_c_final_checks.ipynb`. The configuration above controls execution. Code cells in this section prepare the stage script; the final cell runs it when selected.

In [ ]:
%%writefile master_code/stage_c_final.py
from pathlib import Path
import os

MASTER_PROFILE = globals().get("MASTER_PROFILE", os.environ.get("OPT_PROFILE", "smoke"))
MASTER_ROOT = Path(globals().get("MASTER_ROOT", os.environ.get("OPT_RESULTS_ROOT", "optimisation_results"))).resolve()
MASTER_INCLUDE_TRANSFER = globals().get("MASTER_INCLUDE_TRANSFER", os.environ.get("OPT_INCLUDE_TRANSFER", "1") == "1")
if MASTER_PROFILE not in ("full", "smoke"):
    raise ValueError("Choose full or smoke.")

In [ ]:
%%writefile -a master_code/stage_c_final.py

import gzip
import json
import math
import os
import random
import runpy
import sys
from dataclasses import dataclass
from importlib.metadata import distribution, version
from math import cos, sin, hypot, tau
from pathlib import Path
from time import perf_counter

for variable in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS"):
    os.environ[variable] = "1"

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.linalg import cholesky, cho_solve, solve_triangular
from scipy.optimize import minimize
from scipy.spatial.distance import cdist

pure_cma = distribution("cma").locate_file("cma/more_algorithms/purecma.py")
CMAES = runpy.run_path(str(pure_cma))["CMAES"]

In [ ]:
%%writefile -a master_code/stage_c_final.py

PROFILE = MASTER_PROFILE
OUTPUT_ROOT = MASTER_ROOT / "stage_c_final"
PRIOR = MASTER_ROOT / "stage_c_dynamic" / PROFILE
BUDGET, POPULATION, POOL = 96, 6, 1024
REPEAT_START, SEARCH_PREFIX, EARLY_PREFIX = 48, 72, 60
MAX_CUTOFF_REPEATS, KG_NEW_POINTS = 12, 24
JITTER, LOG_FLOOR = 1e-7, 1e-14
KG_NODES, KG_WEIGHTS = np.polynomial.hermite.hermgauss(24)
KG_NODES *= np.sqrt(2)
KG_WEIGHTS /= np.sqrt(np.pi)

if PROFILE == "full":
    SEEDS = list(range(9900, 9924))
    BENDS, ANGLES, NOISE = (0., .75), (0, 45), (1e-3, 1e-2)
elif PROFILE == "smoke":
    SEEDS = [9900]
    BENDS, ANGLES, NOISE = (.75,), (45,), (1e-2,)
else:
    raise ValueError("Use 'smoke' or 'full'.")
OUT = OUTPUT_ROOT / PROFILE
OUT.mkdir(parents=True, exist_ok=True)
SOFTWARE = {"python": sys.version.split()[0], **{
    pkg: version(pkg) for pkg in ("numpy", "scipy", "pandas", "matplotlib", "cma")
}}
SETTINGS = dict(schema=1, seeds=SEEDS, bends=list(BENDS), angles=list(ANGLES), noise=list(NOISE),
                budget=BUDGET, population=POPULATION, pool=POOL, repeat_start=REPEAT_START,
                cutoff_cap=MAX_CUTOFF_REPEATS, kg_points=KG_NEW_POINTS,
                prefixes=[EARLY_PREFIX, SEARCH_PREFIX], jitter=JITTER, software=SOFTWARE)
print(f"{PROFILE}: {len(SEEDS)} seed blocks under {OUT}")
print(SOFTWARE)

### Audit the earlier dynamic results

This optional cell reads the original paired comparisons for the straight
valley and moderate noise. Those files live in the previous Colab
runtime or wherever you saved them. The new experiment runs independently
of these files.

In [ ]:
%%writefile -a master_code/stage_c_final.py

prior_csv = PRIOR / "paired_policy_effects.csv"
if prior_csv.exists():
    old = pd.read_csv(prior_csv)
    sensitivity = old[(old.scale == "log10") & (old.reference == "Search only")
                      & old.policy.isin(["Cutoff repeats", "Knowledge gradient"])
                      & ((old.bend == 0) | (old.sigma == 1e-3))]
    sensitivity.to_csv(OUT / "earlier_sensitivity.csv", index=False)
    display(sensitivity[["method", "policy", "bend", "angle", "sigma",
                         "mean", "low_95", "high_95"]].round(4))
else:
    print(f"Earlier sensitivity CSV unavailable at {prior_csv}.")

### Search and recommendation functions

The objective, search updates, noise pairing, GP and KG approximation
come from the previous notebook. The BO baseline stores the GP state
after calls 60 and 72. Each KG branch reuses that state and the exact
evaluation prefix. The CMA-ES GP is fitted once after search, using
observed scores and the observation noise variance. True gaps are
computed after the recommendations have been chosen.

In [ ]:
%%writefile -a master_code/stage_c_final.py

def rotate(point, angle):
    x, y = point
    return cos(angle) * x - sin(angle) * y, sin(angle) * x + cos(angle) * y


@dataclass(frozen=True)
class Valley:
    centre: tuple
    angle: float
    curvature_ratio: float = 100.0
    bend: float = 0.0

    @property
    def normaliser(self):
        # Under uniform disk sampling E[X^2] = 1/4 and E[X^4] = 1/8.
        # Terms odd in either coordinate integrate to zero.
        a, b = self.centre
        if self.bend == 0:
            k = self.curvature_ratio
            return (1 + k) / 4 + a*a + k*b*b
        eu2 = .25 + a*a
        ev2 = .25 + b*b
        eu4 = .125 + 1.5*a*a + a**4
        return eu2 + self.curvature_ratio * (
            ev2 + 2*self.bend*b*eu2 + self.bend**2*eu4
        )

    def canonical_gap(self, x, y):
        a, b = self.centre
        u, v = x - a, y - b
        return (u**2 + self.curvature_ratio*(v - self.bend*u**2)**2) / self.normaliser

    def clean(self, point):
        if hypot(*point) > 1.0 + 1e-12:
            raise ValueError("Point outside the unit disk")
        x, y = rotate(point, -self.angle)
        return self.canonical_gap(x, y)

    def observe(self, point, sigma, rng):
        return self.clean(point) + sigma * rng.gauss(0.0, 1.0)


def rotation(degree):
    angle = math.radians(degree)
    return np.array([rotate((1, 0), angle), rotate((0, 1), angle)]).T


def disk(rng, size):
    u = rng.random((size, 2))
    angle = 2 * np.pi * u[:, 0]
    return np.sqrt(u[:, 1:]) * np.column_stack((np.cos(angle), np.sin(angle)))

In [ ]:
%%writefile -a master_code/stage_c_final.py

def covariance(x, y, theta):
    lengths = np.exp(theta[1:])
    return np.exp(theta[0] - .5 * cdist(x / lengths, y / lengths, "sqeuclidean"))


def fit(x, values, theta, refit, noise_sd=0.0, jitter=JITTER):
    output_scale = values.std() or 1
    y = (values - values.mean()) / output_scale
    ident = np.eye(len(x))

    def factor(parameters):
        signal = covariance(x, x, parameters)
        lower = cholesky(signal + (jitter + (noise_sd/output_scale)**2) * ident, lower=True)
        return signal, lower, cho_solve((lower, True), y)

    def likelihood(parameters):
        signal, lower, alpha = factor(parameters)
        loss = .5 * (y @ alpha) + np.log(np.diag(lower)).sum()
        weights = .5 * (cho_solve((lower, True), ident) - np.outer(alpha, alpha)) * signal
        gradient = [weights.sum()]
        if len(parameters) == 2:
            gradient.append((weights * cdist(x, x, "sqeuclidean")).sum() / np.exp(2*parameters[1]))
        else:
            for axis, log_length in enumerate(parameters[1:]):
                diff = x[:, axis, None] - x[:, axis]
                gradient.append((weights * diff**2).sum() / np.exp(2*log_length))
        return loss, np.array(gradient)

    status = -1
    if refit:
        bounds = [(-6, 6)] + [(math.log(.02), math.log(2))] * (len(theta)-1)
        result = minimize(likelihood, theta, jac=True, bounds=bounds, method="L-BFGS-B",
                          options={"maxiter": 50})
        theta, status = result.x, int(result.status)
    _, lower, alpha = factor(theta)
    return theta, lower, alpha, status


def predict(x, candidates, theta, lower, alpha):
    cross = covariance(x, candidates, theta)
    projected = solve_triangular(lower, cross, lower=True, check_finite=False)
    variance = np.maximum(0, np.exp(theta[0]) - np.einsum("ij,ij->j", projected, projected))
    return cross.T @ alpha, np.sqrt(variance)

In [ ]:
%%writefile -a master_code/stage_c_final.py

def disk_constraint(point):
    return 1 - point @ point


def disk_jacobian(point):
    return -2 * point


def refined_candidate(x, pool, theta, lower, alpha, pool_lcb):
    """Refine two promising pool candidates with the same fitted LCB."""
    precision = np.broadcast_to(np.exp(-2 * theta[1:]), (2,))
    amplitude = np.exp(theta[0])

    def lcb(p):
        delta = x - p
        kernel = amplitude * np.exp(-.5 * ((delta ** 2) @ precision))
        derivative = kernel[:, None] * delta * precision
        projected = solve_triangular(lower, kernel, lower=True, check_finite=False)
        weights = solve_triangular(lower.T, projected, lower=False, check_finite=False)
        # A tiny variance floor avoids an undefined derivative at zero.
        variance = amplitude - projected @ projected
        sd = np.sqrt(max(variance, 1e-20))
        mean_gradient = derivative.T @ alpha
        gradient = mean_gradient + 2 * (derivative.T @ weights) / sd if variance > 1e-20 else mean_gradient
        return kernel @ alpha - 2 * sd, gradient

    best = pool[int(np.argmin(pool_lcb))]
    best_lcb = float(np.min(pool_lcb))
    starts = [best]
    distant = np.linalg.norm(pool - best, axis=1) >= .05
    if distant.any():
        starts.append(pool[np.argmin(np.where(distant, pool_lcb, np.inf))])
    warnings, improved = 0, False
    for start in starts:
        result = minimize(lcb, start, jac=True, method="SLSQP",
                          constraints=[{"type": "ineq", "fun": disk_constraint,
                                        "jac": disk_jacobian}],
                          options={"maxiter": 25, "ftol": 1e-10})
        warnings += int(not result.success)
        if not np.all(np.isfinite(result.x)) or np.linalg.norm(result.x) > 1 + 1e-8:
            continue
        candidate = result.x / max(1., np.linalg.norm(result.x))
        # Compare with the stable triangular-factor prediction used for the pool.
        mean, sd = predict(x, candidate[None, :], theta, lower, alpha)
        score = float((mean - 2 * sd)[0])
        if score < best_lcb - 1e-12:
            best, best_lcb, improved = candidate, score, True
    return best, warnings, int(improved)

In [ ]:
%%writefile -a master_code/stage_c_final.py

def run_coupled_cma(evaluate, q, seed, budget=BUDGET, can_start=None,
                    evaluate_population=None):
    """Use C^(1/2) Q z for matched draws, retaining the reference tell update."""
    state = CMAES([0., 0.], .4, popsize=POPULATION)
    rng = np.random.default_rng(seed)
    points, scores, states = [], [], []
    rejected = 0
    while len(points) + POPULATION <= budget and (can_start is None or can_start()):
        state.C.update_eigensystem(state.counteval, state.params.lazy_gap_evals)
        basis = np.asarray(state.C.eigenbasis)
        root = (basis * np.sqrt(state.C.eigenvalues)) @ basis.T
        population = []
        while len(population) < POPULATION:
            point = np.asarray(state.xmean) + state.sigma * (root @ (q @ rng.standard_normal(2)))
            if np.linalg.norm(point) > 1:
                rejected += 1
                if rejected > 10000:
                    raise RuntimeError("CMA rejection cap")
                continue
            population.append(point.tolist())
        values = ([evaluate(p) for p in population] if evaluate_population is None
                  else evaluate_population(population))
        state.tell(population, values)
        points.extend(population)
        scores.extend(values)
        covariance = np.array([[state.C[i][j] for j in range(2)] for i in range(2)])
        eigvals, vectors = np.linalg.eigh(covariance)
        major_axis = vectors[:, -1]
        states.append(dict(call=state.counteval, ratio=float(eigvals[-1]/eigvals[0]),
                           axis_error=float(np.degrees(np.arccos(np.clip(abs(major_axis @ q[:, 0]), 0, 1)))),
                           sigma=state.sigma, covariance=covariance.tolist(), mean=state.xmean))
    assert state.counteval == len(points)
    if can_start is None:
        assert state.counteval == budget
    return points, scores, dict(states=states, rejections=rejected)

In [ ]:
%%writefile -a master_code/stage_c_final.py

class Evaluator:
    def __init__(self, valley, sigma, search_noise, repeat_noise):
        self.valley, self.sigma = valley, sigma
        self.search_noise, self.repeat_noise = search_noise, repeat_noise
        self.search_count = self.repeat_count = 0
        self.trace = []

    def measure(self, point, repeat=False):
        if repeat:
            noise = self.repeat_noise[self.repeat_count]
            self.repeat_count += 1
        else:
            noise = self.search_noise[self.search_count]
            self.search_count += 1
        value = float(self.valley.clean(point) + self.sigma * noise)
        self.trace.append(dict(point=np.asarray(point).tolist(), observed=value, repeat=repeat))
        return value


def posterior_scores(points, values, theta, noise_sd, candidates):
    x = np.asarray(points)
    _, lower, alpha, _ = fit(x, np.asarray(values), theta, False, noise_sd=noise_sd)
    means, _ = predict(x, np.asarray(candidates), theta, lower, alpha)
    return means


def bo_search(evaluator, q, initial, seed, isotropic, late):
    rng = np.random.default_rng(seed)
    points = list(initial)
    training_x = list(initial)
    training_y = [evaluator.measure(p) for p in initial]
    theta = np.log([1, .4] if isotropic else [1, .4, .4])
    checkpoints = {}
    limit = SEARCH_PREFIX if late else BUDGET
    while len(points) < limit:
        x = np.asarray(training_x)
        theta, lower, alpha, _ = fit(
            x, np.asarray(training_y), theta, len(x) % 6 == 0, noise_sd=evaluator.sigma
        )
        pool = disk(rng, POOL) @ q.T
        mean, sd = predict(x, pool, theta, lower, alpha)
        candidate, _, _ = refined_candidate(x, pool, theta, lower, alpha, mean - 2 * sd)
        points.append(candidate)
        training_x.append(candidate)
        training_y.append(evaluator.measure(candidate))
        if late and len(points) > REPEAT_START:
            training_x.append(candidate)
            training_y.append(evaluator.measure(candidate, repeat=True))
        if len(points) in (EARLY_PREFIX, SEARCH_PREFIX, limit):
            scores = posterior_scores(training_x, training_y, theta, evaluator.sigma, points)
            checkpoints[len(points)] = dict(scores=scores, theta=theta.copy())
    return dict(points=np.asarray(points), checkpoints=checkpoints, evaluator=evaluator)


def cma_search(evaluator, q, seed, late):
    def evaluate_candidate(point):
        y = evaluator.measure(point)
        if late and evaluator.search_count > REPEAT_START:
            y = (y + evaluator.measure(point, repeat=True)) / 2
        return y

    limit = SEARCH_PREFIX if late else BUDGET
    points, scores, _ = run_coupled_cma(evaluate_candidate, q, seed, budget=limit)
    checkpoints = {n: {"scores": np.asarray(scores[:n])} for n in (SEARCH_PREFIX, limit)}
    return dict(points=np.asarray(points), checkpoints=checkpoints, evaluator=evaluator,
                update_scores=np.asarray(scores))

In [ ]:
%%writefile -a master_code/stage_c_final.py

def cutoff_cma(meter, q, seed):
    archived_sums, archived_counts = [], []

    def remaining():
        return BUDGET - meter.search_count - meter.repeat_count

    def score_population(population):
        sums = np.array([meter.measure(p) for p in population])
        counts = np.ones(POPULATION, dtype=int)
        if (meter.search_count > REPEAT_START and remaining() >= 2
                and meter.repeat_count + 2 <= MAX_CUTOFF_REPEATS):
            order = np.argsort(sums, kind="stable")
            boundary = order[POPULATION // 2 - 1:POPULATION // 2 + 1]
            if sums[boundary[1]] - sums[boundary[0]] <= np.sqrt(2) * meter.sigma:
                for idx in boundary:
                    sums[idx] += meter.measure(population[idx], repeat=True)
                    counts[idx] += 1
        archived_sums.extend(sums)
        archived_counts.extend(counts)
        return sums / counts

    points, update_scores, _ = run_coupled_cma(
        meter.measure, q, seed, can_start=lambda: remaining() >= POPULATION,
        evaluate_population=score_population,
    )
    assert len(points) >= 84
    update_repeats = meter.repeat_count
    top_two = np.argsort(np.asarray(archived_sums) / archived_counts, kind="stable")[:2]
    for i in range(remaining()):
        idx = top_two[i % len(top_two)]
        archived_sums[idx] += meter.measure(points[idx], repeat=True)
        archived_counts[idx] += 1
    scores = np.asarray(archived_sums) / archived_counts
    return dict(points=np.asarray(points), scores=scores, evaluator=meter,
                update_scores=np.asarray(update_scores), update_repeats=update_repeats)

In [ ]:
%%writefile -a master_code/stage_c_final.py

def distinct_indices(points):
    return np.sort(np.unique(points, axis=0, return_index=True)[1])


def posterior_joint(x, candidates, theta, lower, alpha):
    cross = covariance(x, candidates, theta)
    solved = solve_triangular(lower, cross, lower=True, check_finite=False)
    cov = covariance(candidates, candidates, theta) - solved.T @ solved
    return cross.T @ alpha, (cov + cov.T) / 2


def knowledge_gradient_choice(x, y, theta, lower, alpha, old, new, noise_sd):
    candidates = np.vstack([old, new])
    means, cov = posterior_joint(x, candidates, theta, lower, alpha)
    noise_standardised = noise_sd / (np.std(y) or 1)
    observation_sd = np.sqrt(np.maximum(np.diag(cov), 0) + noise_standardised**2)
    slope = cov / observation_sd[None, :]
    n_old = len(old)
    fantasies = means[:n_old, None, None] + slope[:n_old, :, None] * KG_NODES
    best = fantasies.min(axis=0)
    if len(new):
        new_fantasy = means[n_old:, None] + np.diag(slope)[n_old:, None] * KG_NODES
        best[n_old:] = np.minimum(best[n_old:], new_fantasy)
    return int(np.argmin(best @ KG_WEIGHTS))


def kg_bo(baseline, meter, q, seed, prefix_calls):
    prefix = baseline["points"][:prefix_calls]
    meter.trace = list(baseline["evaluator"].trace[:prefix_calls])
    meter.search_count = prefix_calls
    theta = baseline["checkpoints"][prefix_calls]["theta"].copy()
    rng = np.random.default_rng(seed)
    for _ in range(POPULATION, prefix_calls):
        disk(rng, POOL)
    while meter.search_count + meter.repeat_count < BUDGET:
        x = np.asarray([r["point"] for r in meter.trace])
        y = np.asarray([r["observed"] for r in meter.trace])
        theta, lower, alpha, _ = fit(x, y, theta, len(y) % 6 == 0, noise_sd=meter.sigma)
        old = x[distinct_indices(x)]
        pool = disk(rng, POOL) @ q.T
        mu, sd = predict(x, pool, theta, lower, alpha)
        lcb = mu - 2 * sd
        refined, _, _ = refined_candidate(x, pool, theta, lower, alpha, lcb)
        seen = {tuple(p) for p in old}
        new = []
        for point in (refined, *pool[np.argsort(lcb, kind="stable")[:KG_NEW_POINTS]]):
            if tuple(point) not in seen and len(new) < KG_NEW_POINTS:
                new.append(point)
                seen.add(tuple(point))
        new = np.asarray(new).reshape(-1, old.shape[1])
        chosen = knowledge_gradient_choice(x, y, theta, lower, alpha, old, new, meter.sigma)
        if chosen < len(old):
            meter.measure(old[chosen], repeat=True)
        else:
            meter.measure(new[chosen - len(old)])
    x = np.asarray([r["point"] for r in meter.trace])
    y = np.asarray([r["observed"] for r in meter.trace])
    visited = x[distinct_indices(x)]
    scores = posterior_scores(x, y, theta, meter.sigma, visited)
    np.testing.assert_array_equal(x[:prefix_calls], prefix)
    return dict(points=visited, scores=scores, evaluator=meter)

In [ ]:
%%writefile -a master_code/stage_c_final.py

CMA, BO = "CMA-ES", "Isotropic BO"
STANDARD, CUTOFF = "Standard", "Cutoff repeats"
NATIVE, GP = "Native", "GP reranking"
LCB, KG60, KG72 = "LCB throughout", "KG after 60", "KG after 72"
METRICS = ("recommendation", "best_visited", "selection_loss")


def score_recommendation(valley, candidates, chosen, **labels):
    clean = np.array([valley.clean(p) for p in candidates])
    recommended, best = float(clean[chosen]), float(clean.min())
    return dict(**labels, recommendation=recommended, best_visited=best,
                selection_loss=recommended-best)


def cma_recommendations(valley, trace, native_index, noise_sd, **labels):
    x = np.asarray([r["point"] for r in trace])
    y = np.asarray([r["observed"] for r in trace])
    proposals = np.asarray([r["point"] for r in trace if not r["repeat"]])
    candidates = x[distinct_indices(x)]
    native_point = proposals[native_index]
    native = int(np.flatnonzero(np.all(candidates == native_point, axis=1))[0])
    theta, lower, alpha, _ = fit(x, y, np.log([1, .4]), True, noise_sd=noise_sd)
    means, _ = predict(x, candidates, theta, lower, alpha)
    return [score_recommendation(valley, candidates, idx, **labels, readout=readout,
                                 search_calls=len(proposals), repeat_calls=len(x)-len(proposals))
            for readout, idx in ((NATIVE, native), (GP, int(np.argmin(means))))]


def run_block(seed):
    centre = rotate((.4, 0), random.Random(seed).uniform(0, tau))
    search_noise = np.random.default_rng(np.random.SeedSequence([seed, 40117])).standard_normal(BUDGET)
    repeat_noise = np.random.default_rng(np.random.SeedSequence([seed, 73093])).standard_normal(BUDGET)
    rows, histories = [], []
    for bend in BENDS:
        for sigma in NOISE:
            for angle in ANGLES:
                q = rotation(angle)
                valley = Valley(centre, math.radians(angle), 100, bend)
                labels = dict(seed=seed, bend=bend, sigma=sigma, angle=angle)

                def meter():
                    return Evaluator(valley, sigma, search_noise, repeat_noise)

                standard = cma_search(meter(), q, seed + 30000, late=False)
                cutoff = cutoff_cma(meter(), q, seed + 30000)
                assert (standard["evaluator"].trace[:REPEAT_START] ==
                        cutoff["evaluator"].trace[:REPEAT_START])
                for policy, run, scores in (
                    (STANDARD, standard, standard["checkpoints"][BUDGET]["scores"]),
                    (CUTOFF, cutoff, cutoff["scores"]),
                ):
                    trace = run["evaluator"].trace
                    native_index = int(np.argmin(scores))
                    assert len(trace) == BUDGET
                    rows.extend(cma_recommendations(valley, trace, native_index, sigma,
                                                    **labels, method=CMA, policy=policy))
                    histories.append(dict(**labels, method=CMA, policy=policy,
                                          chosen_index=native_index, trace=trace))

                initial = standard["points"][:POPULATION]
                baseline = bo_search(meter(), q, initial, seed + 20000,
                                     isotropic=True, late=False)
                early = kg_bo(baseline, meter(), q, seed + 20000, EARLY_PREFIX)
                late = kg_bo(baseline, meter(), q, seed + 20000, SEARCH_PREFIX)
                prefix = baseline["evaluator"].trace
                assert [r["observed"] for r in prefix[:POPULATION]] == [
                    r["observed"] for r in standard["evaluator"].trace[:POPULATION]]
                assert prefix[:EARLY_PREFIX] == early["evaluator"].trace[:EARLY_PREFIX]
                assert prefix[:SEARCH_PREFIX] == late["evaluator"].trace[:SEARCH_PREFIX]
                for policy, run, scores in (
                    (LCB, baseline, baseline["checkpoints"][BUDGET]["scores"]),
                    (KG60, early, early["scores"]),
                    (KG72, late, late["scores"]),
                ):
                    evaluator = run["evaluator"]
                    assert len(evaluator.trace) == BUDGET
                    chosen = int(np.argmin(scores))
                    rows.append(score_recommendation(
                        valley, run["points"], chosen, **labels, method=BO, policy=policy,
                        readout=GP, search_calls=evaluator.search_count,
                        repeat_calls=evaluator.repeat_count,
                    ))
                    histories.append(dict(**labels, method=BO, policy=policy,
                                          trace=evaluator.trace, chosen=run["points"][chosen].tolist()))
    return dict(results=rows, histories=histories)

### Paired comparisons

The primary effect is the mean within-seed $\log_{10}$ ratio of true
recommendation gaps. Negative values favour the variant. We also show
absolute gap differences, the fraction of seeds improved and 95% pointwise
bootstrap intervals over paired seeds. With one smoke seed, intervals are
omitted. Interpret the earlier-history audit and the fresh run separately.

In [ ]:
%%writefile -a master_code/stage_c_final.py

CONDITIONS = ["bend", "sigma", "angle", "method"]
COMPARISONS = {
    CMA: [
        ((STANDARD, GP), (STANDARD, NATIVE), "GP vs. native · standard"),
        ((CUTOFF, GP), (CUTOFF, NATIVE), "GP vs. native · cutoff"),
        ((CUTOFF, GP), (STANDARD, GP), "Cutoff vs. standard · both GP"),
        ((CUTOFF, NATIVE), (STANDARD, NATIVE), "Cutoff vs. standard · both native"),
    ],
    BO: [
        ((KG60, GP), (LCB, GP), "KG after 60 vs. LCB"),
        ((KG72, GP), (LCB, GP), "KG after 72 vs. LCB"),
        ((KG60, GP), (KG72, GP), "KG after 60 vs. after 72"),
    ],
}


def paired_effects(results):
    contrasts = []
    for key, group in results.groupby(CONDITIONS):
        labels = dict(zip(CONDITIONS, key))
        groups = {(p, r): t.set_index("seed").sort_index()
                  for (p, r), t in group.groupby(["policy", "readout"])}
        n = len(next(iter(groups.values())))
        indices = (np.random.default_rng(4670).integers(0, n, (10000, n)) if n > 1 else None)
        for left, right, name in COMPARISONS[labels["method"]]:
            a, b = groups[left], groups[right]
            assert a.index.is_unique and a.index.equals(b.index)
            gaps_a, gaps_b = a[list(METRICS)].to_numpy(), b[list(METRICS)].to_numpy()
            raw = gaps_a - gaps_b
            np.testing.assert_allclose(raw[:, 0], raw[:, 1] + raw[:, 2], atol=1e-14)
            log_ratio = np.log10(np.maximum(gaps_a[:, 0], LOG_FLOOR)) - np.log10(
                np.maximum(gaps_b[:, 0], LOG_FLOOR))
            for metric, scale, delta in [
                *[(name, "raw", raw[:, j]) for j, name in enumerate(METRICS)],
                ("recommendation", "log10", log_ratio),
            ]:
                bounds = (np.quantile(delta[indices].mean(axis=1), [.025, .975])
                          if indices is not None else [np.nan, np.nan])
                contrasts.append(dict(**labels, contrast=name, metric=metric, scale=scale,
                                      n=n, mean=delta.mean(), low_95=bounds[0],
                                      high_95=bounds[1], fraction_improved=np.mean(delta < 0)))
    return pd.DataFrame(contrasts)


def primary(table):
    return table[(table.bend == .75) & (table.sigma == 1e-2) & (table.angle == 45)]

### Optional CMA-ES reranking on the earlier histories

Run this cell if the dynamic Stage C block files are available. It
fits the new recommendation model on those saved observations and
recovers the original recommended point from the stored index.
This is an exploratory readout comparison on previously seen seeds.
It spends zero new objective calls.

In [ ]:
%%writefile -a master_code/stage_c_final.py

def archived_recommendations(files):
    rows = []
    for path in files:
        with gzip.open(path, "rt", encoding="utf-8") as file:
            block = json.load(file)
        def key(r):
            return tuple(r[n] for n in ("seed", "bend", "sigma", "angle", "policy"))

        native = {key(r): r for r in block["results"] if r["method"] == CMA}
        for run in block["histories"]:
            if run["method"] != CMA or run["policy"] not in ("Search only", CUTOFF):
                continue
            seed, bend, sigma, angle, policy = key(run)
            centre = rotate((.4, 0), random.Random(seed).uniform(0, tau))
            valley = Valley(centre, math.radians(angle), 100, bend)
            original = native[key(run)]
            assert len(run["trace"]) == BUDGET
            scored = cma_recommendations(
                valley, run["trace"], original["chosen_index"], sigma,
                seed=seed, bend=bend, sigma=sigma, angle=angle, method=CMA,
                policy=STANDARD if policy == "Search only" else CUTOFF,
            )
            np.testing.assert_allclose(scored[0]["recommendation"], original["recommendation"],
                                       rtol=1e-10, atol=1e-13)
            rows.extend(scored)
    return pd.DataFrame(rows)


prior_blocks = sorted((PRIOR / "blocks").glob("*.json.gz"))
if prior_blocks:
    archived = archived_recommendations(prior_blocks)
    archive_effects = paired_effects(archived)
    archived.to_csv(OUT / "archived_cma_run_gaps.csv", index=False)
    archive_effects.to_csv(OUT / "archived_cma_paired_effects.csv", index=False)
    print(f"Exploratory reranking on {archived.seed.nunique()} archived seed blocks.")
    display(primary(archive_effects).query("scale == 'log10'")[
        ["contrast", "n", "mean", "low_95", "high_95", "fraction_improved"]].round(4))
else:
    print(f"No archived blocks found under {PRIOR / 'blocks'}.")

### Run the fresh comparison and summarise it

CMA-ES readouts share the same measured points within each history, so
only the recommendation and selection loss can change in the readout
contrast. The two KG switches can change both the visited set and the
final choice. Their repeat counts describe how each policy spent its
budget.

In [ ]:
%%writefile -a master_code/stage_c_final.py

def block_path(seed):
    return OUT / "blocks" / f"{seed}.json.gz"


def save_blocks():
    (OUT / "blocks").mkdir(parents=True, exist_ok=True)
    manifest = OUT / "settings.json"
    if manifest.exists():
        if json.loads(manifest.read_text(encoding="utf-8")) != SETTINGS:
            raise ValueError("Cached settings or software differ. Choose a new OUTPUT_ROOT.")
    else:
        manifest.write_text(json.dumps(SETTINGS, indent=2), encoding="utf-8")
    for i, seed in enumerate(SEEDS, 1):
        path = block_path(seed)
        if path.exists():
            print(f"[{i}/{len(SEEDS)}] Reusing block {seed}", flush=True)
            continue
        started = perf_counter()
        block = run_block(seed)
        temp = path.with_suffix(".tmp")
        with gzip.open(temp, "wt", encoding="utf-8") as file:
            json.dump(block, file, allow_nan=False, separators=(",", ":"))
        temp.replace(path)
        print(f"[{i}/{len(SEEDS)}] Saved block {seed} ({perf_counter()-started:.1f} s)",
              flush=True)

In [ ]:
%%writefile -a master_code/stage_c_final.py

save_blocks()

In [ ]:
%%writefile -a master_code/stage_c_final.py

records = []
for seed in SEEDS:
    with gzip.open(block_path(seed), "rt", encoding="utf-8") as file:
        records.extend(json.load(file)["results"])
results = pd.DataFrame(records)
expected = len(SEEDS)*len(BENDS)*len(ANGLES)*len(NOISE)*7
assert len(results) == expected
assert not results.duplicated(["seed", *CONDITIONS, "policy", "readout"]).any()
assert (results.search_calls + results.repeat_calls == BUDGET).all()
same_history = results[results.method == CMA].pivot(
    index=["seed", *CONDITIONS, "policy"], columns="readout", values="best_visited")
np.testing.assert_array_equal(same_history[NATIVE], same_history[GP])
results.to_csv(OUT / "fresh_run_gaps.csv", index=False)
effects = paired_effects(results)
effects.to_csv(OUT / "fresh_paired_effects.csv", index=False)
medians = results.groupby([*CONDITIONS, "policy", "readout"], as_index=False)[
    [*METRICS, "search_calls", "repeat_calls"]].median()
medians.to_csv(OUT / "fresh_medians.csv", index=False)
display(primary(medians)[["method", "policy", "readout", *METRICS, "repeat_calls"]]
        .style.format({m: "{:.3e}" for m in METRICS}))
focus = primary(effects).query("scale == 'log10'").reset_index(drop=True)
display(focus[["method", "contrast", "n", "mean", "low_95", "high_95", "fraction_improved"]]
        .round(4))

In [ ]:
%%writefile -a master_code/stage_c_final.py

fig, ax = plt.subplots(figsize=(8.6, 4.6), layout="constrained")
colors = {CMA: "#b45309", BO: "#0072b2"}
for i, row in enumerate(focus.itertuples(index=False)):
    color = colors[row.method]
    if np.isfinite(row.low_95):
        ax.hlines(i, row.low_95, row.high_95, color=color, lw=2)
    ax.scatter(row.mean, i, color=color, zorder=3)
ax.set_yticks(range(len(focus)), focus.contrast)
ax.axvline(0, color=".5", ls="--")
ax.invert_yaxis()
ax.set(title="Recommendation contrasts at 96 calls",
       xlabel="Mean paired log10 ratio of true recommendation gaps")
fig.savefig(OUT / "final-check-paired-effects.png", dpi=220, bbox_inches="tight")
plt.show()
plt.close(fig)

In [ ]:
run_stage("stage_c_final")

## Supporting transfer - original experiment

Source reused from `noisy_policy_transfer.ipynb`. The configuration above controls execution. Code cells in this section prepare the stage script; the final cell runs it when selected.

In [ ]:
%%writefile master_code/transfer_original.py
from pathlib import Path
import os

MASTER_PROFILE = globals().get("MASTER_PROFILE", os.environ.get("OPT_PROFILE", "smoke"))
MASTER_ROOT = Path(globals().get("MASTER_ROOT", os.environ.get("OPT_RESULTS_ROOT", "optimisation_results"))).resolve()
MASTER_INCLUDE_TRANSFER = globals().get("MASTER_INCLUDE_TRANSFER", os.environ.get("OPT_INCLUDE_TRANSFER", "1") == "1")
if MASTER_PROFILE not in ("full", "smoke"):
    raise ValueError("Choose full or smoke.")

In [ ]:
%%writefile -a master_code/transfer_original.py

import gzip
import json
import math
import os
import random
import runpy
import sys
from dataclasses import dataclass
from importlib.metadata import distribution, version
from math import cos, sin, hypot, tau
from pathlib import Path
from time import perf_counter

for variable in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS"):
    os.environ[variable] = "1"

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.linalg import cholesky, cho_solve, solve_triangular
from scipy.optimize import minimize
from scipy.spatial.distance import cdist

pure_cma = distribution("cma").locate_file("cma/more_algorithms/purecma.py")
CMAES = runpy.run_path(str(pure_cma))["CMAES"]

In [ ]:
%%writefile -a master_code/transfer_original.py

import shutil
import cocoex
from scipy.stats import qmc

PROFILE = MASTER_PROFILE
OUTPUT_ROOT = MASTER_ROOT / "transfer_original"
BUDGET, POPULATION, POOL = 96, 6, 1024
SEARCH_PREFIX, REPEAT_START = 72, 48
MAX_CUTOFF_REPEATS, KG_NEW_POINTS = 12, 24
REFERENCE_POWER, REFERENCE_SEED = 12, 23157
JITTER, LOG_FLOOR = 1e-7, 1e-14
KG_NODES, KG_WEIGHTS = np.polynomial.hermite.hermgauss(24)
KG_NODES *= np.sqrt(2)
KG_WEIGHTS /= np.sqrt(np.pi)

PROBLEMS = ("Curved valley", "BBOB f14", "BBOB f15")
if PROFILE == "full":
    SEEDS, NOISE = list(range(10000, 10024)), (1e-3, 1e-2)
elif PROFILE == "smoke":
    SEEDS, NOISE = [10000], (1e-2,)
else:
    raise ValueError("Use 'smoke' or 'full'.")

OUT = OUTPUT_ROOT / PROFILE
SOFTWARE = {"python": sys.version.split()[0], **{
    pkg: version(pkg) for pkg in ("numpy", "scipy", "pandas", "matplotlib", "cma", "coco-experiment")
}}
SETTINGS = dict(schema=1, seeds=SEEDS, problems=list(PROBLEMS), noise=list(NOISE),
                first_bbob_instance=101, budget=BUDGET, population=POPULATION,
                pool=POOL, prefix=SEARCH_PREFIX, repeat_start=REPEAT_START,
                cutoff_cap=MAX_CUTOFF_REPEATS, kg_points=KG_NEW_POINTS, jitter=JITTER,
                reference_power=REFERENCE_POWER, reference_seed=REFERENCE_SEED,
                software=SOFTWARE)
print(f"{PROFILE}: {len(SEEDS)} blocks under {OUT}")
print(SOFTWARE)

The valley uses its original disk, analytic normaliser and 45-degree
orientation. The BBOB functions use the box $[-5,5]^2$, mapped to
$[-1,1]^2$ for both optimisers. Their clean gap is
$(f(x)-f^*)/s$, where $s$ is the mean unnormalised gap on the same
frozen 4,096-point Sobol design for every instance. The evaluator
uses COCO's exact optimum value and adds noise after normalisation.
Scale calibration and offline scoring are benchmark setup and
assessment. They do not count as noisy optimiser calls and are
never provided as candidate observations. This sets a common
global noise scale while local differences can vary across functions.
BBOB already rotates and shifts its instances, so no extra rotation
experiment is run on those functions.

In [ ]:
%%writefile -a master_code/transfer_original.py

def rotate(point, angle):
    x, y = point
    return cos(angle) * x - sin(angle) * y, sin(angle) * x + cos(angle) * y


@dataclass(frozen=True)
class Valley:
    centre: tuple
    angle: float
    curvature_ratio: float = 100.0
    bend: float = 0.0

    @property
    def normaliser(self):
        # Under uniform disk sampling E[X^2] = 1/4 and E[X^4] = 1/8.
        # Terms odd in either coordinate integrate to zero.
        a, b = self.centre
        if self.bend == 0:
            k = self.curvature_ratio
            return (1 + k) / 4 + a*a + k*b*b
        eu2 = .25 + a*a
        ev2 = .25 + b*b
        eu4 = .125 + 1.5*a*a + a**4
        return eu2 + self.curvature_ratio * (
            ev2 + 2*self.bend*b*eu2 + self.bend**2*eu4
        )

    def canonical_gap(self, x, y):
        a, b = self.centre
        u, v = x - a, y - b
        return (u**2 + self.curvature_ratio*(v - self.bend*u**2)**2) / self.normaliser

    def clean(self, point):
        if hypot(*point) > 1.0 + 1e-12:
            raise ValueError("Point outside the unit disk")
        x, y = rotate(point, -self.angle)
        return self.canonical_gap(x, y)

    def observe(self, point, sigma, rng):
        return self.clean(point) + sigma * rng.gauss(0.0, 1.0)


def rotation(degree):
    angle = math.radians(degree)
    return np.array([rotate((1, 0), angle), rotate((0, 1), angle)]).T


def disk(rng, size):
    u = rng.random((size, 2))
    angle = 2 * np.pi * u[:, 0]
    return np.sqrt(u[:, 1:]) * np.column_stack((np.cos(angle), np.sin(angle)))

In [ ]:
%%writefile -a master_code/transfer_original.py

class BBOBGap:
    def __init__(self, function, instance):
        self.problem = cocoex.BareProblem("bbob", function, 2, instance)
        self.optimum = float(self.problem.best_value())
        optimum_point = np.asarray(self.problem.best_parameter(), dtype=float)
        if optimum_point.shape != (2,) or np.any(np.abs(optimum_point) > 5):
            raise ValueError("BBOB optimum must lie within the study box.")
        np.testing.assert_allclose(self.problem(optimum_point), self.optimum,
                                   atol=1e-10, rtol=1e-12)
        reference = qmc.Sobol(d=2, scramble=True, seed=REFERENCE_SEED)
        x = np.ascontiguousarray(10 * reference.random_base2(REFERENCE_POWER) - 5)
        self.scale = float(np.mean(self.problem(x) - self.optimum))
        assert np.isfinite(self.scale) and self.scale > 0

    def clean(self, point):
        point = np.asarray(point, dtype=float)
        if point.shape != (2,) or np.any(np.abs(point) > 1 + 1e-12):
            raise ValueError("Candidate is outside the normalised BBOB box.")
        gap = (float(self.problem(5 * point)) - self.optimum) / self.scale
        if gap < -1e-12:
            raise ValueError("Negative BBOB gap exceeds roundoff tolerance.")
        return max(0.0, gap)


def proposal_pool(rng, q, box):
    if box:
        return rng.uniform(-1., 1., size=(POOL, 2))
    return disk(rng, POOL) @ q.T

In [ ]:
%%writefile -a master_code/transfer_original.py

def covariance(x, y, theta):
    lengths = np.exp(theta[1:])
    return np.exp(theta[0] - .5 * cdist(x / lengths, y / lengths, "sqeuclidean"))


def fit(x, values, theta, refit, noise_sd=0.0, jitter=JITTER):
    output_scale = values.std() or 1
    y = (values - values.mean()) / output_scale
    ident = np.eye(len(x))

    def factor(parameters):
        signal = covariance(x, x, parameters)
        lower = cholesky(signal + (jitter + (noise_sd/output_scale)**2) * ident, lower=True)
        return signal, lower, cho_solve((lower, True), y)

    def likelihood(parameters):
        signal, lower, alpha = factor(parameters)
        loss = .5 * (y @ alpha) + np.log(np.diag(lower)).sum()
        weights = .5 * (cho_solve((lower, True), ident) - np.outer(alpha, alpha)) * signal
        gradient = [weights.sum()]
        if len(parameters) == 2:
            gradient.append((weights * cdist(x, x, "sqeuclidean")).sum() / np.exp(2*parameters[1]))
        else:
            for axis, log_length in enumerate(parameters[1:]):
                diff = x[:, axis, None] - x[:, axis]
                gradient.append((weights * diff**2).sum() / np.exp(2*log_length))
        return loss, np.array(gradient)

    status = -1
    if refit:
        bounds = [(-6, 6)] + [(math.log(.02), math.log(2))] * (len(theta)-1)
        result = minimize(likelihood, theta, jac=True, bounds=bounds, method="L-BFGS-B",
                          options={"maxiter": 50})
        theta, status = result.x, int(result.status)
    _, lower, alpha = factor(theta)
    return theta, lower, alpha, status


def predict(x, candidates, theta, lower, alpha):
    cross = covariance(x, candidates, theta)
    projected = solve_triangular(lower, cross, lower=True, check_finite=False)
    variance = np.maximum(0, np.exp(theta[0]) - np.einsum("ij,ij->j", projected, projected))
    return cross.T @ alpha, np.sqrt(variance)

In [ ]:
%%writefile -a master_code/transfer_original.py

def disk_constraint(point):
    return 1 - point @ point


def disk_jacobian(point):
    return -2 * point


def refined_candidate(x, pool, theta, lower, alpha, pool_lcb, box=False):
    """Refine two promising pool candidates with the same fitted LCB."""
    precision = np.broadcast_to(np.exp(-2 * theta[1:]), (2,))
    amplitude = np.exp(theta[0])

    def lcb(p):
        delta = x - p
        kernel = amplitude * np.exp(-.5 * ((delta ** 2) @ precision))
        derivative = kernel[:, None] * delta * precision
        projected = solve_triangular(lower, kernel, lower=True, check_finite=False)
        weights = solve_triangular(lower.T, projected, lower=False, check_finite=False)
        # A tiny variance floor avoids an undefined derivative at zero.
        variance = amplitude - projected @ projected
        sd = np.sqrt(max(variance, 1e-20))
        mean_gradient = derivative.T @ alpha
        gradient = mean_gradient + 2 * (derivative.T @ weights) / sd if variance > 1e-20 else mean_gradient
        return kernel @ alpha - 2 * sd, gradient

    best = pool[int(np.argmin(pool_lcb))]
    best_lcb = float(np.min(pool_lcb))
    starts = [best]
    distant = np.linalg.norm(pool - best, axis=1) >= .05
    if distant.any():
        starts.append(pool[np.argmin(np.where(distant, pool_lcb, np.inf))])
    warnings, improved = 0, False
    for start in starts:
        constraints = () if box else [{"type": "ineq", "fun": disk_constraint,
                                      "jac": disk_jacobian}]
        result = minimize(lcb, start, jac=True, method="SLSQP",
                          constraints=constraints, bounds=[(-1., 1.)]*2 if box else None,
                          options={"maxiter": 25, "ftol": 1e-10})
        warnings += int(not result.success)
        if not np.all(np.isfinite(result.x)):
            continue
        if box:
            if np.any(np.abs(result.x) > 1 + 1e-8):
                continue
            candidate = np.clip(result.x, -1., 1.)
        else:
            if np.linalg.norm(result.x) > 1 + 1e-8:
                continue
            candidate = result.x / max(1., np.linalg.norm(result.x))
        # Compare with the stable triangular-factor prediction used for the pool.
        mean, sd = predict(x, candidate[None, :], theta, lower, alpha)
        score = float((mean - 2 * sd)[0])
        if score < best_lcb - 1e-12:
            best, best_lcb, improved = candidate, score, True
    return best, warnings, int(improved)

In [ ]:
%%writefile -a master_code/transfer_original.py

def run_coupled_cma(evaluate, q, seed, budget=BUDGET, can_start=None,
                    evaluate_population=None, box=False):
    """Use C^(1/2) Q z for matched draws, retaining the reference tell update."""
    state = CMAES([0., 0.], .4, popsize=POPULATION)
    rng = np.random.default_rng(seed)
    points, scores, states = [], [], []
    rejected = 0
    while len(points) + POPULATION <= budget and (can_start is None or can_start()):
        state.C.update_eigensystem(state.counteval, state.params.lazy_gap_evals)
        basis = np.asarray(state.C.eigenbasis)
        root = (basis * np.sqrt(state.C.eigenvalues)) @ basis.T
        population = []
        while len(population) < POPULATION:
            point = np.asarray(state.xmean) + state.sigma * (root @ (q @ rng.standard_normal(2)))
            outside = np.any(np.abs(point) > 1) if box else np.linalg.norm(point) > 1
            if outside:
                rejected += 1
                if rejected > 10000:
                    raise RuntimeError("CMA rejection cap")
                continue
            population.append(point.tolist())
        values = ([evaluate(p) for p in population] if evaluate_population is None
                  else evaluate_population(population))
        state.tell(population, values)
        points.extend(population)
        scores.extend(values)
        covariance = np.array([[state.C[i][j] for j in range(2)] for i in range(2)])
        eigvals, vectors = np.linalg.eigh(covariance)
        major_axis = vectors[:, -1]
        states.append(dict(call=state.counteval, ratio=float(eigvals[-1]/eigvals[0]),
                           axis_error=float(np.degrees(np.arccos(np.clip(abs(major_axis @ q[:, 0]), 0, 1)))),
                           sigma=state.sigma, covariance=covariance.tolist(), mean=state.xmean))
    assert state.counteval == len(points)
    if can_start is None:
        assert state.counteval == budget
    return points, scores, dict(states=states, rejections=rejected)

In [ ]:
%%writefile -a master_code/transfer_original.py

class Evaluator:
    def __init__(self, valley, sigma, search_noise, repeat_noise):
        self.valley, self.sigma = valley, sigma
        self.search_noise, self.repeat_noise = search_noise, repeat_noise
        self.search_count = self.repeat_count = 0
        self.trace = []

    def measure(self, point, repeat=False):
        if repeat:
            noise = self.repeat_noise[self.repeat_count]
            self.repeat_count += 1
        else:
            noise = self.search_noise[self.search_count]
            self.search_count += 1
        value = float(self.valley.clean(point) + self.sigma * noise)
        self.trace.append(dict(point=np.asarray(point).tolist(), observed=value, repeat=repeat))
        return value


def posterior_scores(points, values, theta, noise_sd, candidates):
    x = np.asarray(points)
    _, lower, alpha, _ = fit(x, np.asarray(values), theta, False, noise_sd=noise_sd)
    means, _ = predict(x, np.asarray(candidates), theta, lower, alpha)
    return means

In [ ]:
%%writefile -a master_code/transfer_original.py

def cutoff_cma(meter, q, seed, box=False):
    archived_sums, archived_counts = [], []

    def remaining():
        return BUDGET - meter.search_count - meter.repeat_count

    def score_population(population):
        sums = np.array([meter.measure(p) for p in population])
        counts = np.ones(POPULATION, dtype=int)
        if (meter.search_count > REPEAT_START and remaining() >= 2
                and meter.repeat_count + 2 <= MAX_CUTOFF_REPEATS):
            order = np.argsort(sums, kind="stable")
            boundary = order[POPULATION // 2 - 1:POPULATION // 2 + 1]
            if sums[boundary[1]] - sums[boundary[0]] <= np.sqrt(2) * meter.sigma:
                for idx in boundary:
                    sums[idx] += meter.measure(population[idx], repeat=True)
                    counts[idx] += 1
        archived_sums.extend(sums)
        archived_counts.extend(counts)
        return sums / counts

    points, update_scores, _ = run_coupled_cma(
        meter.measure, q, seed, can_start=lambda: remaining() >= POPULATION,
        evaluate_population=score_population, box=box,
    )
    assert len(points) >= 84
    update_repeats = meter.repeat_count
    top_two = np.argsort(np.asarray(archived_sums) / archived_counts, kind="stable")[:2]
    for i in range(remaining()):
        idx = top_two[i % len(top_two)]
        archived_sums[idx] += meter.measure(points[idx], repeat=True)
        archived_counts[idx] += 1
    scores = np.asarray(archived_sums) / archived_counts
    return dict(points=np.asarray(points), scores=scores, evaluator=meter,
                update_scores=np.asarray(update_scores), update_repeats=update_repeats)

In [ ]:
%%writefile -a master_code/transfer_original.py

def distinct_indices(points):
    return np.sort(np.unique(points, axis=0, return_index=True)[1])


def posterior_joint(x, candidates, theta, lower, alpha):
    cross = covariance(x, candidates, theta)
    solved = solve_triangular(lower, cross, lower=True, check_finite=False)
    cov = covariance(candidates, candidates, theta) - solved.T @ solved
    return cross.T @ alpha, (cov + cov.T) / 2


def knowledge_gradient_choice(x, y, theta, lower, alpha, old, new, noise_sd):
    candidates = np.vstack([old, new])
    means, cov = posterior_joint(x, candidates, theta, lower, alpha)
    noise_standardised = noise_sd / (np.std(y) or 1)
    observation_sd = np.sqrt(np.maximum(np.diag(cov), 0) + noise_standardised**2)
    slope = cov / observation_sd[None, :]
    n_old = len(old)
    fantasies = means[:n_old, None, None] + slope[:n_old, :, None] * KG_NODES
    best = fantasies.min(axis=0)
    if len(new):
        new_fantasy = means[n_old:, None] + np.diag(slope)[n_old:, None] * KG_NODES
        best[n_old:] = np.minimum(best[n_old:], new_fantasy)
    return int(np.argmin(best @ KG_WEIGHTS))

Within each instance and noise level, CMA-ES and BO share the first six
locations and observations. New proposal errors are paired by proposal
number across policies, and intentional repeats use a separate stream.
The BO branches share the exact 72-call prefix, and both use the same
model and hyperparameter schedule as continued LCB. Alternation begins
with an LCB proposal and then repeats the current posterior-mean leader.
KG considers every distinct visited point and up to 24 fresh LCB candidates,
including the refined proposal when it is distinct.

In [ ]:
%%writefile -a master_code/transfer_original.py

CMA, BO = "CMA-ES", "Isotropic BO"
STANDARD, CUTOFF = "Standard", "Cutoff repeats"
NATIVE, GP = "Native", "GP reranking"
LCB, ALTERNATE, KG = "LCB throughout", "Alternating 12 + 12", "KG after 72"
METRICS = ("recommendation", "best_visited", "selection_loss")


def bo_search(meter, q, initial, seed, box, policy, prefix=None):
    rng = np.random.default_rng(seed)
    theta = np.log([1., .4])
    if prefix is None:
        for point in initial:
            meter.measure(point)
    else:
        meter.trace = list(prefix["trace"])
        meter.search_count = SEARCH_PREFIX
        theta = prefix["theta"].copy()
        for _ in range(POPULATION, SEARCH_PREFIX):
            proposal_pool(rng, q, box)

    checkpoint = None
    while len(meter.trace) < BUDGET:
        x = np.asarray([r["point"] for r in meter.trace])
        y = np.asarray([r["observed"] for r in meter.trace])
        t = len(y)
        theta, lower, alpha, _ = fit(x, y, theta, t % POPULATION == 0,
                                    noise_sd=meter.sigma)
        pool = proposal_pool(rng, q, box)
        old = x[distinct_indices(x)]
        if policy == ALTERNATE and (t - SEARCH_PREFIX) % 2 == 1:
            means, _ = predict(x, old, theta, lower, alpha)
            meter.measure(old[np.argmin(means)], repeat=True)
            continue

        mean, sd = predict(x, pool, theta, lower, alpha)
        lcb = mean - 2 * sd
        refined, _, _ = refined_candidate(x, pool, theta, lower, alpha, lcb, box=box)
        if policy == KG:
            seen = {tuple(p) for p in old}
            new = []
            for point in (refined, *pool[np.argsort(lcb, kind="stable")[:KG_NEW_POINTS]]):
                if tuple(point) not in seen and len(new) < KG_NEW_POINTS:
                    new.append(point)
                    seen.add(tuple(point))
            new = np.asarray(new).reshape(-1, 2)
            chosen = knowledge_gradient_choice(x, y, theta, lower, alpha, old, new, meter.sigma)
            if chosen < len(old):
                meter.measure(old[chosen], repeat=True)
            else:
                meter.measure(new[chosen - len(old)])
        else:
            meter.measure(refined)
        if len(meter.trace) == SEARCH_PREFIX:
            checkpoint = dict(trace=list(meter.trace), theta=theta.copy())
    x = np.asarray([r["point"] for r in meter.trace])
    y = np.asarray([r["observed"] for r in meter.trace])
    candidates = x[distinct_indices(x)]
    scores = posterior_scores(x, y, theta, meter.sigma, candidates)
    return dict(points=candidates, scores=scores, evaluator=meter, prefix=checkpoint)

In [ ]:
%%writefile -a master_code/transfer_original.py

def score_recommendation(valley, candidates, chosen, **labels):
    clean = np.array([valley.clean(p) for p in candidates])
    recommended, best = float(clean[chosen]), float(clean.min())
    return dict(**labels, recommendation=recommended, best_visited=best,
                selection_loss=recommended-best)


def cma_recommendations(valley, trace, native_index, noise_sd, **labels):
    x = np.asarray([r["point"] for r in trace])
    y = np.asarray([r["observed"] for r in trace])
    proposals = np.asarray([r["point"] for r in trace if not r["repeat"]])
    candidates = x[distinct_indices(x)]
    native_point = proposals[native_index]
    native = int(np.flatnonzero(np.all(candidates == native_point, axis=1))[0])
    theta, lower, alpha, _ = fit(x, y, np.log([1, .4]), True, noise_sd=noise_sd)
    means, _ = predict(x, candidates, theta, lower, alpha)
    return [score_recommendation(valley, candidates, idx, **labels, readout=readout,
                                 search_calls=len(proposals), repeat_calls=len(x)-len(proposals))
            for readout, idx in ((NATIVE, native), (GP, int(np.argmin(means))))]

The primary metric is the true recommendation gap at call 96. Also
record the best visited gap and their difference, selection loss.
For CMA-ES, compare the parents actually used with the clean top three
and record the clean gap between the third and fourth candidates.
This margin, divided by the noise SD, describes local ranking difficulty.

In [ ]:
%%writefile -a master_code/transfer_original.py

def parent_diagnostics(objective, points, update_scores, noise_sd, **labels):
    clean = np.array([objective.clean(p) for p in points])
    rows = []
    mu = POPULATION // 2
    for start in range(0, len(points), POPULATION):
        end = start + POPULATION
        order = np.argsort(clean[start:end], kind="stable")
        selected = np.argsort(update_scores[start:end], kind="stable")[:mu]
        replaced = len(set(selected) - set(order[:mu]))
        ordered_gaps = clean[start:end][order]
        rows.append(dict(**labels, generation=end // POPULATION,
                         parent_error=replaced / mu,
                         margin_over_noise=float((ordered_gaps[mu] - ordered_gaps[mu - 1]) / noise_sd)))
    return rows


def run_block(seed):
    centre = rotate((.4, 0), random.Random(seed).uniform(0, tau))
    instance = 101 + seed - SEEDS[0]
    search_noise = np.random.default_rng(
        np.random.SeedSequence([seed, 40117])).standard_normal(BUDGET)
    repeat_noise = np.random.default_rng(
        np.random.SeedSequence([seed, 73093])).standard_normal(BUDGET)
    rows, parents, histories, scales = [], [], [], []

    for name in PROBLEMS:
        box = name != "Curved valley"
        if box:
            objective = BBOBGap(int(name.split("f")[-1]), instance)
            q = np.eye(2)
            scale = objective.scale
            problem_id = str(objective.problem)
        else:
            objective = Valley(centre, math.pi/4, 100, .75)
            q = rotation(45)
            scale = float(objective.normaliser)
            problem_id = f"curved_k100_45_seed{seed}"
        scales.append(dict(seed=seed, problem=name, problem_id=problem_id, gap_scale=scale))

        for sigma in NOISE:
            labels = dict(seed=seed, problem=name, problem_id=problem_id, sigma=sigma)

            def meter():
                return Evaluator(objective, sigma, search_noise, repeat_noise)

            standard_meter = meter()
            points, scores, _ = run_coupled_cma(standard_meter.measure, q,
                                                seed + 30000, box=box)
            standard_points, standard_scores = np.asarray(points), np.asarray(scores)
            cutoff = cutoff_cma(meter(), q, seed + 30000, box=box)
            assert (standard_meter.trace[:REPEAT_START] ==
                    cutoff["evaluator"].trace[:REPEAT_START])
            for policy, evaluation, candidates, native_scores, update_scores in (
                (STANDARD, standard_meter, standard_points, standard_scores, standard_scores),
                (CUTOFF, cutoff["evaluator"], cutoff["points"], cutoff["scores"],
                 cutoff["update_scores"]),
            ):
                assert len(evaluation.trace) == BUDGET
                rows.extend(cma_recommendations(
                    objective, evaluation.trace, int(np.argmin(native_scores)), sigma,
                    **labels, method=CMA, policy=policy,
                ))
                parents.extend(parent_diagnostics(objective, candidates, update_scores, sigma,
                                                  **labels, policy=policy))
                histories.append(dict(**labels, method=CMA, policy=policy,
                                      trace=evaluation.trace))

            initial = standard_points[:POPULATION]
            baseline = bo_search(meter(), q, initial, seed + 20000, box, LCB)
            prefix = baseline["prefix"]
            assert len(prefix["trace"]) == SEARCH_PREFIX
            assert prefix["trace"][:POPULATION] == standard_meter.trace[:POPULATION]
            alternate = bo_search(meter(), q, initial, seed + 20000, box, ALTERNATE, prefix)
            kg = bo_search(meter(), q, initial, seed + 20000, box, KG, prefix)
            assert alternate["evaluator"].repeat_count == 12
            for policy, run in ((LCB, baseline), (ALTERNATE, alternate), (KG, kg)):
                evaluator = run["evaluator"]
                assert len(evaluator.trace) == BUDGET
                assert evaluator.trace[:SEARCH_PREFIX] == prefix["trace"]
                chosen = int(np.argmin(run["scores"]))
                rows.append(score_recommendation(
                    objective, run["points"], chosen, **labels, method=BO, policy=policy,
                    readout=GP, search_calls=evaluator.search_count,
                    repeat_calls=evaluator.repeat_count,
                ))
                histories.append(dict(**labels, method=BO, policy=policy,
                                      trace=evaluator.trace))
    return dict(results=rows, parents=parents, histories=histories, scales=scales)

In [ ]:
%%writefile -a master_code/transfer_original.py

def block_path(seed):
    return OUT / "blocks" / f"{seed}.json.gz"


def save_blocks():
    (OUT / "blocks").mkdir(parents=True, exist_ok=True)
    manifest = OUT / "settings.json"
    if manifest.exists():
        if json.loads(manifest.read_text(encoding="utf-8")) != SETTINGS:
            raise ValueError("Cached settings or software differ. Choose a new OUTPUT_ROOT.")
    else:
        manifest.write_text(json.dumps(SETTINGS, indent=2), encoding="utf-8")
    for i, seed in enumerate(SEEDS, 1):
        path = block_path(seed)
        if path.exists():
            print(f"[{i}/{len(SEEDS)}] Reusing block {seed}", flush=True)
            continue
        started = perf_counter()
        block = run_block(seed)
        temp = path.with_suffix(".tmp")
        with gzip.open(temp, "wt", encoding="utf-8") as file:
            json.dump(block, file, allow_nan=False, separators=(",", ":"))
        temp.replace(path)
        print(f"[{i}/{len(SEEDS)}] Saved block {seed} ({perf_counter()-started:.1f} s)",
              flush=True)

In [ ]:
%%writefile -a master_code/transfer_original.py

save_blocks()

Each contrast is paired by instance and seed. The recommendation gap
ratio is the geometric mean of within-seed ratios. A value below one
favours the first policy. Intervals are pointwise percentile intervals
from 10,000 resamples of complete seed blocks. Raw gap differences
are also saved, so changes in search and selection can be separated.
The table shows each function and noise level separately.

In [ ]:
%%writefile -a master_code/transfer_original.py

CONDITIONS = ["problem", "sigma", "method"]
COMPARISONS = {
    CMA: [
        ((STANDARD, GP), (STANDARD, NATIVE), "GP vs. native · standard"),
        ((CUTOFF, GP), (CUTOFF, NATIVE), "GP vs. native · cutoff"),
        ((CUTOFF, GP), (STANDARD, GP), "Cutoff vs. standard · both GP"),
        ((CUTOFF, NATIVE), (STANDARD, NATIVE), "Cutoff vs. standard · both native"),
    ],
    BO: [
        ((ALTERNATE, GP), (LCB, GP), "Alternation vs. LCB"),
        ((KG, GP), (LCB, GP), "KG vs. LCB"),
        ((KG, GP), (ALTERNATE, GP), "KG vs. alternation"),
    ],
}


def paired_effects(results):
    contrasts = []
    for key, group in results.groupby(CONDITIONS):
        labels = dict(zip(CONDITIONS, key))
        groups = {(p, r): t.set_index("seed").sort_index()
                  for (p, r), t in group.groupby(["policy", "readout"])}
        n = len(next(iter(groups.values())))
        indices = (np.random.default_rng(4670).integers(0, n, (10000, n)) if n > 1 else None)
        for left, right, name in COMPARISONS[labels["method"]]:
            a, b = groups[left], groups[right]
            assert a.index.is_unique and a.index.equals(b.index)
            gaps_a, gaps_b = a[list(METRICS)].to_numpy(), b[list(METRICS)].to_numpy()
            raw = gaps_a - gaps_b
            np.testing.assert_allclose(raw[:, 0], raw[:, 1] + raw[:, 2], atol=1e-14)
            log_ratio = np.log10(np.maximum(gaps_a[:, 0], LOG_FLOOR)) - np.log10(
                np.maximum(gaps_b[:, 0], LOG_FLOOR))
            for metric, scale, delta in [
                *[(name, "raw", raw[:, j]) for j, name in enumerate(METRICS)],
                ("recommendation", "log10", log_ratio),
            ]:
                bounds = (np.quantile(delta[indices].mean(axis=1), [.025, .975])
                          if indices is not None else [np.nan, np.nan])
                contrasts.append(dict(**labels, contrast=name, metric=metric, scale=scale,
                                      n=n, mean=delta.mean(), low_95=bounds[0],
                                      high_95=bounds[1], fraction_improved=np.mean(delta < 0)))
    return pd.DataFrame(contrasts)

In [ ]:
%%writefile -a master_code/transfer_original.py

records, rank_rows, scale_rows = [], [], []
for seed in SEEDS:
    with gzip.open(block_path(seed), "rt", encoding="utf-8") as file:
        block = json.load(file)
    records.extend(block["results"])
    rank_rows.extend(block["parents"])
    scale_rows.extend(block["scales"])
results = pd.DataFrame(records)
ranks = pd.DataFrame(rank_rows)
scales = pd.DataFrame(scale_rows)
assert len(results) == len(SEEDS)*len(PROBLEMS)*len(NOISE)*7
assert not results.duplicated(["seed", *CONDITIONS, "policy", "readout"]).any()
assert (results.search_calls + results.repeat_calls == BUDGET).all()
cma_best = results[results.method == CMA].pivot(
    index=["seed", *CONDITIONS, "policy"], columns="readout", values="best_visited")
np.testing.assert_array_equal(cma_best[NATIVE], cma_best[GP])
results.to_csv(OUT / "run_gaps.csv", index=False)
ranks.to_csv(OUT / "cma_parent_diagnostics.csv", index=False)
scales.to_csv(OUT / "instance_normalisers.csv", index=False)

medians = results.groupby([*CONDITIONS, "policy", "readout"], as_index=False)[
    [*METRICS, "search_calls", "repeat_calls"]].median()
medians.to_csv(OUT / "median_gaps_and_allocation.csv", index=False)
effects = paired_effects(results)
effects.to_csv(OUT / "paired_effects.csv", index=False)

ratio = effects[effects.scale == "log10"].copy()
ratio[["ratio", "lower", "upper"]] = 10.0 ** ratio[["mean", "low_95", "high_95"]]
ratio["ratio [95% interval]"] = [
    f"{r:.2f} [{lo:.2f}, {hi:.2f}]" if np.isfinite(lo) else f"{r:.2f}"
    for r, lo, hi in zip(ratio.ratio, ratio.lower, ratio.upper)
]
compact = ratio[ratio.contrast.isin([
    "GP vs. native · standard", "Cutoff vs. standard · both GP",
    "Alternation vs. LCB", "KG vs. LCB", "KG vs. alternation",
])]
table = compact.pivot(index=["method", "contrast"], columns=["problem", "sigma"],
                      values="ratio [95% interval]")
display(table.reindex(columns=pd.MultiIndex.from_product([PROBLEMS, NOISE])))
table.to_csv(OUT / "transfer_table.csv")

In [ ]:
%%writefile -a master_code/transfer_original.py

fig, axes = plt.subplots(1, len(PROBLEMS), sharey=True, figsize=(12, 4.4),
                         layout="constrained")
contrasts = list(compact.contrast.unique())
labels = {name: i for i, name in enumerate(contrasts)}
for ax, problem in zip(axes, PROBLEMS):
    data = compact[compact.problem == problem]
    for j, (sigma, color) in enumerate(zip(NOISE, ("#0072b2", "#b45309"))):
        group = data[data.sigma == sigma]
        for i, row in enumerate(group.itertuples()):
            y = labels[row.contrast] + (.22*(j-.5) if len(NOISE) > 1 else 0)
            if np.isfinite(row.lower):
                ax.hlines(y, row.lower, row.upper, color=color, lw=1.8)
            ax.scatter(row.ratio, y, color=color, s=22,
                       label=f"Noise SD {sigma:g}" if i == 0 else None)
    ax.axvline(1, color=".55", ls="--")
    ax.set_xscale("log")
    ax.set_title(problem)
    ax.set_yticks(range(len(contrasts)), contrasts)
axes[0].invert_yaxis()
axes[-1].legend(loc="best", fontsize=8)
fig.supxlabel("Paired true recommendation gap ratio")
fig.savefig(OUT / "policy-transfer.png", dpi=220, bbox_inches="tight")
plt.show()
plt.close(fig)

In [ ]:
%%writefile -a master_code/transfer_original.py

archive_path = shutil.make_archive(str(OUTPUT_ROOT / f"noisy_policy_transfer_{PROFILE}"),
                                   "zip", root_dir=OUT)
print(f"Download or save this archive from the Colab Files panel: {archive_path}")

In [ ]:
run_stage("transfer_original")

## New Stage C - search, selection and evaluation allocation

### Frozen design

The main experiment crosses straight/curved valleys with 0/45 degree orientations,
at kappa 100. Each policy receives exactly 96 calls including initialisation.
Noise SDs include zero, 0.001, and 0.01. Seeds 10100-10123 are distinct from the
earlier experiments. BBOB f14 and f15 use the same policies as a supporting check.

The primary new contrast is unrestricted KG versus KG restricted to new points on
the rotated curved valley at noise SD 0.01. Other contrasts diagnose search/selection
effects or sensitivity. The original alternation baseline is retained.

| BO branch | Calls 1-72 | Calls 73-96 |
| --- | --- | --- |
| Continued LCB | Shared LCB history | New proposals |
| Alternation | Same history | 12 new proposals and 12 repeats |
| KG unrestricted | Same history | New proposals or visited points |
| KG new only | Same history | New proposals |
| KG repeats only | Same history | Visited points |

All BO branches share the GP, hyperparameter schedule, screened candidate generator,
24 quadrature nodes, and posterior-mean recommendation rule. The measurement
restriction changes which actions can be selected. Each branch subsequently develops
its own history. The rule is a finite-candidate one-step approximation to the expected
quality of the final recommendation over visited points plus the proposed next point.
This is the existing implementation with two explicit action restrictions.

CMA-ES retains standard search and the existing cutoff-repeat heuristic. Each history
is scored by its lowest observed/averaged score and by a common isotropic GP readout.
The cutoff heuristic can spend remaining calls on its two archived leaders, as before.

### Measurements and interpretation

For a recommendation among visited points, $R=S+L$, where $R$ is recommendation gap,
$S$ is best visited gap, and $L$ is selection loss. The code checks this identity for
every run and for paired raw differences. Means preserve the decomposition. The
component intervals are marginal pointwise intervals and their endpoints do not add.

The 72-call oracle diagnostic compares the best visited gap with the 96-call baseline
recommendation. A positive `oracle_advantage` means perfect selection at 72 could
have beaten that baseline. It bounds pure verification on the frozen candidate set.
It does not bound branches that discover new candidates. No oracle values enter a
policy decision.

Local noise diagnostics use the four candidates ranked highest by each prefix readout.
They record shortlist loss, clean pairwise gaps, and the implied probability of an
incorrect comparison using one or six fresh independent measurements per candidate.
These probabilities concern fixed pairs conditional on their locations, not the
adaptive shortlist-selection procedure or the probability of selecting the best of four.
At zero noise the ratio to noise is undefined and is saved as missing; clean gaps and
pairwise error probabilities remain available. GP smoothing still has fixed jitter.

CMA-ES records clean third-fourth margins and parent replacement fractions by
generation. Comparisons of ranks use matched generations, while final gaps include
the cost of completing fewer updates. A single scalar margin cannot describe all
possible misrankings and is interpreted together with the observed replacement rate.

The prefix GP is refitted at the 72-call boundary, matching the next scheduled BO
refit. The native CMA prefix readout uses its single observed scores. Prefix diagnostics
use the unmodified standard/LCB histories, before the new branches diverge.

All raw histories, prefix diagnostics, individual paired effects and instance scales
are exported so the later report revision can test interactions and practical effect
sizes. The current analysis retains pointwise paired bootstrap intervals. Statistical
reframing and report prose are deferred until the results are reviewed.

In [ ]:
%%writefile master_code/mechanisms.py
from pathlib import Path
import os

MASTER_PROFILE = globals().get("MASTER_PROFILE", os.environ.get("OPT_PROFILE", "smoke"))
MASTER_ROOT = Path(globals().get("MASTER_ROOT", os.environ.get("OPT_RESULTS_ROOT", "optimisation_results"))).resolve()
MASTER_INCLUDE_TRANSFER = globals().get("MASTER_INCLUDE_TRANSFER", os.environ.get("OPT_INCLUDE_TRANSFER", "1") == "1")
if MASTER_PROFILE not in ("full", "smoke"):
    raise ValueError("Choose full or smoke.")

### Imports and experiment configuration

In [ ]:
%%writefile -a master_code/mechanisms.py

import gzip
import json
import math
import os
import random
import runpy
import sys
from dataclasses import dataclass
from importlib.metadata import distribution, version
from math import cos, sin, hypot, tau
from pathlib import Path
from time import perf_counter

for variable in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS"):
    os.environ[variable] = "1"

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.linalg import cholesky, cho_solve, solve_triangular
from scipy.optimize import minimize
from scipy.spatial.distance import cdist

pure_cma = distribution("cma").locate_file("cma/more_algorithms/purecma.py")
CMAES = runpy.run_path(str(pure_cma))["CMAES"]

import hashlib
import cocoex
from scipy.stats import qmc
from scipy.special import ndtr

In [ ]:
%%writefile -a master_code/mechanisms.py

PROFILE = MASTER_PROFILE
OUTPUT_ROOT = MASTER_ROOT / "stage_c_mechanisms"
BUDGET, POPULATION, POOL = 96, 6, 1024
SEARCH_PREFIX, REPEAT_START = 72, 48
MAX_CUTOFF_REPEATS, KG_NEW_POINTS = 12, 24
REFERENCE_POWER, REFERENCE_SEED = 12, 23157
JITTER, LOG_FLOOR = 1e-7, 1e-14
KG_NODES, KG_WEIGHTS = np.polynomial.hermite.hermgauss(24)
KG_NODES *= np.sqrt(2)
KG_WEIGHTS /= np.sqrt(np.pi)
SEEDS = list(range(10100, 10124)) if PROFILE == "full" else [10100, 10101]
NOISE = (0., 1e-3, 1e-2) if PROFILE == "full" else (0., 1e-2)
PROBLEM_SPECS = [
    dict(problem="Straight 0 deg", scope="Main valleys", bend=0., angle=0, function=None),
    dict(problem="Straight 45 deg", scope="Main valleys", bend=0., angle=45, function=None),
    dict(problem="Curved 0 deg", scope="Main valleys", bend=.75, angle=0, function=None),
    dict(problem="Curved 45 deg", scope="Main valleys", bend=.75, angle=45, function=None),
]
if MASTER_INCLUDE_TRANSFER:
    PROBLEM_SPECS.extend([
        dict(problem="BBOB f14", scope="Supporting transfer", bend=None, angle=None, function=14),
        dict(problem="BBOB f15", scope="Supporting transfer", bend=None, angle=None, function=15),
    ])
OUT = OUTPUT_ROOT / PROFILE
SOFTWARE = {"python": sys.version.split()[0], **{
    name: version(name) for name in ("numpy", "scipy", "pandas", "matplotlib", "cma", "coco-experiment")
}}
SETTINGS = dict(
    schema=2, profile=PROFILE, seeds=SEEDS, problems=PROBLEM_SPECS, noise=list(NOISE),
    first_bbob_instance=201, budget=BUDGET, population=POPULATION, pool=POOL,
    prefix=SEARCH_PREFIX, repeat_start=REPEAT_START, cutoff_cap=MAX_CUTOFF_REPEATS,
    kg_points=KG_NEW_POINTS, quadrature_nodes=24, jitter=JITTER,
    reference_power=REFERENCE_POWER, reference_seed=REFERENCE_SEED,
    software=SOFTWARE, source_sha256=hashlib.sha256(Path(__file__).read_bytes()).hexdigest(),
)
print(f"Stage C mechanisms: {PROFILE}, {len(SEEDS)} paired blocks, "
      f"{len(PROBLEM_SPECS)} geometries, noise SDs {NOISE}")
print("Primary comparison: unrestricted vs new-only KG on Curved 45 deg at noise SD 0.01.")
print("Other comparisons diagnose mechanisms and sensitivity. All intervals are pointwise.")

### Objectives and normalisation

In [ ]:
%%writefile -a master_code/mechanisms.py

def rotate(point, angle):
    x, y = point
    return cos(angle) * x - sin(angle) * y, sin(angle) * x + cos(angle) * y


@dataclass(frozen=True)
class Valley:
    centre: tuple
    angle: float
    curvature_ratio: float = 100.0
    bend: float = 0.0

    @property
    def normaliser(self):
        # Under uniform disk sampling E[X^2] = 1/4 and E[X^4] = 1/8.
        # Terms odd in either coordinate integrate to zero.
        a, b = self.centre
        if self.bend == 0:
            k = self.curvature_ratio
            return (1 + k) / 4 + a*a + k*b*b
        eu2 = .25 + a*a
        ev2 = .25 + b*b
        eu4 = .125 + 1.5*a*a + a**4
        return eu2 + self.curvature_ratio * (
            ev2 + 2*self.bend*b*eu2 + self.bend**2*eu4
        )

    def canonical_gap(self, x, y):
        a, b = self.centre
        u, v = x - a, y - b
        return (u**2 + self.curvature_ratio*(v - self.bend*u**2)**2) / self.normaliser

    def clean(self, point):
        if hypot(*point) > 1.0 + 1e-12:
            raise ValueError("Point outside the unit disk")
        x, y = rotate(point, -self.angle)
        return self.canonical_gap(x, y)

    def observe(self, point, sigma, rng):
        return self.clean(point) + sigma * rng.gauss(0.0, 1.0)


def rotation(degree):
    angle = math.radians(degree)
    return np.array([rotate((1, 0), angle), rotate((0, 1), angle)]).T


def disk(rng, size):
    u = rng.random((size, 2))
    angle = 2 * np.pi * u[:, 0]
    return np.sqrt(u[:, 1:]) * np.column_stack((np.cos(angle), np.sin(angle)))


class BBOBGap:
    def __init__(self, function, instance):
        self.problem = cocoex.BareProblem("bbob", function, 2, instance)
        self.optimum = float(self.problem.best_value())
        optimum_point = np.asarray(self.problem.best_parameter(), dtype=float)
        if optimum_point.shape != (2,) or np.any(np.abs(optimum_point) > 5):
            raise ValueError("BBOB optimum must lie within the study box.")
        np.testing.assert_allclose(self.problem(optimum_point), self.optimum,
                                   atol=1e-10, rtol=1e-12)
        reference = qmc.Sobol(d=2, scramble=True, seed=REFERENCE_SEED)
        x = np.ascontiguousarray(10 * reference.random_base2(REFERENCE_POWER) - 5)
        self.scale = float(np.mean(self.problem(x) - self.optimum))
        assert np.isfinite(self.scale) and self.scale > 0

    def clean(self, point):
        point = np.asarray(point, dtype=float)
        if point.shape != (2,) or np.any(np.abs(point) > 1 + 1e-12):
            raise ValueError("Candidate is outside the normalised BBOB box.")
        gap = (float(self.problem(5 * point)) - self.optimum) / self.scale
        if gap < -1e-12:
            raise ValueError("Negative BBOB gap exceeds roundoff tolerance.")
        return max(0.0, gap)


def proposal_pool(rng, q, box):
    if box:
        return rng.uniform(-1., 1., size=(POOL, 2))
    return disk(rng, POOL) @ q.T

### Reused GP and acquisition routines

In [ ]:
%%writefile -a master_code/mechanisms.py

def covariance(x, y, theta):
    lengths = np.exp(theta[1:])
    return np.exp(theta[0] - .5 * cdist(x / lengths, y / lengths, "sqeuclidean"))


def fit(x, values, theta, refit, noise_sd=0.0, jitter=JITTER):
    output_scale = values.std() or 1
    y = (values - values.mean()) / output_scale
    ident = np.eye(len(x))

    def factor(parameters):
        signal = covariance(x, x, parameters)
        lower = cholesky(signal + (jitter + (noise_sd/output_scale)**2) * ident, lower=True)
        return signal, lower, cho_solve((lower, True), y)

    def likelihood(parameters):
        signal, lower, alpha = factor(parameters)
        loss = .5 * (y @ alpha) + np.log(np.diag(lower)).sum()
        weights = .5 * (cho_solve((lower, True), ident) - np.outer(alpha, alpha)) * signal
        gradient = [weights.sum()]
        if len(parameters) == 2:
            gradient.append((weights * cdist(x, x, "sqeuclidean")).sum() / np.exp(2*parameters[1]))
        else:
            for axis, log_length in enumerate(parameters[1:]):
                diff = x[:, axis, None] - x[:, axis]
                gradient.append((weights * diff**2).sum() / np.exp(2*log_length))
        return loss, np.array(gradient)

    status = -1
    if refit:
        bounds = [(-6, 6)] + [(math.log(.02), math.log(2))] * (len(theta)-1)
        result = minimize(likelihood, theta, jac=True, bounds=bounds, method="L-BFGS-B",
                          options={"maxiter": 50})
        theta, status = result.x, int(result.status)
    _, lower, alpha = factor(theta)
    return theta, lower, alpha, status


def predict(x, candidates, theta, lower, alpha):
    cross = covariance(x, candidates, theta)
    projected = solve_triangular(lower, cross, lower=True, check_finite=False)
    variance = np.maximum(0, np.exp(theta[0]) - np.einsum("ij,ij->j", projected, projected))
    return cross.T @ alpha, np.sqrt(variance)


def disk_constraint(point):
    return 1 - point @ point


def disk_jacobian(point):
    return -2 * point


def refined_candidate(x, pool, theta, lower, alpha, pool_lcb, box=False):
    """Refine two promising pool candidates with the same fitted LCB."""
    precision = np.broadcast_to(np.exp(-2 * theta[1:]), (2,))
    amplitude = np.exp(theta[0])

    def lcb(p):
        delta = x - p
        kernel = amplitude * np.exp(-.5 * ((delta ** 2) @ precision))
        derivative = kernel[:, None] * delta * precision
        projected = solve_triangular(lower, kernel, lower=True, check_finite=False)
        weights = solve_triangular(lower.T, projected, lower=False, check_finite=False)
        # A tiny variance floor avoids an undefined derivative at zero.
        variance = amplitude - projected @ projected
        sd = np.sqrt(max(variance, 1e-20))
        mean_gradient = derivative.T @ alpha
        gradient = mean_gradient + 2 * (derivative.T @ weights) / sd if variance > 1e-20 else mean_gradient
        return kernel @ alpha - 2 * sd, gradient

    best = pool[int(np.argmin(pool_lcb))]
    best_lcb = float(np.min(pool_lcb))
    starts = [best]
    distant = np.linalg.norm(pool - best, axis=1) >= .05
    if distant.any():
        starts.append(pool[np.argmin(np.where(distant, pool_lcb, np.inf))])
    warnings, improved = 0, False
    for start in starts:
        constraints = () if box else [{"type": "ineq", "fun": disk_constraint,
                                      "jac": disk_jacobian}]
        result = minimize(lcb, start, jac=True, method="SLSQP",
                          constraints=constraints, bounds=[(-1., 1.)]*2 if box else None,
                          options={"maxiter": 25, "ftol": 1e-10})
        warnings += int(not result.success)
        if not np.all(np.isfinite(result.x)):
            continue
        if box:
            if np.any(np.abs(result.x) > 1 + 1e-8):
                continue
            candidate = np.clip(result.x, -1., 1.)
        else:
            if np.linalg.norm(result.x) > 1 + 1e-8:
                continue
            candidate = result.x / max(1., np.linalg.norm(result.x))
        # Compare with the stable triangular-factor prediction used for the pool.
        mean, sd = predict(x, candidate[None, :], theta, lower, alpha)
        score = float((mean - 2 * sd)[0])
        if score < best_lcb - 1e-12:
            best, best_lcb, improved = candidate, score, True
    return best, warnings, int(improved)

### Reused CMA-ES and measurement accounting

In [ ]:
%%writefile -a master_code/mechanisms.py

def run_coupled_cma(evaluate, q, seed, budget=BUDGET, can_start=None,
                    evaluate_population=None, box=False):
    """Use C^(1/2) Q z for matched draws, retaining the reference tell update."""
    state = CMAES([0., 0.], .4, popsize=POPULATION)
    rng = np.random.default_rng(seed)
    points, scores, states = [], [], []
    rejected = 0
    while len(points) + POPULATION <= budget and (can_start is None or can_start()):
        state.C.update_eigensystem(state.counteval, state.params.lazy_gap_evals)
        basis = np.asarray(state.C.eigenbasis)
        root = (basis * np.sqrt(state.C.eigenvalues)) @ basis.T
        population = []
        while len(population) < POPULATION:
            point = np.asarray(state.xmean) + state.sigma * (root @ (q @ rng.standard_normal(2)))
            outside = np.any(np.abs(point) > 1) if box else np.linalg.norm(point) > 1
            if outside:
                rejected += 1
                if rejected > 10000:
                    raise RuntimeError("CMA rejection cap")
                continue
            population.append(point.tolist())
        values = ([evaluate(p) for p in population] if evaluate_population is None
                  else evaluate_population(population))
        state.tell(population, values)
        points.extend(population)
        scores.extend(values)
        covariance = np.array([[state.C[i][j] for j in range(2)] for i in range(2)])
        eigvals, vectors = np.linalg.eigh(covariance)
        major_axis = vectors[:, -1]
        states.append(dict(call=state.counteval, ratio=float(eigvals[-1]/eigvals[0]),
                           axis_error=float(np.degrees(np.arccos(np.clip(abs(major_axis @ q[:, 0]), 0, 1)))),
                           sigma=state.sigma, covariance=covariance.tolist(), mean=state.xmean))
    assert state.counteval == len(points)
    if can_start is None:
        assert state.counteval == budget
    return points, scores, dict(states=states, rejections=rejected)


class Evaluator:
    def __init__(self, valley, sigma, search_noise, repeat_noise):
        self.valley, self.sigma = valley, sigma
        self.search_noise, self.repeat_noise = search_noise, repeat_noise
        self.search_count = self.repeat_count = 0
        self.trace = []

    def measure(self, point, repeat=False):
        if repeat:
            noise = self.repeat_noise[self.repeat_count]
            self.repeat_count += 1
        else:
            noise = self.search_noise[self.search_count]
            self.search_count += 1
        value = float(self.valley.clean(point) + self.sigma * noise)
        self.trace.append(dict(point=np.asarray(point).tolist(), observed=value, repeat=repeat))
        return value


def posterior_scores(points, values, theta, noise_sd, candidates):
    x = np.asarray(points)
    _, lower, alpha, _ = fit(x, np.asarray(values), theta, False, noise_sd=noise_sd)
    means, _ = predict(x, np.asarray(candidates), theta, lower, alpha)
    return means


def cutoff_cma(meter, q, seed, box=False):
    archived_sums, archived_counts = [], []

    def remaining():
        return BUDGET - meter.search_count - meter.repeat_count

    def score_population(population):
        sums = np.array([meter.measure(p) for p in population])
        counts = np.ones(POPULATION, dtype=int)
        if (meter.search_count > REPEAT_START and remaining() >= 2
                and meter.repeat_count + 2 <= MAX_CUTOFF_REPEATS):
            order = np.argsort(sums, kind="stable")
            boundary = order[POPULATION // 2 - 1:POPULATION // 2 + 1]
            if sums[boundary[1]] - sums[boundary[0]] <= np.sqrt(2) * meter.sigma:
                for idx in boundary:
                    sums[idx] += meter.measure(population[idx], repeat=True)
                    counts[idx] += 1
        archived_sums.extend(sums)
        archived_counts.extend(counts)
        return sums / counts

    points, update_scores, _ = run_coupled_cma(
        meter.measure, q, seed, can_start=lambda: remaining() >= POPULATION,
        evaluate_population=score_population, box=box,
    )
    assert len(points) >= 84
    update_repeats = meter.repeat_count
    top_two = np.argsort(np.asarray(archived_sums) / archived_counts, kind="stable")[:2]
    for i in range(remaining()):
        idx = top_two[i % len(top_two)]
        archived_sums[idx] += meter.measure(points[idx], repeat=True)
        archived_counts[idx] += 1
    scores = np.asarray(archived_sums) / archived_counts
    return dict(points=np.asarray(points), scores=scores, evaluator=meter,
                update_scores=np.asarray(update_scores), update_repeats=update_repeats)

### KG action restrictions and common BO prefix

In [ ]:
%%writefile -a master_code/mechanisms.py

def distinct_indices(points):
    return np.sort(np.unique(points, axis=0, return_index=True)[1])


def posterior_joint(x, candidates, theta, lower, alpha):
    cross = covariance(x, candidates, theta)
    solved = solve_triangular(lower, cross, lower=True, check_finite=False)
    cov = covariance(candidates, candidates, theta) - solved.T @ solved
    return cross.T @ alpha, (cov + cov.T) / 2


def knowledge_gradient_choice(x, y, theta, lower, alpha, old, new, noise_sd, allowed=None):
    candidates = np.vstack([old, new])
    means, cov = posterior_joint(x, candidates, theta, lower, alpha)
    noise_standardised = noise_sd / (np.std(y) or 1)
    observation_sd = np.sqrt(np.maximum(np.diag(cov), 0) + noise_standardised**2)
    slope = cov / observation_sd[None, :]
    n_old = len(old)
    fantasies = means[:n_old, None, None] + slope[:n_old, :, None] * KG_NODES
    best = fantasies.min(axis=0)
    if len(new):
        new_fantasy = means[n_old:, None] + np.diag(slope)[n_old:, None] * KG_NODES
        best[n_old:] = np.minimum(best[n_old:], new_fantasy)
    expected_best = best @ KG_WEIGHTS
    if allowed is None:
        allowed = np.arange(len(candidates))
    return int(allowed[np.argmin(expected_best[allowed])])



CMA, BO = "CMA-ES", "Isotropic BO"
STANDARD, CUTOFF = "Standard", "Cutoff repeats"
NATIVE, GP = "Lowest observed/averaged score", "GP reranking"
LCB, ALTERNATE = "LCB throughout", "Alternating 12 + 12"
KG, KG_NEW, KG_REPEAT = "KG unrestricted", "KG new only", "KG repeats only"
BO_POLICIES = (LCB, ALTERNATE, KG, KG_NEW, KG_REPEAT)
METRICS = ("recommendation", "best_visited", "selection_loss")


def bo_search(meter, q, initial, seed, box, policy, prefix=None):
    """Reuse the original LCB and KG mechanics with explicit measurement restrictions."""
    rng = np.random.default_rng(seed)
    theta = np.log([1., .4])
    if prefix is None:
        for point in initial:
            meter.measure(point)
    else:
        meter.trace = list(prefix["trace"])
        meter.search_count = SEARCH_PREFIX
        theta = prefix["theta"].copy()
        for _ in range(POPULATION, SEARCH_PREFIX):
            proposal_pool(rng, q, box)

    checkpoint = None
    decisions = []
    while len(meter.trace) < BUDGET:
        x = np.asarray([row["point"] for row in meter.trace])
        y = np.asarray([row["observed"] for row in meter.trace])
        t = len(y)
        theta, lower, alpha, _ = fit(x, y, theta, t % POPULATION == 0, noise_sd=meter.sigma)
        pool = proposal_pool(rng, q, box)
        old = x[distinct_indices(x)]
        if policy == ALTERNATE and (t - SEARCH_PREFIX) % 2 == 1:
            means, _ = predict(x, old, theta, lower, alpha)
            point = old[np.argmin(means)]
            repeated = True
        else:
            mean, sd = predict(x, pool, theta, lower, alpha)
            lcb = mean - 2 * sd
            refined, _, _ = refined_candidate(x, pool, theta, lower, alpha, lcb, box=box)
            seen = {tuple(point) for point in old}
            if policy in (KG, KG_NEW, KG_REPEAT):
                new = []
                # Retain the original 24 LCB-screened candidates for every KG branch.
                for candidate in (refined, *pool[np.argsort(lcb, kind="stable")[:KG_NEW_POINTS]]):
                    if tuple(candidate) not in seen and len(new) < KG_NEW_POINTS:
                        new.append(candidate)
                        seen.add(tuple(candidate))
                new = np.asarray(new).reshape(-1, 2)
                if policy == KG_NEW and not len(new):
                    raise RuntimeError("No distinct new KG measurement candidate.")
                allowed = {
                    KG: np.arange(len(old) + len(new)),
                    KG_NEW: np.arange(len(old), len(old) + len(new)),
                    KG_REPEAT: np.arange(len(old)),
                }[policy]
                chosen = knowledge_gradient_choice(
                    x, y, theta, lower, alpha, old, new, meter.sigma, allowed=allowed
                )
                repeated = chosen < len(old)
                point = old[chosen] if repeated else new[chosen - len(old)]
            else:
                # LCB's new-proposal arm must actually choose a distinct location.
                if tuple(refined) in seen:
                    available = [p for p in pool[np.argsort(lcb, kind="stable")]
                                 if tuple(p) not in seen]
                    if not available:
                        raise RuntimeError("No distinct LCB measurement candidate.")
                    refined = available[0]
                point, repeated = refined, False
        meter.measure(point, repeat=repeated)
        decisions.append(dict(call=len(meter.trace), repeat=bool(repeated), theta=theta.tolist()))
        if len(meter.trace) == SEARCH_PREFIX:
            checkpoint = dict(trace=list(meter.trace), theta=theta.copy())

    x = np.asarray([row["point"] for row in meter.trace])
    y = np.asarray([row["observed"] for row in meter.trace])
    candidates = x[distinct_indices(x)]
    scores = posterior_scores(x, y, theta, meter.sigma, candidates)
    return dict(points=candidates, scores=scores, evaluator=meter, prefix=checkpoint,
                decisions=decisions, theta=theta.tolist())

### Clean scoring and parent diagnostics

In [ ]:
%%writefile -a master_code/mechanisms.py

def score_recommendation(valley, candidates, chosen, **labels):
    clean = np.array([valley.clean(p) for p in candidates])
    recommended, best = float(clean[chosen]), float(clean.min())
    return dict(**labels, recommendation=recommended, best_visited=best,
                selection_loss=recommended-best)


def cma_recommendations(valley, trace, native_index, noise_sd, **labels):
    x = np.asarray([r["point"] for r in trace])
    y = np.asarray([r["observed"] for r in trace])
    proposals = np.asarray([r["point"] for r in trace if not r["repeat"]])
    candidates = x[distinct_indices(x)]
    native_point = proposals[native_index]
    native = int(np.flatnonzero(np.all(candidates == native_point, axis=1))[0])
    theta, lower, alpha, _ = fit(x, y, np.log([1, .4]), True, noise_sd=noise_sd)
    means, _ = predict(x, candidates, theta, lower, alpha)
    return [score_recommendation(valley, candidates, idx, **labels, readout=readout,
                                 search_calls=len(proposals), repeat_calls=len(x)-len(proposals))
            for readout, idx in ((NATIVE, native), (GP, int(np.argmin(means))))]


def parent_diagnostics(objective, points, update_scores, noise_sd, **labels):
    clean = np.array([objective.clean(p) for p in points])
    rows = []
    mu = POPULATION // 2
    for start in range(0, len(points), POPULATION):
        end = start + POPULATION
        order = np.argsort(clean[start:end], kind="stable")
        selected = np.argsort(update_scores[start:end], kind="stable")[:mu]
        replaced = len(set(selected) - set(order[:mu]))
        ordered_gaps = clean[start:end][order]
        rows.append(dict(**labels, generation=end // POPULATION,
                         parent_error=replaced / mu,
                         clean_margin=float(ordered_gaps[mu] - ordered_gaps[mu - 1]),
                         margin_over_noise=(float((ordered_gaps[mu] - ordered_gaps[mu - 1]) / noise_sd)
                                            if noise_sd > 0 else None)))
    return rows

### Prefix diagnostics and paired runs

In [ ]:
%%writefile -a master_code/mechanisms.py

def pair_error_probability(delta, noise_sd, repeats):
    """Probability for fresh independent means at two fixed, distinct locations."""
    if noise_sd == 0:
        return 0.5 if delta == 0 else 0.0
    return float(ndtr(-delta * np.sqrt(repeats) / (np.sqrt(2) * noise_sd)))


def prefix_diagnostics(objective, trace, noise_sd, scores, candidates, final_row, **labels):
    """Score after the optimiser finishes; clean values never guide its decisions."""
    clean = np.array([objective.clean(point) for point in candidates])
    chosen = int(np.argmin(scores))
    best = float(clean.min())
    recommendation = float(clean[chosen])
    shortlist = np.argsort(scores, kind="stable")[:4]
    best_finalist = float(clean[shortlist].min())
    ordered_finalists = np.sort(clean[shortlist])
    delta = float(ordered_finalists[1] - ordered_finalists[0])
    row = dict(
        **labels, prefix_calls=len(trace), distinct_points=len(candidates),
        prefix_best_visited=best, prefix_recommendation=recommendation,
        prefix_selection_loss=recommendation-best,
        shortlist_loss=best_finalist-best,
        leader_loss_within_shortlist=recommendation-best_finalist,
        shortlist_best_second_margin=delta,
        shortlist_margin_over_noise=delta/noise_sd if noise_sd > 0 else None,
        fresh_pair_error_1=pair_error_probability(delta, noise_sd, 1),
        fresh_pair_error_6=pair_error_probability(delta, noise_sd, 6),
        continued_recommendation=final_row["recommendation"],
        continued_best_visited=final_row["best_visited"],
        continued_selection_loss=final_row["selection_loss"],
        oracle_advantage=final_row["recommendation"]-best,
        oracle_can_strictly_improve=bool(best < final_row["recommendation"]),
        remaining_search_gain=best-final_row["best_visited"],
        selection_change=final_row["selection_loss"]-(recommendation-best),
    )
    assert row["remaining_search_gain"] >= -1e-12
    np.testing.assert_allclose(row["prefix_selection_loss"],
                               row["shortlist_loss"] + row["leader_loss_within_shortlist"])
    pairs = []
    for i in range(len(shortlist)):
        for j in range(i+1, len(shortlist)):
            gap = float(abs(clean[shortlist[i]] - clean[shortlist[j]]))
            pairs.append(dict(
                **labels, rank_a=i+1, rank_b=j+1, clean_difference=gap,
                difference_over_noise=gap/noise_sd if noise_sd > 0 else None,
                fresh_pair_error_1=pair_error_probability(gap, noise_sd, 1),
                fresh_pair_error_6=pair_error_probability(gap, noise_sd, 6),
            ))
    return row, pairs


def prefix_readouts(objective, trace, noise_sd, theta, final_rows, **labels):
    x = np.asarray([row["point"] for row in trace])
    y = np.asarray([row["observed"] for row in trace])
    unique = distinct_indices(x)
    candidates = x[unique]
    # Fit at the 72-call branch boundary, matching BO's next scheduled refit.
    fitted, lower, alpha, _ = fit(x, y, theta, True, noise_sd=noise_sd)
    posterior_mean, _ = predict(x, candidates, fitted, lower, alpha)
    rows, pair_rows = [], []
    for final_row in final_rows:
        scores = posterior_mean if final_row["readout"] == GP else y[unique]
        row, pairs = prefix_diagnostics(
            objective, trace, noise_sd, scores, candidates, final_row,
            **labels, readout=final_row["readout"],
        )
        rows.append(row)
        pair_rows.extend(pairs)
    return rows, pair_rows


def run_block(seed):
    centre = rotate((.4, 0), random.Random(seed).uniform(0, tau))
    # Stable instance mapping, independent of any subset of seeds requested later.
    instance = 201 + seed - 10100
    search_noise = np.random.default_rng(
        np.random.SeedSequence([seed, 40117])).standard_normal(BUDGET)
    repeat_noise = np.random.default_rng(
        np.random.SeedSequence([seed, 73093])).standard_normal(BUDGET)
    rows, parents, histories, scales, prefixes, pairs = [], [], [], [], [], []
    for spec in PROBLEM_SPECS:
        box = spec["function"] is not None
        if box:
            objective = BBOBGap(spec["function"], instance)
            q, scale = np.eye(2), objective.scale
            problem_id = str(objective.problem)
        else:
            objective = Valley(centre, math.radians(spec["angle"]), 100, spec["bend"])
            q, scale = rotation(spec["angle"]), float(objective.normaliser)
            problem_id = f"{spec['problem']}_k100_seed{seed}"
        scales.append(dict(seed=seed, **spec, problem_id=problem_id, gap_scale=scale))
        for sigma in NOISE:
            labels = dict(seed=seed, **spec, problem_id=problem_id, sigma=sigma)

            def meter():
                return Evaluator(objective, sigma, search_noise, repeat_noise)

            standard_meter = meter()
            points, scores, _ = run_coupled_cma(standard_meter.measure, q, seed+30000, box=box)
            standard_points, standard_scores = np.asarray(points), np.asarray(scores)
            cutoff = cutoff_cma(meter(), q, seed+30000, box=box)
            assert standard_meter.trace[:REPEAT_START] == cutoff["evaluator"].trace[:REPEAT_START]
            standard_rows = None
            for policy, evaluation, candidates, native_scores, update_scores in (
                (STANDARD, standard_meter, standard_points, standard_scores, standard_scores),
                (CUTOFF, cutoff["evaluator"], cutoff["points"], cutoff["scores"], cutoff["update_scores"]),
            ):
                assert len(evaluation.trace) == BUDGET
                policy_rows = cma_recommendations(
                    objective, evaluation.trace, int(np.argmin(native_scores)), sigma,
                    **labels, method=CMA, policy=policy,
                )
                rows.extend(policy_rows)
                if policy == STANDARD:
                    standard_rows = policy_rows
                parents.extend(parent_diagnostics(
                    objective, candidates, update_scores, sigma, **labels, policy=policy
                ))
                histories.append(dict(**labels, method=CMA, policy=policy, trace=evaluation.trace))
            pre, pair = prefix_readouts(
                objective, standard_meter.trace[:SEARCH_PREFIX], sigma, np.log([1., .4]),
                standard_rows, **labels, method=CMA, policy=STANDARD,
            )
            prefixes.extend(pre)
            pairs.extend(pair)

            initial = standard_points[:POPULATION]
            baseline = bo_search(meter(), q, initial, seed+20000, box, LCB)
            prefix = baseline["prefix"]
            assert len(prefix["trace"]) == SEARCH_PREFIX
            assert prefix["trace"][:POPULATION] == standard_meter.trace[:POPULATION]
            for policy in BO_POLICIES:
                run = baseline if policy == LCB else bo_search(
                    meter(), q, initial, seed+20000, box, policy, prefix
                )
                evaluator = run["evaluator"]
                assert len(evaluator.trace) == BUDGET
                assert evaluator.trace[:SEARCH_PREFIX] == prefix["trace"]
                if policy in (LCB, KG_NEW):
                    assert evaluator.repeat_count == 0
                    assert len(run["points"]) == BUDGET
                if policy == KG_REPEAT:
                    assert evaluator.repeat_count == BUDGET-SEARCH_PREFIX
                    np.testing.assert_array_equal(run["points"], np.asarray(
                        [row["point"] for row in prefix["trace"]]))
                if policy == ALTERNATE:
                    assert evaluator.repeat_count == 12
                chosen = int(np.argmin(run["scores"]))
                row = score_recommendation(
                    objective, run["points"], chosen, **labels, method=BO, policy=policy,
                    readout=GP, search_calls=evaluator.search_count, repeat_calls=evaluator.repeat_count,
                )
                rows.append(row)
                histories.append(dict(**labels, method=BO, policy=policy,
                                      trace=evaluator.trace, decisions=run["decisions"],
                                      final_theta=run["theta"]))
                if policy == LCB:
                    pre, pair = prefix_readouts(
                        objective, prefix["trace"], sigma, prefix["theta"], [row],
                        **labels, method=BO, policy=LCB,
                    )
                    prefixes.extend(pre)
                    pairs.extend(pair)
    return dict(results=rows, parents=parents, histories=histories, scales=scales,
                prefixes=prefixes, finalist_pairs=pairs)

### Resumable per-seed storage

In [ ]:
%%writefile -a master_code/mechanisms.py

def block_path(seed):
    return OUT / "blocks" / f"{seed}.json.gz"


def validate_block(block, seed):
    expected = len(PROBLEM_SPECS) * len(NOISE) * (4 + len(BO_POLICIES))
    assert len(block["results"]) == expected
    for row in block["results"]:
        assert row["seed"] == seed
        assert row["search_calls"] + row["repeat_calls"] == BUDGET
        np.testing.assert_allclose(row["recommendation"], row["best_visited"] + row["selection_loss"], atol=1e-12)
    assert len(block["prefixes"]) == len(PROBLEM_SPECS) * len(NOISE) * 3
    for history in block["histories"]:
        assert len(history["trace"]) == BUDGET


def save_blocks():
    (OUT / "blocks").mkdir(parents=True, exist_ok=True)
    manifest = OUT / "settings.json"
    if manifest.exists() and json.loads(manifest.read_text(encoding="utf-8")) != SETTINGS:
        raise ValueError("Cached configuration, software, or source differs. Use a new output root.")
    manifest.write_text(json.dumps(SETTINGS, indent=2), encoding="utf-8")
    for i, seed in enumerate(SEEDS, 1):
        path = block_path(seed)
        if path.exists():
            with gzip.open(path, "rt", encoding="utf-8") as stream:
                validate_block(json.load(stream), seed)
            print(f"[{i}/{len(SEEDS)}] Reusing validated block {seed}", flush=True)
            continue
        start = perf_counter()
        block = run_block(seed)
        validate_block(block, seed)
        temporary = path.with_suffix(".tmp")
        with gzip.open(temporary, "wt", encoding="utf-8") as stream:
            json.dump(block, stream, allow_nan=False, separators=(",", ":"))
        temporary.replace(path)
        print(f"[{i}/{len(SEEDS)}] Saved block {seed} ({perf_counter()-start:.1f} s)", flush=True)


save_blocks()

### Paired decomposition and summaries

In [ ]:
%%writefile -a master_code/mechanisms.py

CONDITIONS = ["scope", "problem", "sigma", "method"]
COMPARISONS = {
    CMA: [
        ((STANDARD, GP), (STANDARD, NATIVE), "GP vs native, standard CMA"),
        ((CUTOFF, GP), (CUTOFF, NATIVE), "GP vs native, cutoff CMA"),
        ((CUTOFF, GP), (STANDARD, GP), "Cutoff vs standard CMA, both GP"),
        ((CUTOFF, NATIVE), (STANDARD, NATIVE), "Cutoff vs standard CMA, both native"),
    ],
    BO: [
        ((ALTERNATE, GP), (LCB, GP), "Alternation vs LCB"),
        ((KG, GP), (LCB, GP), "KG unrestricted vs LCB"),
        ((KG_NEW, GP), (LCB, GP), "KG new only vs LCB"),
        ((KG_REPEAT, GP), (LCB, GP), "KG repeats only vs LCB"),
        ((KG, GP), (KG_NEW, GP), "KG unrestricted vs new only"),
        ((KG, GP), (KG_REPEAT, GP), "KG unrestricted vs repeats only"),
        ((KG, GP), (ALTERNATE, GP), "KG unrestricted vs alternation"),
    ],
}


def paired_effects(results):
    contrasts, paired_rows = [], []
    for key, group in results.groupby(CONDITIONS, sort=False):
        labels = dict(zip(CONDITIONS, key))
        groups = {(policy, readout): table.set_index("seed").sort_index()
                  for (policy, readout), table in group.groupby(["policy", "readout"])}
        n = len(next(iter(groups.values())))
        # Two-seed smoke runs test computation; they do not receive inferential intervals.
        indices = (np.random.default_rng(4670).integers(0, n, (10000, n))
                   if PROFILE == "full" else None)
        for left, right, name in COMPARISONS[labels["method"]]:
            a, b = groups[left], groups[right]
            assert a.index.is_unique and a.index.equals(b.index)
            raw = a[list(METRICS)].to_numpy() - b[list(METRICS)].to_numpy()
            np.testing.assert_allclose(raw[:, 0], raw[:, 1]+raw[:, 2], atol=1e-12)
            log_ratio = np.log10(np.maximum(a.recommendation.to_numpy(), LOG_FLOOR)) - np.log10(
                np.maximum(b.recommendation.to_numpy(), LOG_FLOOR))
            primary = (labels["problem"] == "Curved 45 deg" and labels["sigma"] == 1e-2
                       and name == "KG unrestricted vs new only")
            for i, seed in enumerate(a.index):
                paired_rows.append(dict(**labels, contrast=name, seed=int(seed),
                                        primary=primary, delta_recommendation=raw[i, 0],
                                        delta_best_visited=raw[i, 1], delta_selection_loss=raw[i, 2],
                                        log10_recommendation_ratio=log_ratio[i]))
            for metric, scale, delta in [
                *[(metric, "raw", raw[:, j]) for j, metric in enumerate(METRICS)],
                ("recommendation", "log10", log_ratio),
            ]:
                bounds = (np.quantile(delta[indices].mean(axis=1), [.025, .975])
                          if indices is not None else [np.nan, np.nan])
                contrasts.append(dict(**labels, contrast=name, metric=metric, scale=scale,
                                      primary=primary, n=n, mean=delta.mean(),
                                      low_95=bounds[0], high_95=bounds[1],
                                      fraction_improved=np.mean(delta < 0)))
    return pd.DataFrame(contrasts), pd.DataFrame(paired_rows)


tables = {name: [] for name in ("results", "parents", "scales", "prefixes", "finalist_pairs")}
for seed in SEEDS:
    with gzip.open(block_path(seed), "rt", encoding="utf-8") as stream:
        block = json.load(stream)
    validate_block(block, seed)
    for name in tables:
        tables[name].extend(block[name])
results, parents, scales, prefixes, finalist_pairs = [pd.DataFrame(tables[name]) for name in tables]
assert not results.duplicated(["seed", *CONDITIONS, "policy", "readout"]).any()
same_history = results[results.method == CMA].pivot(
    index=["seed", *CONDITIONS, "policy"], columns="readout", values="best_visited")
np.testing.assert_array_equal(same_history[NATIVE], same_history[GP])
np.testing.assert_allclose(results.recommendation,
                           results.best_visited + results.selection_loss, atol=1e-12)
effects, paired = paired_effects(results)
for filename, table in {
    "run_gaps.csv": results,
    "cma_parent_diagnostics.csv": parents,
    "instance_normalisers.csv": scales,
    "prefix_diagnostics.csv": prefixes,
    "prefix_finalist_pairs.csv": finalist_pairs,
    "paired_effects.csv": effects,
    "paired_run_differences.csv": paired,
}.items():
    table.to_csv(OUT / filename, index=False)

means = results.groupby([*CONDITIONS, "policy", "readout"], as_index=False)[
    [*METRICS, "search_calls", "repeat_calls"]].mean()
np.testing.assert_allclose(means.recommendation, means.best_visited + means.selection_loss, atol=1e-12)
means.to_csv(OUT / "mean_gaps_and_allocation.csv", index=False)
prefix_summary = prefixes.groupby([*CONDITIONS, "readout"], as_index=False)[[
    "prefix_best_visited", "prefix_selection_loss", "shortlist_loss",
    "oracle_advantage", "oracle_can_strictly_improve", "remaining_search_gain",
    "selection_change", "shortlist_margin_over_noise", "fresh_pair_error_1", "fresh_pair_error_6",
]].mean()
prefix_summary = prefix_summary.rename(columns={"oracle_can_strictly_improve": "oracle_opportunity_fraction"})
prefix_summary.to_csv(OUT / "prefix_summary.csv", index=False)
ratio = effects[effects.scale == "log10"].copy()
ratio[["ratio", "lower", "upper"]] = 10.0 ** ratio[["mean", "low_95", "high_95"]]
ratio.to_csv(OUT / "recommendation_ratios.csv", index=False)
primary_panel = ratio[(ratio.problem == "Curved 45 deg") & (ratio.sigma == 1e-2)]
display(primary_panel[["method", "contrast", "n", "primary", "ratio", "lower", "upper"]])
display(prefix_summary[(prefix_summary.problem == "Curved 45 deg") & (prefix_summary.sigma == 1e-2)])
if PROFILE == "smoke":
    print("SMOKE CHECK ONLY. Numerical values are not report findings; intervals are suppressed.")

### Figures separating the main valleys from supporting transfer

In [ ]:
%%writefile -a master_code/mechanisms.py

PLOT_CONTRASTS = [
    "GP vs native, standard CMA", "Cutoff vs standard CMA, both GP",
    "KG unrestricted vs LCB", "KG new only vs LCB", "KG repeats only vs LCB",
    "KG unrestricted vs new only", "KG unrestricted vs repeats only",
]
COMPONENTS = [
    ("best_visited", "Search component", "#0072b2", -.20),
    ("selection_loss", "Selection component", "#d55e00", 0.),
    ("recommendation", "Total effect", "#222222", .20),
]


def save_decomposition(scope, sigma):
    names = [spec["problem"] for spec in PROBLEM_SPECS if spec["scope"] == scope]
    ncols = 2
    nrows = math.ceil(len(names)/ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(16, 5.4*nrows), squeeze=False,
                             layout="constrained")
    for ax, problem in zip(axes.flat, names):
        selected = effects[(effects.problem == problem) & (effects.sigma == sigma)
                           & (effects.scale == "raw")].set_index(["contrast", "metric"])
        for metric, label, color, offset in COMPONENTS:
            for i, contrast in enumerate(PLOT_CONTRASTS):
                row = selected.loc[(contrast, metric)]
                y = i + offset
                ax.plot(row["mean"], y, "o", color=color, ms=4, label=label if i == 0 else None)
                if np.isfinite(row.low_95):
                    ax.hlines(y, row.low_95, row.high_95, color=color, lw=1.5)
        ax.axvline(0, color=".55", lw=1)
        ax.set_yticks(range(len(PLOT_CONTRASTS)), PLOT_CONTRASTS, fontsize=8)
        ax.invert_yaxis()
        ax.set_title(problem)
        ax.set_xlabel("Mean paired change in normalised gap (negative favours first policy)")
        ax.ticklabel_format(axis="x", style="sci", scilimits=(-2, 2))
        ax.grid(axis="x", alpha=.15)
    for ax in list(axes.flat)[len(names):]:
        ax.set_visible(False)
    axes.flat[0].legend(fontsize=8, loc="best")
    fig.suptitle(f"{scope}: search and selection effects at 96 calls | noise SD {sigma:g}\n"
                 + ("Pointwise 95% bootstrap intervals" if PROFILE == "full" else "Smoke check only"))
    suffix = "main" if scope == "Main valleys" else "supporting"
    fig.savefig(OUT / f"decomposition-{suffix}-sigma-{sigma:g}.png", dpi=180, bbox_inches="tight")
    plt.show()
    plt.close(fig)


for scope in results.scope.unique():
    for sigma in NOISE:
        save_decomposition(scope, sigma)

parent_summary = parents.groupby(["scope", "problem", "sigma", "policy", "generation"],
                                 as_index=False).agg(
    parent_error=("parent_error", "mean"),
    median_margin_over_noise=("margin_over_noise", "median"),
    blocks=("seed", "nunique"),
)
parent_summary.to_csv(OUT / "cma_parent_summary.csv", index=False)
fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
for policy, color in [(STANDARD, ".3"), (CUTOFF, "#d55e00")]:
    panel = parent_summary[(parent_summary.problem == "Curved 45 deg")
                           & (parent_summary.sigma == 1e-2)
                           & (parent_summary.policy == policy)
                           & (parent_summary.generation <= 14)]
    assert panel.blocks.eq(len(SEEDS)).all()
    axes[0].plot(panel.generation, panel.parent_error, marker="o", color=color, label=policy)
    axes[1].plot(panel.generation, panel.median_margin_over_noise, marker="o", color=color)
axes[0].set(xlabel="Generation (matched across policies)", ylabel="Mean parent replacement fraction",
            ylim=(-.02, 1.02))
axes[1].set(xlabel="Generation (matched across policies)", ylabel="Median clean third-fourth margin / noise SD")
axes[0].legend()
fig.suptitle("Rotated curved valley, noise SD 0.01: ranking and local noise difficulty")
fig.savefig(OUT / "cma-ranking-and-noise.png", dpi=180, bbox_inches="tight")
plt.show()
plt.close(fig)

print(f"Results, complete histories, and diagnostics saved under {OUT}")

In [ ]:
run_stage("mechanisms")

## Stage C follow-up - shortlist size and verification

The cross-valley table counts where the best of all 72 points and the best of
the four highest-ranked finalists could each beat continued LCB at call 96.
Counts are per valley, with seed blocks shared across conditions.

The verification experiment holds each saved search history fixed and allocates
24 fresh calls as 4×6, 8×3, or 12×2. Shortlists are nested in order of the
72-call GP posterior mean. Both readouts choose from the same shortlist.
Fresh means use the verification observations alone. The updated GP also uses
the search history, with the 72-call fit held fixed and a Gaussian update using
the fresh sample means. At zero noise the measured values are exact and both
rules return the best finalist.

Fresh measurement noise is simulated under the benchmark's known observation
model. The full profile averages 4,096 draws per seed and condition, sharing
draws across sizes and readouts. The clean objective generates the benchmark
observations and scores recommendations. Selection uses only noisy observations
and the fitted model. Bootstrap intervals resample the 24 search seed blocks
and are pointwise. The result compares expected performance conditional on the
saved histories, with no additional search trajectories.

In [ ]:
%%writefile master_code/shortlist.py
from pathlib import Path
import os

MASTER_PROFILE = globals().get("MASTER_PROFILE", os.environ.get("OPT_PROFILE", "smoke"))
MASTER_ROOT = Path(globals().get("MASTER_ROOT", os.environ.get("OPT_RESULTS_ROOT", "optimisation_results"))).resolve()
MASTER_INCLUDE_TRANSFER = globals().get("MASTER_INCLUDE_TRANSFER", os.environ.get("OPT_INCLUDE_TRANSFER", "1") == "1")
if MASTER_PROFILE not in ("full", "smoke"):
    raise ValueError("Choose full or smoke.")

### Reuse the original valley and GP definitions

In [ ]:
%%writefile -a master_code/shortlist.py

import gzip
import hashlib
import json
import math
import random
import zipfile
from dataclasses import dataclass
from math import cos, hypot, sin, tau

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.linalg import cho_solve, cholesky, solve_triangular
from scipy.optimize import minimize
from scipy.spatial.distance import cdist


JITTER = 1e-7

def rotate(point, angle):
    x, y = point
    return cos(angle) * x - sin(angle) * y, sin(angle) * x + cos(angle) * y

@dataclass(frozen=True)
class Valley:
    centre: tuple
    angle: float
    curvature_ratio: float = 100.0
    bend: float = 0.0

    @property
    def normaliser(self):
        # Under uniform disk sampling E[X^2] = 1/4 and E[X^4] = 1/8.
        # Terms odd in either coordinate integrate to zero.
        a, b = self.centre
        if self.bend == 0:
            k = self.curvature_ratio
            return (1 + k) / 4 + a*a + k*b*b
        eu2 = .25 + a*a
        ev2 = .25 + b*b
        eu4 = .125 + 1.5*a*a + a**4
        return eu2 + self.curvature_ratio * (
            ev2 + 2*self.bend*b*eu2 + self.bend**2*eu4
        )

    def canonical_gap(self, x, y):
        a, b = self.centre
        u, v = x - a, y - b
        return (u**2 + self.curvature_ratio*(v - self.bend*u**2)**2) / self.normaliser

    def clean(self, point):
        if hypot(*point) > 1.0 + 1e-12:
            raise ValueError("Point outside the unit disk")
        x, y = rotate(point, -self.angle)
        return self.canonical_gap(x, y)

    def observe(self, point, sigma, rng):
        return self.clean(point) + sigma * rng.gauss(0.0, 1.0)

def covariance(x, y, theta):
    lengths = np.exp(theta[1:])
    return np.exp(theta[0] - .5 * cdist(x / lengths, y / lengths, "sqeuclidean"))

def fit(x, values, theta, refit, noise_sd=0.0, jitter=JITTER):
    output_scale = values.std() or 1
    y = (values - values.mean()) / output_scale
    ident = np.eye(len(x))

    def factor(parameters):
        signal = covariance(x, x, parameters)
        lower = cholesky(signal + (jitter + (noise_sd/output_scale)**2) * ident, lower=True)
        return signal, lower, cho_solve((lower, True), y)

    def likelihood(parameters):
        signal, lower, alpha = factor(parameters)
        loss = .5 * (y @ alpha) + np.log(np.diag(lower)).sum()
        weights = .5 * (cho_solve((lower, True), ident) - np.outer(alpha, alpha)) * signal
        gradient = [weights.sum()]
        if len(parameters) == 2:
            gradient.append((weights * cdist(x, x, "sqeuclidean")).sum() / np.exp(2*parameters[1]))
        else:
            for axis, log_length in enumerate(parameters[1:]):
                diff = x[:, axis, None] - x[:, axis]
                gradient.append((weights * diff**2).sum() / np.exp(2*log_length))
        return loss, np.array(gradient)

    status = -1
    if refit:
        bounds = [(-6, 6)] + [(math.log(.02), math.log(2))] * (len(theta)-1)
        result = minimize(likelihood, theta, jac=True, bounds=bounds, method="L-BFGS-B",
                          options={"maxiter": 50})
        theta, status = result.x, int(result.status)
    _, lower, alpha = factor(theta)
    return theta, lower, alpha, status

def predict(x, candidates, theta, lower, alpha):
    cross = covariance(x, candidates, theta)
    projected = solve_triangular(lower, cross, lower=True, check_finite=False)
    variance = np.maximum(0, np.exp(theta[0]) - np.einsum("ij,ij->j", projected, projected))
    return cross.T @ alpha, np.sqrt(variance)

### Verification simulation, paired intervals and output

In [ ]:
%%writefile -a master_code/shortlist.py

SHORTLIST_SIZES = (4, 8, 12)
METHOD = "Isotropic BO"
BASELINE = "LCB throughout"
PREFIX, BUDGET = 72, 96


def simulate_shortlists(clean, mu, cov, scale, sigma, noise, baseline, labels):
    """Both readouts use the same noisy measurements and choose within the shortlist."""
    rows = []
    for size in SHORTLIST_SIZES:
        repeats = (BUDGET - PREFIX) // size
        assert size * repeats == BUDGET - PREFIX
        gaps = clean[:size]
        fresh = gaps + sigma * noise[:, :size, :repeats].mean(axis=2)
        # mu is in original gap units. cov is in standardised units.
        # Condition the prefix GP on independent sample means with variance sigma²/r.
        if sigma > 0:
            prior_cov = cov[:size, :size]
            observation_cov = prior_cov + ((sigma / scale) ** 2 / repeats) * np.eye(size)
            factor = cholesky(observation_cov, lower=True)
            weights = cho_solve((factor, True), prior_cov)
            gp_scores = mu[:size] + (fresh - mu[:size]) @ weights
        else:
            gp_scores = fresh  # the clean repeat directly reveals every finalist's value
        for rule, scores in (("Fresh means", fresh), ("Updated GP", gp_scores)):
            chosen = gaps[scores.argmin(axis=1)]
            oracle_gap = float(gaps.min())
            rows.append(dict(**labels, shortlist_size=size, repeats_per_finalist=repeats,
                             rule=rule, prefix_best=float(clean.min()), shortlist_best=oracle_gap,
                             shortlist_loss=oracle_gap - float(clean.min()),
                             finalist_ranking_loss=float((chosen - oracle_gap).mean()),
                             recommendation=float(chosen.mean()),
                             delta_vs_lcb=float((chosen - baseline).mean()),
                             probability_beating_lcb=float((chosen < baseline).mean()),
                             mc_se_recommendation=float(chosen.std(ddof=1) / np.sqrt(len(chosen)))))
    return rows


def analyse_blocks(settings, read, draws_per_block):
    oracle_rows, simulations = [], []
    for seed in settings["seeds"]:
        block = json.loads(gzip.decompress(read(f"blocks/{seed}.json.gz")))
        prefixes = {(row["problem"], row["sigma"]): row for row in block["prefixes"]
                    if row["method"] == METHOD and row["scope"] == "Main valleys"}
        if len(prefixes) != 4 * len(settings["noise"]):
            raise ValueError(f"Seed {seed} is missing main-valley prefix records.")
        noise = np.random.default_rng(np.random.SeedSequence([seed, 58101])).standard_normal(
            (draws_per_block, max(SHORTLIST_SIZES), 6))
        centre = rotate((.4, 0), random.Random(seed).uniform(0, tau))
        histories = [h for h in block["histories"]
                     if h["method"] == METHOD and h["policy"] == BASELINE
                     and h["scope"] == "Main valleys"]
        if len(histories) != len(prefixes):
            raise ValueError(f"Seed {seed} is missing continued LCB histories.")
        for h in histories:
            problem, sigma = h["problem"], h["sigma"]
            diagnostic = prefixes[problem, sigma]
            assert h["seed"] == diagnostic["seed"] == seed and len(h["trace"]) == BUDGET
            labels = dict(seed=seed, problem=problem, sigma=sigma)
            baseline = diagnostic["continued_recommendation"]
            trace = h["trace"][:PREFIX]
            x = np.array([r["point"] for r in trace])
            y = np.array([r["observed"] for r in trace])
            # The policy records the fitted hyperparameters used to choose call 73.
            # Using them reproduces the GP at the 72-call branch boundary.
            theta = np.array(next(d["theta"] for d in h["decisions"] if d["call"] == PREFIX + 1))
            _, lower, alpha, _ = fit(x, y, theta, False, noise_sd=sigma, jitter=settings["jitter"])
            assert len(np.unique(x, axis=0)) == PREFIX
            scale = y.std() or 1.
            posterior, _ = predict(x, x, theta, lower, alpha)
            order = np.argsort(posterior, kind="stable")
            objective = Valley(centre, math.radians(h["angle"]), 100, h["bend"])
            clean = np.array([objective.clean(p) for p in x[order]])
            np.testing.assert_allclose(clean.min(), diagnostic["prefix_best_visited"], atol=1e-12, rtol=1e-8)
            # A replay can break numerical near ties differently on another platform.
            # Keep the original oracle table separate from the reconstructed shortlists.
            oracle_rows.append(dict(**labels, continued_recommendation=baseline,
                                    prefix_best=diagnostic["prefix_best_visited"],
                                    selection_loss=diagnostic["prefix_selection_loss"],
                                    shortlist_loss=diagnostic["shortlist_loss"],
                                    all_points_can_beat_lcb=diagnostic["oracle_can_strictly_improve"],
                                    top_four_can_beat_lcb=diagnostic["oracle_advantage"] > diagnostic["shortlist_loss"],
                                    replay_top_four_loss_delta=clean[:4].min() - clean.min() - diagnostic["shortlist_loss"],
                                    replay_leader_loss_delta=clean[0] - clean.min() - diagnostic["prefix_selection_loss"]))
            candidates = x[order[:max(SHORTLIST_SIZES)]]
            projected = solve_triangular(lower, covariance(x, candidates, theta), lower=True,
                                         check_finite=False)
            cov = covariance(candidates, candidates, theta) - projected.T @ projected
            cov = (cov + cov.T) / 2
            simulations.extend(simulate_shortlists(clean, posterior[order] * scale + y.mean(),
                                                   cov, scale, sigma, noise, baseline, labels))
    return pd.DataFrame(oracle_rows), pd.DataFrame(simulations)


def paired_summary(results, seeds):
    """Average within blocks first. Monte Carlo draws are not independent search runs."""
    bootstrap = np.random.default_rng(58102).integers(0, len(seeds), (10000, len(seeds)))
    rows = []
    for (problem, sigma), group in results.groupby(["problem", "sigma"], sort=True):
        by_policy = group.pivot(index="seed", columns=["rule", "shortlist_size"],
                                values="recommendation").reindex(seeds)
        assert not by_policy.isna().any().any()
        comparisons = []
        for rule in ("Fresh means", "Updated GP"):
            for size in SHORTLIST_SIZES:
                comparisons.append((f"{rule}, K={size} vs LCB",
                                    group[(group.rule == rule) & (group.shortlist_size == size)]
                                    .set_index("seed").reindex(seeds).delta_vs_lcb.to_numpy()))
                if size != 4:
                    comparisons.append((f"{rule}, K={size} vs K=4",
                                        (by_policy[rule, size] - by_policy[rule, 4]).to_numpy()))
        for size in SHORTLIST_SIZES:
            comparisons.append((f"Updated GP vs fresh means, K={size}",
                                (by_policy["Updated GP", size] - by_policy["Fresh means", size]).to_numpy()))
        for label, delta in comparisons:
            lo, hi = np.quantile(delta[bootstrap].mean(axis=1), [.025, .975]) if len(seeds) > 2 else (np.nan, np.nan)
            rows.append(dict(problem=problem, sigma=sigma, contrast=label, n=len(seeds),
                             mean_delta=delta.mean(), low_95=lo, high_95=hi))
    return pd.DataFrame(rows)


def save_results(oracle, results, settings, output):
    summary = oracle.groupby(["problem", "sigma"], as_index=False).agg(
        blocks=("seed", "size"), all_points_wins=("all_points_can_beat_lcb", "sum"),
        top_four_wins=("top_four_can_beat_lcb", "sum"),
        mean_shortlist_loss=("shortlist_loss", "mean"), mean_selection_loss=("selection_loss", "mean"))
    summary["share_of_mean_selection_loss"] = summary.mean_shortlist_loss / summary.mean_selection_loss
    metrics = ["prefix_best", "shortlist_best", "shortlist_loss", "finalist_ranking_loss",
               "recommendation", "delta_vs_lcb", "probability_beating_lcb", "mc_se_recommendation"]
    grouped = results.groupby(["problem", "sigma", "shortlist_size", "rule"], as_index=False)[metrics].mean()
    np.testing.assert_allclose(grouped.recommendation,
                               grouped.prefix_best + grouped.shortlist_loss + grouped.finalist_ranking_loss)
    output.mkdir(parents=True, exist_ok=True)
    for name, table in (("oracle_by_seed", oracle), ("cross_valley_oracle", summary),
                        ("verification_by_seed", results), ("verification_summary", grouped),
                        ("verification_paired_effects", paired_summary(results, settings["seeds"]))):
        table.to_csv(output / f"{name}.csv", index=False)

    high_noise = grouped[grouped.sigma == .01]
    display(summary[summary.sigma == .01])
    display(high_noise[["problem", "shortlist_size", "rule", "shortlist_loss",
                        "finalist_ranking_loss", "delta_vs_lcb", "probability_beating_lcb"]])
    fig, axes = plt.subplots(2, 2, figsize=(10, 6), layout="constrained")
    for ax, (problem, panel) in zip(axes.flat, high_noise.groupby("problem")):
        for rule, curve in panel.groupby("rule"):
            ax.plot(curve.shortlist_size, curve.delta_vs_lcb, marker="o", label=rule)
        shortlist = panel[panel.rule == "Fresh means"]
        ax.plot(shortlist.shortlist_size,
                shortlist.delta_vs_lcb - shortlist.finalist_ranking_loss,
                marker="o", linestyle=":", label="Shortlist oracle")
        ax.axhline(0, color="0.5", linewidth=1)
        ax.set(title=problem, xlabel="Shortlist size", ylabel="Mean gap change vs LCB",
               xticks=SHORTLIST_SIZES)
    axes[0, 0].legend()
    fig.suptitle("72 search calls + 24 verification calls | noise SD 0.01")
    fig.savefig(output / "verification_high_noise.png", dpi=160)
    plt.show()
    plt.close(fig)


def run(read, source_label):
    settings = json.loads(read("settings.json"))
    source_hash = hashlib.sha256(read("experiment.py")).hexdigest()
    expected_seeds = list(range(10100, 10124 if MASTER_PROFILE == "full" else 10102))
    if (settings["profile"] != MASTER_PROFILE or settings["seeds"] != expected_seeds
            or settings["prefix"] != PREFIX or settings["budget"] != BUDGET
            or source_hash != settings["source_sha256"]):
        raise ValueError("Use a complete Stage C mechanisms archive for the selected profile.")
    draws_per_block = 4096 if MASTER_PROFILE == "full" else 128
    output = MASTER_ROOT / "shortlist" / MASTER_PROFILE
    oracle, simulations = analyse_blocks(settings, read, draws_per_block)
    changed = oracle[np.abs(oracle.replay_top_four_loss_delta) > 1e-12]
    print(f"Reconstructed shortlist oracle differed from the saved one in {len(changed)} "
          f"of {len(oracle)} seed conditions.")
    if len(changed):
        display(changed[["seed", "problem", "sigma", "replay_top_four_loss_delta"]])
    save_results(oracle, simulations, settings, output)
    (output / "analysis_settings.json").write_text(json.dumps(
        dict(source=source_label, source_sha256=source_hash, seeds=settings["seeds"],
             shortlist_sizes=SHORTLIST_SIZES, simulations_per_seed_condition=draws_per_block,
             prefix_calls=PREFIX, total_calls=BUDGET), indent=2), encoding="utf-8")
    print("Saved shortlist results:", output)


source = MASTER_ROOT / "stage_c_mechanisms" / MASTER_PROFILE
archive = Path(globals().get("MASTER_SHORTLIST_ARCHIVE", "optimisation_results_full.zip"))
if (source / "settings.json").is_file():
    run(lambda name: (source / name).read_bytes(), str(source))
elif archive.is_file():
    with zipfile.ZipFile(archive) as bundle:
        base = f"results/stage_c_mechanisms/{MASTER_PROFILE}/"
        run(lambda name: bundle.read(base + name), str(archive))
else:
    raise FileNotFoundError(f"Place the completed results ZIP at {archive}, or run the mechanisms stage.")

In [ ]:
run_stage("shortlist")

## Export and next report revision

The archive contains every generated stage script and all results currently present
under the chosen output root. Completed stages can be rerun to regenerate summaries
from cached seed blocks. Changed source or settings require a new output root.
Each executed stage saves its exact source as `experiment.py` beside its provenance.
Keep both the ZIP and a downloaded copy of this executed notebook.

Use the `stage_c_mechanisms/full/` files for the next review. Start with
`paired_effects.csv`, `prefix_diagnostics.csv`, `prefix_summary.csv`, and
`cma_parent_diagnostics.csv`. The `blocks/` directory retains every noisy history.
Keep main-valley results central and report supporting transfer separately.

The later report revision will address direct effect differences, practical effect
sizes, multiple comparisons, literature positioning, and wording of the cutoff
heuristic. The repository still needs the reviewed report source, bibliography,
figures, author details, and its actual repository URL before submission.

In [ ]:
SOURCE_PROVENANCE = {'baseline': {'original': 'stage_a_baseline.ipynb', 'sha256': '67bafd9b17fda26956b65d0c7cd011a2bad2d077f5c2f8d6b371622ff2e81a57'}, 'geometry_pilot': {'original': 'stage-a-geometry.ipynb', 'sha256': '78b5535b19b92e42278396a9e2542328117ba017e01150e576e075c20d883500'}, 'stage_a': {'original': 'stage-a-straight-and-curved.ipynb', 'sha256': 'ef16117cc2083d774b6a67a8ab5d226fa44a06da5a0272cc196146b5cb6468f2'}, 'stage_b': {'original': 'stage_b_noise.ipynb', 'sha256': 'be6816ca24597aeb2829a4fec5a8db27720e578ed861ef7862c6d2505bcaa82f'}, 'stage_b_budget': {'original': 'stage_b_budget_extension.ipynb', 'sha256': '416eb5698d095f5889b57685b10e958959c5be0d18fbe3e7a6b0910732b8e305'}, 'stage_c_fixed': {'original': 'stage_c_replication.ipynb', 'sha256': '9953c04e5773cb54aae8d476891fa4fbbc7d95ee24efd50d60a82ad14221d941'}, 'stage_c_dynamic': {'original': 'stage_c_dynamic_evaluation.ipynb', 'sha256': '7927f6382706a5e5c369608b40396f3b7f82e5e9ac5ed03fd6bb51c8066caade'}, 'stage_c_final': {'original': 'stage_c_final_checks.ipynb', 'sha256': '55bd5c87aad1554a8b6797ff912e5b23bf58954931b60f21951a5e735ff5a309'}, 'transfer_original': {'original': 'noisy_policy_transfer.ipynb', 'sha256': '3996e4604209a5a92ceed0d363cc5d07e1a32f8209d1ab7ef480af9ec79e7d74'}, 'mechanisms': {'reused_from': 'noisy_policy_transfer.ipynb', 'additions': ['four main valley geometries', 'fresh seeds', 'zero-noise control', 'KG measurement restrictions', 'prefix oracle', 'local noise diagnostics', 'raw paired decomposition', 'source-validated resume']}, 'shortlist': {'reused_from': 'mechanisms', 'additions': ['cross-valley oracle counts', 'fixed-budget verification simulation']}}

manifest = dict(created_utc=datetime.now(timezone.utc).isoformat(), profile=PROFILE,
                run_mode=RUN_MODE, requested=RUN_STAGES, status=RUN_STATUS,
                supporting_transfer=INCLUDE_SUPPORTING_TRANSFER,
                python=sys.version.split()[0], packages=VERSIONS, sources=SOURCE_PROVENANCE)
(RESULTS_ROOT / "master_run_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
archive = RESULTS_ROOT.parent / f"{RESULTS_ROOT.name}_{PROFILE}.zip"
if archive.resolve() == SHORTLIST_ARCHIVE.resolve():
    archive = archive.with_name(f"{archive.stem}_updated.zip")
temporary = archive.with_suffix(".tmp")
with zipfile.ZipFile(temporary, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(RESULTS_ROOT.rglob("*")):
        if path.is_file() and path.suffix != ".tmp":
            bundle.write(path, Path("results") / path.relative_to(RESULTS_ROOT))
    for path in sorted(Path("master_code").glob("*.py")):
        bundle.write(path, Path("code") / path.name)
    notebook = Path("noisy_optimisation_master_shortlist.ipynb")
    if notebook.exists():
        bundle.write(notebook, notebook.name)
    requirements = "\n".join(f"{name}=={version}" for name, version in VERSIONS.items()) + "\n"
    bundle.writestr("requirements.txt", requirements)
    bundle.writestr("readme.md", "# Noisy optimisation experiment outputs\n\n"
                    "Open noisy_optimisation_master.ipynb and run its configuration and source cells.\n"
                    "Choose shortlist, new, report, everything, or custom, and select full or smoke.\n"
                    "Scripts may also run independently with OPT_PROFILE and OPT_RESULTS_ROOT set.\n"
                    "Use the recorded package versions in requirements.txt. Python version is in the manifest.\n"
                    "All experiment inputs are synthetic. BBOB is provided by coco-experiment.\n"
                    "The full new experiment uses 24 fresh seed blocks. Smoke outputs are workflow checks.\n"
                    "Download the executed notebook separately if it is absent from this archive.\n")
temporary.replace(archive)
print("Saved results archive:", archive)
print("Also save a copy of the executed notebook from the notebook File menu.")

In [ ]:
# Set True in Colab to trigger the browser download after Run all.
DOWNLOAD_ARCHIVE = False
if DOWNLOAD_ARCHIVE:
    from google.colab import files
    files.download(str(archive))

## References for the experiment design

- Astete-Morales, Cauwet and Teytaud. Analysis of Different Types of Regret in Continuous Noisy Optimization. https://arxiv.org/abs/1607.06651
- Boesel, Nelson and Kim. Using Ranking and Selection to Clean Up after Simulation Optimization. https://doi.org/10.1287/opre.51.5.814.16751
- Frazier. A Tutorial on Bayesian Optimization, noisy-evaluation section. https://arxiv.org/abs/1807.02811
- Hansen et al. A Method for Handling Uncertainty in Evolutionary Optimization. https://doi.org/10.1109/TEVC.2008.924423
- Scott, Frazier and Powell. The Correlated Knowledge Gradient for Simulation Optimization of Continuous Parameters using Gaussian Process Regression. https://doi.org/10.1137/100801275
- COCO f14. https://coco-platform.org/testsuites/bbob/functions/f14.html
- COCO f15. https://coco-platform.org/testsuites/bbob/functions/f15.html